# SMALL ULTIMATE V1 — Data-driven pocket chemistry and topology

**Task:** `small_binding` residue prediction  
**Development:** exact DevSet1010 official bindNode24 5-fold protein split  
**External confirmation:** TestSet300 + TestSetNew46  
**Primary selection metric:** residue-level Dev CV AUPRC

This notebook is designed from the completed all-binding anatomy audit.

## Audit-derived small-binding phenotype

Across DevSet1010 / TestSet300 / TestSetNew46:

- small-binding prevalence is relatively stable (~5.4%, 6.2%, 6.8%);
- positive-protein fraction is high (~60%, 73%, 54%);
- positive interfaces are medium-sized (median ~13, 16, 11 residues);
- positive AA signatures are highly stable from Dev→300 and Dev→46;
- the strongest consistent residue enrichments are **W, C, H, Y, F, M**;
- acidic residues are consistently depleted;
- aromatic, sulfur-containing, gly/pro-rich, and hydrophobic local contexts show interpretable signal;
- cached-feature fingerprints consistently favor **ACCESS**, with meaningful **TOPO / ATOM / CHEM / SPATIAL** contributions;
- generic `CONF` is unstable on Test46 and is not a core feature;
- EVO did not consistently improve the prior small-binding pipeline and is retained only as an optional context/control.

This implies a mechanism distinct from both metal and nuclear binding:

```text
Metal   → coordination chemistry
Nuclear → electrostatic surface patch
Small   → pocket accessibility + pocket geometry
          + aromatic/hydrophobic/heteroatom chemistry
          + local topology/packing
```

## Scientific contract

```text
DevSet1010 only
   │
   ├── feature-family MLP screen
   ├── mechanism-combination ladder
   ├── multimodal architecture screen
   └── pocket-hypergraph screen
   │
   ▼
GLOBAL LOCK
   │
   ▼
TestSet300 + TestSetNew46
```

No feature/model choice may depend on TestSet300 or TestSetNew46.


In [ ]:
# ============================================================
# 0. DRIVE / INSTALL / IMPORT
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, math, shutil, subprocess, platform
from collections import Counter, defaultdict

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "pandas", "scipy", "scikit-learn",
     "biopython", "tqdm"],
    check=True
)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset
from scipy.spatial import cKDTree
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, matthews_corrcoef,
    precision_score, recall_score
)

from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.SeqUtils import seq1
from copy import deepcopy

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')

ULTIMATE_CANDIDATES = [
    BASE/'GEAR_ULTIMATE_OTHER_BINDINGS',
    BASE/'GEAR_ULTIMATE',
]

ULTIMATE_DIR = next(
    (p for p in ULTIMATE_CANDIDATES if (p/'gear_ultimate').exists()),
    None
)

if ULTIMATE_DIR is None:
    raise FileNotFoundError(
        'GEAR backend not found under GEAR_ULTIMATE_OTHER_BINDINGS or GEAR_ULTIMATE.'
    )

req = ULTIMATE_DIR/'requirements.txt'
if req.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )

sys.path.insert(0, str(ULTIMATE_DIR))

try:
    from gear_ultimate import GEAR_ULTIMATE_VERSION as GEAR_VERSION
except Exception:
    from gear_ultimate import __version__ as GEAR_VERSION

from gear_ultimate import gear_core as core
from gear_ultimate import gear_structure as structure
from gear_ultimate import gear_mechanistic as mech
from gear_ultimate import gear_evolution as evo
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

RUN_ROOT = BASE/'results'/'SMALL_ULTIMATE_V1_DATA_DRIVEN_POCKET'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = BASE/'cache'/'small_ultimate_v1'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('GEAR backend:', ULTIMATE_DIR)
print('GEAR version:', GEAR_VERSION)
print('Device:', DEVICE)
print('Run root:', RUN_ROOT)


Mounted at /content/drive
BASE: /content/drive/MyDrive/Protein-binding-Phi
GEAR backend: /content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE_OTHER_BINDINGS
GEAR version: 2026.08-other-bindings-v1
Device: cuda
Run root: /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_ULTIMATE_V1_DATA_DRIVEN_POCKET


## 1. Master switches

Default = full small-binding run.

All generated feature blocks, pocket graphs, fold states, and validation predictions are persisted for resume.


In [ ]:
# ============================================================
# 1. MASTER SWITCHES
# ============================================================
ESM_BATCH_SIZE = 8

RUN_FEATURE_GENERATION = True
RUN_FAMILY_SCREEN = True
RUN_COMBINATION_LADDER = True
RUN_CUSTOM_ARCHITECTURES = True
RUN_POCKET_HYPERGRAPH = True
RUN_FINAL_EXTERNAL = True

ATTACH_EVO_IF_CACHED = True
ATTACH_ESMIF_IF_CACHED = True

FORCE_REBUILD_SM_FEATURES = False
FORCE_REBUILD_POCKETS = False

MIN_MEAN_AUPRC_GAIN = 0.002
GLOBAL_COMPLEXITY_MARGIN = 0.002

CUSTOM_BATCH = 4096
CUSTOM_EPOCHS = 45
CUSTOM_PATIENCE = 7
CUSTOM_LR = 3e-4
CUSTOM_WEIGHT_DECAY = 1e-4

POCKET_RADIUS_A = 8.0
POCKET_MAX_PER_PROTEIN = 160
POCKET_PROTEIN_BATCH = 4
POCKET_EPOCHS = 32
POCKET_PATIENCE = 6
POCKET_LR = 3e-4
POCKET_WEIGHT_DECAY = 1e-4
POCKET_LATENT = 192

SEED = 2026

print('Full run:', {
    'features': RUN_FEATURE_GENERATION,
    'family_screen': RUN_FAMILY_SCREEN,
    'combination_ladder': RUN_COMBINATION_LADDER,
    'architectures': RUN_CUSTOM_ARCHITECTURES,
    'pocket_hypergraph': RUN_POCKET_HYPERGRAPH,
    'external': RUN_FINAL_EXTERNAL,
})


Full run: {'features': True, 'family_screen': True, 'combination_ladder': True, 'architectures': True, 'pocket_hypergraph': True, 'external': True}


In [ ]:
# ============================================================
# 2. PREPARE / REUSE SMALL-BINDING GEAR CONTEXT
# ============================================================
ctx = suite.prepare_context(
    base=BASE,
    run_version='small_ultimate_v1_context',
    label_column='small_binding',
    models=('MLP',),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=ESM_BATCH_SIZE,
    device=DEVICE,
)

display(suite.audit_context(ctx))
display(reporting.dimension_table(ctx.stores['train']))

assert len(ctx.stores['train']) == 1010
assert len(ctx.stores['test300']) == 300
assert len(ctx.stores['test46']) == 46

print('Context sizes: PASS')


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

,split,proteins,residues,positives,prevalence
0,train,1010,170181,9236,0.054272
1,test300,300,62689,3906,0.062308
2,test46,46,6227,425,0.068251


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
3,LOCAL,8
10,METALSPEC,18


Context sizes: PASS


In [ ]:
# ============================================================
# 3. DATAFRAME / OFFICIAL-FOLD AUDIT
# ============================================================
TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)

    for c in [
        'metal_binding',
        'nuclear_binding',
        'small_binding',
        'any_ligand_binding_sites',
    ]:
        if c in df.columns:
            df[c] = df[c].apply(parse_list)

    for r in df.itertuples():
        if len(r.sequence) != len(r.small_binding):
            raise RuntimeError(f'{r.prot_id}: small label/sequence mismatch')

    return df

raw_train = load_df(TRAIN_CSV)
test300_df = load_df(TEST300_CSV)
test46_df = load_df(TEST46_CSV)

BASELINE_WORK = BASE/'baseline_reproduction_master_v6'
BN24_CANDIDATES = [
    BASELINE_WORK/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
]

BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_small_v1')
    if not BN24.exists():
        subprocess.run(
            ['git','clone','-q','https://github.com/erckert/bindNode24.git',str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
OFFICIAL_FOLDS = {}
official_order = []
seen = set()

for k in range(1,6):
    ids = [
        x.strip()
        for x in (fold_root/f'ids_split{k}.txt').read_text().splitlines()
        if x.strip()
    ]

    if seen.intersection(ids):
        raise RuntimeError(f'Fold overlap at fold {k}')

    OFFICIAL_FOLDS[k] = ids
    official_order.extend(ids)
    seen.update(ids)

train_lookup = raw_train.set_index('prot_id', drop=False)

missing = set(official_order) - set(raw_train.prot_id)
if missing:
    raise RuntimeError(
        f'Official Dev IDs missing from project CSV: {sorted(missing)[:20]}'
    )

train_df = train_lookup.loc[official_order].reset_index(drop=True)

assert len(train_df) == 1010
assert len(test300_df) == 300
assert len(test46_df) == 46

SPLIT_DF = {
    'DevSet1010': train_df,
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}

STORE_KEY = {
    'DevSet1010':'train',
    'TestSet300':'test300',
    'TestSetNew46':'test46',
}

for split,df in SPLIT_DF.items():
    store_ids = set(map(str, ctx.stores[STORE_KEY[split]].keys()))
    csv_ids = set(df.prot_id.astype(str))

    if store_ids != csv_ids:
        raise RuntimeError(
            f'{split}: store/CSV mismatch. '
            f'missing_in_store={sorted(csv_ids-store_ids)[:10]} '
            f'extra_in_store={sorted(store_ids-csv_ids)[:10]}'
        )

print('Exact dataset/fold alignment: PASS')
print('Fold sizes:', {k:len(v) for k,v in OFFICIAL_FOLDS.items()})


Exact dataset/fold alignment: PASS
Fold sizes: {1: 202, 2: 203, 3: 203, 4: 202, 5: 200}


## 4. Optional biological-context reuse

Prior small-binding runs showed that EVO does not consistently improve the small-binding model, so EVO is an optional control, not a core assumption.

ESM-IF is also reuse-only: it is attached only if a complete cache already exists.


In [ ]:
# ============================================================
# 4. STORE HELPERS + OPTIONAL EVO / ESMIF
# ============================================================
def get_record(store,pid):
    pid = str(pid)
    if pid in store:
        return store[pid]
    raise KeyError(pid)

def get_block(store,pid,block):
    rec = get_record(store,pid)

    if isinstance(rec,dict):
        if block in rec:
            return np.asarray(rec[block])
        if isinstance(rec.get('features'),dict) and block in rec['features']:
            return np.asarray(rec['features'][block])

    if hasattr(rec,block):
        return np.asarray(getattr(rec,block))

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec,attr,None)
        if isinstance(d,dict) and block in d:
            return np.asarray(d[block])

    raise KeyError(f'{pid}/{block}')

def set_block(store,pid,block,arr):
    rec = get_record(store,pid)
    arr = np.asarray(arr,dtype=np.float32)

    if isinstance(rec,dict):
        if isinstance(rec.get('features'),dict):
            rec['features'][block] = arr
        else:
            rec[block] = arr
        return

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec,attr,None)
        if isinstance(d,dict):
            d[block] = arr
            return

    setattr(rec,block,arr)

EVO_AVAILABLE = False

if ATTACH_EVO_IF_CACHED:
    try:
        coverage = evo.attach_evo(
            ctx.stores,
            BASE,
            splits=('train','test300','test46'),
            require_msa=True,
        )
        display(coverage)
        EVO_AVAILABLE = True
        print('EVO48 attached as optional control.')
    except Exception as e:
        print('EVO unavailable/incomplete -> skip:', repr(e))

ESMIF_AVAILABLE = False
ESMIF_CANDIDATE_ROOTS = [
    BASE/'cache'/'gear_metal_v3'/'esmif_v1',
    BASE/'cache'/'nuclear_ultimate_v1'/'esmif_v1',
]

if ATTACH_ESMIF_IF_CACHED:
    for root in ESMIF_CANDIDATE_ROOTS:
        split_dirs = {
            'DevSet1010':root/'DevSet1010'/'esm_if',
            'TestSet300':root/'TestSet300'/'esm_if',
            'TestSetNew46':root/'TestSetNew46'/'esm_if',
        }

        if not all(d.exists() for d in split_dirs.values()):
            continue

        good = True

        for split,df in SPLIT_DF.items():
            d = split_dirs[split]

            for r in df.itertuples():
                fp = d/f'{r.prot_id}.npy'

                if not fp.exists():
                    good = False
                    break

                a = np.load(fp,mmap_mode='r')

                if a.shape != (len(r.sequence),512):
                    good = False
                    break

            if not good:
                break

        if good:
            for split,df in SPLIT_DF.items():
                store = ctx.stores[STORE_KEY[split]]
                d = split_dirs[split]

                for r in df.itertuples():
                    set_block(
                        store,
                        r.prot_id,
                        'ESMIF',
                        np.load(
                            d/f'{r.prot_id}.npy'
                        ).astype(np.float32)
                    )

            ESMIF_AVAILABLE = True
            print('Reused complete ESMIF cache:',root)
            break

print('EVO_AVAILABLE  =',EVO_AVAILABLE)
print('ESMIF_AVAILABLE=',ESMIF_AVAILABLE)


EVO cache:   0%|          | 0/1353 [00:00<?, ?it/s]

,split,protein,msa_depth,has_real_msa
0,train,A0A0F7RDM3,977,True
1,train,A0A0F7RHX8,1024,True
2,train,A0A0H2W6Y8,1023,True
3,train,A0A2B6C3P9,1024,True
4,train,A0A384LKY8,542,True
...,...,...,...,...
1351,test46,Q9KDJ7,1023,True
1352,test46,Q9LFM3,1018,True
1353,test46,Q9SJ89,1022,True
1354,test46,U2EQ00,1021,True


EVO48 attached as optional control.
EVO_AVAILABLE  = True
ESMIF_AVAILABLE= False


# Part A — Small-molecule-specific feature generation

## `SM_SEQCHEM32`
Sequence-local aromatic/hydrophobic/sulfur/flexibility chemistry.

## `SM_POCKET3D44`
3D pocket microenvironment: local packing, shell occupancy, aromatic/hydrophobic/polar composition, tertiary-contact geometry.

## `SM_LIGCHEM20`
Explicit ligand-contact chemistry: H-bond donors/acceptors, aromatic, sulfur, polar, charged and amide/hydroxyl context.

## `SM_TOPO16`
Local spatial graph topology and compactness.

## `SM_AROMSTACK12`
Aromatic-cluster / π-contact proxy. Unlike nuclear, aromatic chemistry is a core hypothesis for small binding because W/F/Y/H enrichment is stable across all three splits.

## `SM_PROTCTX8`
Soft global context only.

Existing `ACCESS`, `TOPO`, `CHEM`, `SOLV`, `ATOM`, and generic structure remain useful controls.


In [ ]:
# # ============================================================
# # 5. NORMALIZE EXACT-SEQUENCE PDB STRUCTURES
# # ============================================================
# AA3 = {
#     'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
#     'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
# }

# PDB_NORM_ROOT = CACHE_ROOT/'normalized_pdb'
# PDB_NORM_ROOT.mkdir(parents=True,exist_ok=True)

# # Reuse already-normalized nuclear structures if present.
# NUC_NORM = BASE/'cache'/'nuclear_ultimate_v1'/'normalized_pdb'

# pdb_roots = [
#     BASE/'esmFold_pdb_files',
#     BASE/'experimental_pdb',
#     BASELINE_WORK/'cache'/'metal_external_normalized_structures',
#     BN24/'dataset'/'AlphaFold_predictions',
# ]

# all_pdb = []

# for root in pdb_roots:
#     if root.exists():
#         all_pdb.extend(root.rglob('*.pdb'))

# exact_index = {}
# for p in all_pdb:
#     exact_index.setdefault(p.stem,p)

# def standard_residues(chain):
#     return [
#         r for r in chain
#         if r.id[0]==' ' and r.resname.strip().upper() in AA3
#     ]

# def residue_sequence(rr):
#     return ''.join(
#         seq1(r.resname.strip().upper())
#         for r in rr
#     )

# def cached_norm_ok(path,target_seq):
#     if not path.exists():
#         return False

#     try:
#         st = PDBParser(QUIET=True).get_structure('x',str(path))
#         m = next(st.get_models())

#         if 'A' not in m:
#             return False

#         return (
#             residue_sequence(
#                 standard_residues(m['A'])
#             )
#             == target_seq
#         )

#     except Exception:
#         return False

# def find_source_pdb(split,pid,seq):
#     # Prefer prior normalized nuclear cache if available.
#     p = NUC_NORM/split/f'{pid}.pdb'
#     if cached_norm_ok(p,seq):
#         return p

#     if str(pid) in exact_index:
#         return exact_index[str(pid)]

#     for p in [
#         BASE/'esmFold_pdb_files'/f'{pid}.pdb',
#         BN24/'dataset'/'AlphaFold_predictions'/f'{pid}.pdb',
#     ]:
#         if p.exists():
#             return p

#     hits = [p for p in all_pdb if str(pid) in p.stem]

#     if len(hits)==1:
#         return hits[0]

#     return None

# def normalize_exact_chain_A(src,target_seq,dst):
#     st = PDBParser(QUIET=True).get_structure('src',str(src))
#     m = next(st.get_models())

#     exact = []
#     diag = []

#     for ch in m:
#         rr = standard_residues(ch)
#         seq = residue_sequence(rr)
#         diag.append((str(ch.id),len(seq)))

#         if seq==target_seq:
#             exact.append((ch,rr))

#     if not exact:
#         raise RuntimeError(
#             f'{src}: no chain exactly matches length {len(target_seq)}; {diag}'
#         )

#     exact.sort(
#         key=lambda x:(
#             0 if str(x[0].id)=='A' else 1,
#             str(x[0].id)
#         )
#     )

#     _,rr = exact[0]

#     out_st = Structure('normalized')
#     out_m = Model(0)
#     out_ch = Chain('A')
#     out_st.add(out_m)
#     out_m.add(out_ch)

#     for i,res in enumerate(rr,start=1):
#         r2 = deepcopy(res)
#         r2.detach_parent()
#         r2.id = (' ',i,' ')
#         out_ch.add(r2)

#     dst.parent.mkdir(parents=True,exist_ok=True)

#     io = PDBIO()
#     io.set_structure(out_st)
#     io.save(str(dst))

#     if not cached_norm_ok(dst,target_seq):
#         raise RuntimeError(f'{dst}: read-back mismatch')

# for split,df in SPLIT_DF.items():
#     d = PDB_NORM_ROOT/split
#     d.mkdir(parents=True,exist_ok=True)

#     missing = []

#     for n,r in enumerate(df.itertuples(),start=1):
#         dst = d/f'{r.prot_id}.pdb'

#         if cached_norm_ok(dst,r.sequence):
#             continue

#         src = find_source_pdb(split,r.prot_id,r.sequence)

#         if src is None:
#             missing.append(str(r.prot_id))
#             continue

#         normalize_exact_chain_A(src,r.sequence,dst)

#         if n%250==0:
#             print(split,n,'/',len(df))

#     if missing:
#         raise FileNotFoundError(
#             f'{split}: missing source PDB for {len(missing)} proteins: {missing[:20]}'
#         )

# print('Exact-sequence normalized structures: PASS')


In [ ]:
# ============================================================
# 5. SHARED EXACT PDB CACHE — NUCLEAR + SMALL
# ============================================================
from pathlib import Path
import shutil
import tarfile
import json
import hashlib
import os

AA3 = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
}

# ------------------------------------------------------------
# ONE local cache shared by nuclear + small
# ------------------------------------------------------------
LOCAL_SHARED_ROOT = Path('/content/shared_exact_pdb_v1')

DRIVE_ARCHIVE = (
    BASE / 'cache' / 'shared_exact_pdb_v1.tar.gz'
)

DRIVE_ARCHIVE.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Downstream cells continue using PDB_NORM_ROOT.
PDB_NORM_ROOT = LOCAL_SHARED_ROOT

# Existing work that we should salvage rather than redo.
OLD_NORMALIZED_ROOTS = [
    BASE / 'cache' / 'nuclear_ultimate_v1' / 'normalized_pdb',
    BASE / 'cache' / 'small_ultimate_v1' / 'normalized_pdb',

    BASELINE_WORK
    / 'cache'
    / 'metal_external_normalized_structures',
]

# Raw source roots only used when no normalized copy exists.
RAW_ROOTS = [
    BASE / 'esmFold_pdb_files',
    BASE / 'experimental_pdb',
    BN24 / 'dataset' / 'AlphaFold_predictions',
]

_parser = PDBParser(QUIET=True)


# ============================================================
# helpers
# ============================================================
def standard_residues(chain):
    return [
        r for r in chain
        if (
            r.id[0] == ' '
            and r.resname.strip().upper() in AA3
        )
    ]


def residue_sequence(rr):
    return ''.join(
        seq1(r.resname.strip().upper())
        for r in rr
    )


def pdb_exact_ok(path, target_seq):
    """
    Check whether PDB is already exact chain-A normalized.

    Important:
    use this mainly on /content or on a candidate file,
    NOT recursively over Drive.
    """
    path = Path(path)

    if not path.exists():
        return False

    try:
        st = _parser.get_structure(
            'check',
            str(path)
        )

        model = next(st.get_models())

        if 'A' not in model:
            return False

        rr = standard_residues(
            model['A']
        )

        return (
            residue_sequence(rr)
            == target_seq
        )

    except Exception:
        return False


def normalize_exact_chain_A(src, target_seq, dst):
    """
    Normalize ONLY if we truly do not already have
    an exact-normalized copy.
    """
    st = _parser.get_structure(
        'src',
        str(src)
    )

    model = next(
        st.get_models()
    )

    exact = []
    diag = []

    for ch in model:
        rr = standard_residues(ch)
        seq = residue_sequence(rr)

        diag.append(
            (
                str(ch.id),
                len(seq)
            )
        )

        if seq == target_seq:
            exact.append(
                (
                    ch,
                    rr
                )
            )

    if not exact:
        raise RuntimeError(
            f'{src}: no exact chain match; '
            f'target={len(target_seq)}, '
            f'chains={diag}'
        )

    exact.sort(
        key=lambda x: (
            0
            if str(x[0].id) == 'A'
            else 1,

            str(x[0].id)
        )
    )

    _, rr = exact[0]

    out_st = Structure('normalized')
    out_model = Model(0)
    out_chain = Chain('A')

    out_st.add(out_model)
    out_model.add(out_chain)

    for i, res in enumerate(
        rr,
        start=1
    ):
        r2 = deepcopy(res)
        r2.detach_parent()

        r2.id = (
            ' ',
            i,
            ' '
        )

        out_chain.add(
            r2
        )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    io = PDBIO()
    io.set_structure(
        out_st
    )

    io.save(
        str(dst)
    )


# ============================================================
# 1. RESTORE SHARED CACHE IF IT EXISTS
# ============================================================
if (
    DRIVE_ARCHIVE.exists()
    and not LOCAL_SHARED_ROOT.exists()
):
    print(
        'Found shared PDB archive.'
    )

    print(
        'Copy Drive -> /content ...'
    )

    local_tar = (
        Path('/content')
        / DRIVE_ARCHIVE.name
    )

    shutil.copy2(
        DRIVE_ARCHIVE,
        local_tar
    )

    print(
        'Extracting locally ...'
    )

    with tarfile.open(
        local_tar,
        'r:gz'
    ) as tf:
        tf.extractall(
            '/content'
        )

    local_tar.unlink(
        missing_ok=True
    )


LOCAL_SHARED_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FAST LOOKUP OF OLD NORMALIZED STRUCTURES
# ============================================================
# Direct paths first — no rglob.
def normalized_candidates(
    split,
    pid
):
    pid = str(pid)

    aliases = {
        'DevSet1010': [
            'DevSet1010',
            'train',
        ],

        'TestSet300': [
            'TestSet300',
            'test300',
        ],

        'TestSetNew46': [
            'TestSetNew46',
            'test46',
        ],
    }

    out = []

    for root in OLD_NORMALIZED_ROOTS:
        out.append(
            root / f'{pid}.pdb'
        )

        for alias in aliases.get(
            split,
            []
        ):
            out.append(
                root
                / alias
                / f'{pid}.pdb'
            )

    return out


def raw_candidates(
    pid
):
    pid = str(pid)

    return [
        BASE
        / 'esmFold_pdb_files'
        / f'{pid}.pdb',

        BASE
        / 'experimental_pdb'
        / f'{pid}.pdb',

        BN24
        / 'dataset'
        / 'AlphaFold_predictions'
        / f'{pid}.pdb',
    ]


# Only build fallback index if direct paths actually fail.
RAW_FALLBACK_INDEX = None


def get_raw_fallback_index():
    global RAW_FALLBACK_INDEX

    if RAW_FALLBACK_INDEX is not None:
        return RAW_FALLBACK_INDEX

    print(
        '\nDirect raw paths missed at least one protein.'
    )

    print(
        'Building ONE fallback source index...'
    )

    idx = {}

    for root in RAW_ROOTS:
        if not root.exists():
            continue

        print(
            ' scanning:',
            root
        )

        for dirpath, _, filenames in os.walk(
            root
        ):
            for fn in filenames:
                if not fn.endswith(
                    '.pdb'
                ):
                    continue

                p = (
                    Path(dirpath)
                    / fn
                )

                idx.setdefault(
                    p.stem,
                    p
                )

    RAW_FALLBACK_INDEX = idx

    print(
        'fallback PDBs indexed:',
        len(idx)
    )

    return idx


# ============================================================
# 3. BUILD SHARED LOCAL CACHE
# ============================================================
stats = {
    'already_local': 0,
    'reused_old_normalized': 0,
    'normalized_raw': 0,
}

missing = []

TOTAL = sum(
    len(df)
    for df in SPLIT_DF.values()
)

done = 0


for split, df in SPLIT_DF.items():

    outdir = (
        LOCAL_SHARED_ROOT
        / split
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        '\n',
        split,
        len(df),
        'proteins'
    )

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        dst = (
            outdir
            / f'{pid}.pdb'
        )

        # ----------------------------------
        # A. already local shared
        # ----------------------------------
        if pdb_exact_ok(
            dst,
            r.sequence
        ):
            stats[
                'already_local'
            ] += 1

            done += 1
            continue


        # ----------------------------------
        # B. reuse nuclear/small/metal
        #    normalized PDB AS-IS
        # ----------------------------------
        reused = False

        for cand in normalized_candidates(
            split,
            pid
        ):

            if not cand.exists():
                continue

            if pdb_exact_ok(
                cand,
                r.sequence
            ):
                shutil.copy2(
                    cand,
                    dst
                )

                stats[
                    'reused_old_normalized'
                ] += 1

                reused = True
                break


        if reused:
            done += 1

            if done % 100 == 0:
                print(
                    f'  {done}/{TOTAL}',
                    stats
                )

            continue


        # ----------------------------------
        # C. raw source direct lookup
        # ----------------------------------
        src = None

        for cand in raw_candidates(
            pid
        ):
            if cand.exists():
                src = cand
                break


        # ----------------------------------
        # D. last-resort ONE-TIME index
        # ----------------------------------
        if src is None:
            src = get_raw_fallback_index().get(
                pid
            )


        if src is None:
            missing.append(
                (
                    split,
                    pid
                )
            )

            continue


        # ----------------------------------
        # E. normalize only genuinely missing
        # ----------------------------------
        normalize_exact_chain_A(
            src,
            r.sequence,
            dst
        )

        if not pdb_exact_ok(
            dst,
            r.sequence
        ):
            raise RuntimeError(
                f'{split}/{pid}: '
                'normalized PDB failed validation'
            )

        stats[
            'normalized_raw'
        ] += 1

        done += 1


        if done % 100 == 0:
            print(
                f'  {done}/{TOTAL}',
                stats
            )


if missing:
    raise FileNotFoundError(
        f'Missing source PDB for '
        f'{len(missing)} proteins: '
        f'{missing[:20]}'
    )


print(
    '\nCACHE BUILD COMPLETE'
)

print(
    stats
)


# ============================================================
# 4. SAVE ONE ARCHIVE TO DRIVE
# ============================================================
print(
    '\nPacking shared local cache...'
)

local_tar = (
    Path('/content')
    / 'shared_exact_pdb_v1.tar.gz'
)

with tarfile.open(
    local_tar,
    'w:gz'
) as tf:

    tf.add(
        LOCAL_SHARED_ROOT,
        arcname='shared_exact_pdb_v1'
    )


print(
    'Copying ONE archive to Drive...'
)

shutil.copy2(
    local_tar,
    DRIVE_ARCHIVE
)

local_tar.unlink(
    missing_ok=True
)


print(
    '\nExact-sequence shared structures: PASS'
)

print(
    'PDB_NORM_ROOT =',
    PDB_NORM_ROOT
)

print(
    'Persistent archive =',
    DRIVE_ARCHIVE
)

Found shared PDB archive.
Copy Drive -> /content ...
Extracting locally ...


/tmp/ipykernel_709/1301324207.py:237: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(



 DevSet1010 1010 proteins

 TestSet300 300 proteins

 TestSetNew46 46 proteins

CACHE BUILD COMPLETE
{'already_local': 1356, 'reused_old_normalized': 0, 'normalized_raw': 0}

Packing shared local cache...
Copying ONE archive to Drive...

Exact-sequence shared structures: PASS
PDB_NORM_ROOT = /content/shared_exact_pdb_v1
Persistent archive = /content/drive/MyDrive/Protein-binding-Phi/cache/shared_exact_pdb_v1.tar.gz


In [ ]:
# ============================================================
# 6. SMALL-BINDING-SPECIFIC FEATURE DEFINITIONS
# ============================================================
FEATURE_CACHE = CACHE_ROOT/'small_explicit_v1'
FEATURE_CACHE.mkdir(parents=True,exist_ok=True)

# Kyte-Doolittle hydropathy.
HYDROPATHY = {
    'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,
    'G':-0.4,'H':-3.2,'I':4.5,'K':-3.9,'L':3.8,
    'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,
    'S':-0.8,'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3,
}

GROUPS = {
    'AROM': set('FWYH'),
    'STRONG_AROM': set('FWY'),
    'SULFUR': set('CM'),
    'HYDRO': set('AVILMFWY'),
    'ALIPHATIC': set('AVILM'),
    'POLAR': set('NQSTY'),
    'ACIDIC': set('DE'),
    'BASIC': set('KRH'),
    'GP': set('GP'),
    'HBD': set('KRHNQSTYW'),
    'HBA': set('DEHNQSTYC'),
    'AMIDE': set('NQ'),
    'HYDROXYL': set('STY'),
}

CHARGE = {
    'K':1.0,
    'R':1.0,
    'H':0.10,
    'D':-1.0,
    'E':-1.0,
}

BACKBONE_ATOMS = {'N','CA','C','O','OXT'}

RING_ATOMS = {
    'PHE':['CG','CD1','CE1','CZ','CE2','CD2'],
    'TYR':['CG','CD1','CE1','CZ','CE2','CD2'],
    'HIS':['CG','ND1','CE1','NE2','CD2'],
    'TRP':['CG','CD1','NE1','CE2','CD2','CE3','CZ3','CH2','CZ2'],
}

def representative_coord(res):
    side = [
        np.asarray(a.coord,dtype=np.float32)
        for a in res
        if a.element != 'H' and a.name not in BACKBONE_ATOMS
    ]

    if side:
        return np.mean(np.stack(side),axis=0)

    if 'CA' in res:
        return np.asarray(res['CA'].coord,dtype=np.float32)

    atoms = [
        np.asarray(a.coord,dtype=np.float32)
        for a in res
    ]

    if not atoms:
        raise RuntimeError('Residue has no atoms')

    return np.mean(np.stack(atoms),axis=0)

def ring_geometry(res):
    rn = res.resname.strip().upper()
    atoms = RING_ATOMS.get(rn)

    if atoms is None:
        return None,None

    cc = [
        np.asarray(res[a].coord,dtype=np.float32)
        for a in atoms
        if a in res
    ]

    if len(cc) < 3:
        return None,None

    x = np.stack(cc)
    centroid = x.mean(axis=0)
    centered = x-centroid[None,:]
    cov = centered.T@centered/max(len(x)-1,1)
    eigval,eigvec = np.linalg.eigh(cov)
    normal = eigvec[:,np.argmin(eigval)]
    normal = normal/(np.linalg.norm(normal)+1e-8)

    return centroid.astype(np.float32),normal.astype(np.float32)

def connected_component_sizes(indices,coords,threshold):
    if len(indices)==0:
        return {}

    idx = np.asarray(indices,dtype=int)
    cc = np.asarray(coords,dtype=np.float32)
    parent = list(range(len(idx)))

    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x

    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb:
            parent[rb]=ra

    if len(idx)>1:
        tree = cKDTree(cc)
        for a,b in tree.query_pairs(threshold):
            union(a,b)

    comp = defaultdict(list)

    for j,ridx in enumerate(idx):
        comp[find(j)].append(int(ridx))

    out={}

    for members in comp.values():
        n=len(members)
        for r in members:
            out[r]=n

    return out

def sequence_chem_block(seq):
    L=len(seq)
    out=np.zeros((L,32),dtype=np.float32)

    hydro=np.asarray(
        [HYDROPATHY.get(a,0.0)/4.5 for a in seq],
        dtype=np.float32
    )

    for i,a in enumerate(seq):
        out[i,0]=hydro[i]
        out[i,1]=float(a in GROUPS['AROM'])
        out[i,2]=float(a in GROUPS['SULFUR'])
        out[i,3]=float(a in GROUPS['HYDRO'])
        out[i,4]=float(a in GROUPS['ACIDIC'])
        out[i,5]=float(a in GROUPS['POLAR'])
        out[i,6]=float(a in GROUPS['GP'])
        out[i,7]=float(a in GROUPS['BASIC'])

        col=8

        for rad in [2,4,7,10]:
            lo=max(0,i-rad)
            hi=min(L,i+rad+1)
            w=seq[lo:hi]
            h=hydro[lo:hi]

            vals=[
                sum(x in GROUPS['AROM'] for x in w)/len(w),
                sum(x in GROUPS['HYDRO'] for x in w)/len(w),
                sum(x in GROUPS['SULFUR'] for x in w)/len(w),
                sum(x in GROUPS['ACIDIC'] for x in w)/len(w),
                sum(x in GROUPS['GP'] for x in w)/len(w),
                float(np.mean(h)),
            ]

            out[i,col:col+6]=vals
            col+=6

    return out

def build_small_3d_blocks(pdb_path,seq):
    st=PDBParser(QUIET=True).get_structure('p',str(pdb_path))
    m=next(st.get_models())
    residues=standard_residues(m['A'])

    if residue_sequence(residues)!=seq:
        raise RuntimeError(f'{pdb_path}: sequence mismatch')

    coords=np.stack(
        [representative_coord(r) for r in residues],
        axis=0
    ).astype(np.float32)

    L=len(seq)
    tree=cKDTree(coords)

    hydro=np.asarray(
        [HYDROPATHY.get(a,0.0)/4.5 for a in seq],
        dtype=np.float32
    )

    q=np.asarray(
        [CHARGE.get(a,0.0) for a in seq],
        dtype=np.float32
    )

    arom_idx=[
        i for i,a in enumerate(seq)
        if a in GROUPS['AROM']
    ]

    hydro_idx=[
        i for i,a in enumerate(seq)
        if a in GROUPS['HYDRO']
    ]

    arom_comp=connected_component_sizes(
        arom_idx,
        coords[arom_idx] if arom_idx else np.zeros((0,3),dtype=np.float32),
        threshold=8.0
    )

    hydro_comp=connected_component_sizes(
        hydro_idx,
        coords[hydro_idx] if hydro_idx else np.zeros((0,3),dtype=np.float32),
        threshold=8.0
    )

    ring_centroid={}
    ring_normal={}

    for i,res in enumerate(residues):
        c,n=ring_geometry(res)

        if c is not None:
            ring_centroid[i]=c
            ring_normal[i]=n

    pocket=np.zeros((L,44),dtype=np.float32)
    ligchem=np.zeros((L,20),dtype=np.float32)
    topo=np.zeros((L,16),dtype=np.float32)
    arom=np.zeros((L,12),dtype=np.float32)

    neigh6=[
        tree.query_ball_point(coords[i],6.0)
        for i in range(L)
    ]
    neigh8=[
        tree.query_ball_point(coords[i],8.0)
        for i in range(L)
    ]
    neigh10=[
        tree.query_ball_point(coords[i],10.0)
        for i in range(L)
    ]

    for i,a in enumerate(seq):
        # ----------------------------------------------------
        # SM_LIGCHEM20: own interaction chemistry
        # ----------------------------------------------------
        ligchem[i,0]=float(a in GROUPS['HBD'])
        ligchem[i,1]=float(a in GROUPS['HBA'])
        ligchem[i,2]=float(a in GROUPS['AROM'])
        ligchem[i,3]=float(a in GROUPS['SULFUR'])
        ligchem[i,4]=float(a in GROUPS['HYDRO'])
        ligchem[i,5]=float(a in GROUPS['POLAR'])
        ligchem[i,6]=float(a in GROUPS['ACIDIC'])
        ligchem[i,7]=float(a in GROUPS['BASIC'])
        ligchem[i,8]=float(a in GROUPS['AMIDE'])
        ligchem[i,9]=float(a in GROUPS['HYDROXYL'])

        c=10

        for radius in [6.0,10.0]:
            ids=np.asarray(
                tree.query_ball_point(coords[i],radius),
                dtype=int
            )

            ids=ids[ids!=i]
            aa=[seq[j] for j in ids]
            den=max(len(aa),1)

            ligchem[i,c:c+5]=[
                sum(x in GROUPS['HBD'] for x in aa)/den,
                sum(x in GROUPS['HBA'] for x in aa)/den,
                sum(x in GROUPS['AROM'] for x in aa)/den,
                sum(x in GROUPS['SULFUR'] for x in aa)/den,
                sum(x in GROUPS['POLAR'] for x in aa)/den,
            ]

            c+=5

        # ----------------------------------------------------
        # SM_POCKET3D44: four radii x eight descriptors
        # ----------------------------------------------------
        c=0

        for radius in [5.0,7.0,9.0,11.0]:
            ids=np.asarray(
                tree.query_ball_point(coords[i],radius),
                dtype=int
            )
            ids=ids[ids!=i]

            if len(ids):
                aa=[seq[j] for j in ids]
                hh=hydro[ids]
                dd=np.linalg.norm(
                    coords[ids]-coords[i][None,:],
                    axis=1
                )

                vals=[
                    min(len(ids),40)/40.0,
                    sum(x in GROUPS['AROM'] for x in aa)/len(aa),
                    sum(x in GROUPS['HYDRO'] for x in aa)/len(aa),
                    sum(x in GROUPS['SULFUR'] for x in aa)/len(aa),
                    sum(x in GROUPS['POLAR'] for x in aa)/len(aa),
                    sum(x in GROUPS['ACIDIC'] for x in aa)/len(aa),
                    float(np.mean(hh)),
                    min(float(np.mean(dd))/12.0,1.0),
                ]
            else:
                vals=[0.0]*8

            pocket[i,c:c+8]=vals
            c+=8

        def nearest(chars):
            ids=[
                j for j,x in enumerate(seq)
                if x in chars and j!=i
            ]

            if not ids:
                return 1.0

            d=np.linalg.norm(
                coords[ids]-coords[i][None,:],
                axis=1
            )

            return min(float(d.min())/20.0,1.0)

        pocket[i,32]=nearest(GROUPS['AROM'])
        pocket[i,33]=nearest(GROUPS['SULFUR'])
        pocket[i,34]=nearest(GROUPS['POLAR'])
        pocket[i,35]=nearest(GROUPS['HYDRO'])

        # Shell occupancy: packing/cavity proxy.
        d_all=np.linalg.norm(
            coords-coords[i][None,:],
            axis=1
        )

        shell_counts=[
            np.sum((d_all>0.1)&(d_all<=4.0)),
            np.sum((d_all>4.0)&(d_all<=6.0)),
            np.sum((d_all>6.0)&(d_all<=8.0)),
            np.sum((d_all>8.0)&(d_all<=10.0)),
        ]

        pocket[i,36:40]=np.clip(
            np.asarray(shell_counts,dtype=np.float32)/20.0,
            0,1
        )

        ids=np.asarray(neigh8[i],dtype=int)
        ids=ids[ids!=i]

        if len(ids)>=3:
            cc=coords[ids]-coords[ids].mean(axis=0,keepdims=True)
            cov=cc.T@cc/max(len(ids)-1,1)
            eig=np.sort(
                np.clip(np.linalg.eigvalsh(cov),0,None)
            )[::-1]

            total=float(eig.sum()+1e-8)

            linearity=(eig[0]-eig[1])/(eig[0]+1e-8)
            planarity=(eig[1]-eig[2])/(eig[0]+1e-8)
            isotropy=eig[2]/total

            pocket[i,40]=np.clip(linearity,0,1)
            pocket[i,41]=np.clip(planarity,0,1)
            pocket[i,42]=np.clip(isotropy,0,1)

            seqsep=np.abs(ids-i)
            pocket[i,43]=float(np.mean(seqsep>10))

        # ----------------------------------------------------
        # SM_TOPO16
        # ----------------------------------------------------
        n6=[j for j in neigh6[i] if j!=i]
        n8=[j for j in neigh8[i] if j!=i]
        n10=[j for j in neigh10[i] if j!=i]

        topo[i,0]=min(len(n6),25)/25.0
        topo[i,1]=min(len(n8),35)/35.0
        topo[i,2]=min(len(n10),45)/45.0

        a8=[j for j in n8 if seq[j] in GROUPS['AROM']]
        h8=[j for j in n8 if seq[j] in GROUPS['HYDRO']]
        s8=[j for j in n8 if seq[j] in GROUPS['SULFUR']]
        p8=[j for j in n8 if seq[j] in GROUPS['POLAR']]
        d8=[j for j in n8 if seq[j] in GROUPS['ACIDIC']]

        topo[i,3]=min(len(a8),12)/12.0
        topo[i,4]=min(len(h8),20)/20.0
        topo[i,5]=min(len(s8),8)/8.0
        topo[i,6]=min(len(p8),20)/20.0
        topo[i,7]=min(len(d8),15)/15.0

        topo[i,8]=min(arom_comp.get(i,0),15)/15.0
        topo[i,9]=min(hydro_comp.get(i,0),25)/25.0

        if len(n8)>=2:
            cc=coords[n8]
            t2=cKDTree(cc)
            edges=len(t2.query_pairs(8.0))
            possible=len(n8)*(len(n8)-1)/2
            topo[i,10]=edges/max(possible,1)

            seqsep=np.abs(np.asarray(n8)-i)
            topo[i,11]=float(np.mean(seqsep>10))
            topo[i,12]=min(float(np.mean(seqsep))/50.0,1.0)

            comp=Counter(seq[j] for j in n8)
            probs=np.asarray(
                [x/len(n8) for x in comp.values()],
                dtype=float
            )
            topo[i,13]=float(
                -np.sum(probs*np.log(probs+1e-12))/math.log(20)
            )

        # inner/outer packing balance
        topo[i,14]=np.tanh(
            (len(n6)-(len(n10)-len(n6)))/10.0
        )

        # hydrophobic/aromatic density vs acidic density
        topo[i,15]=np.tanh(
            (len(h8)+len(a8)-len(d8))/10.0
        )

        # ----------------------------------------------------
        # SM_AROMSTACK12
        # ----------------------------------------------------
        arom[i,0]=float(a in GROUPS['AROM'])
        arom[i,1]=float(a=='W')
        arom[i,2]=float(a=='F')
        arom[i,3]=float(a=='Y')
        arom[i,4]=float(a=='H')

        for col2,radius in zip([5,6,7],[6.0,8.0,10.0]):
            ids=[
                j for j in tree.query_ball_point(coords[i],radius)
                if j!=i
            ]

            arom[i,col2]=(
                sum(seq[j] in GROUPS['AROM'] for j in ids)
                / max(len(ids),1)
            )

        arom[i,8]=nearest(GROUPS['AROM'])
        arom[i,9]=min(arom_comp.get(i,0),15)/15.0

        # Pairwise aromatic orientation coherence in 10 Å neighborhood.
        ar_nei=[
            j for j in neigh10[i]
            if j in ring_normal and j!=i
        ]

        dots=[]

        for a1 in range(len(ar_nei)):
            for a2 in range(a1+1,len(ar_nei)):
                dots.append(
                    abs(float(np.dot(
                        ring_normal[ar_nei[a1]],
                        ring_normal[ar_nei[a2]]
                    )))
                )

        arom[i,10]=float(np.mean(dots)) if dots else 0.0

        if ar_nei:
            cent=np.mean(
                np.stack([ring_centroid[j] for j in ar_nei]),
                axis=0
            )

            arom[i,11]=min(
                float(np.linalg.norm(coords[i]-cent))/20.0,
                1.0
            )

    return pocket,ligchem,topo,arom

def protein_context(seq):
    L=len(seq)

    p=np.array([
        math.log1p(L)/8.0,
        sum(a in GROUPS['AROM'] for a in seq)/L,
        sum(a in GROUPS['HYDRO'] for a in seq)/L,
        sum(a in GROUPS['SULFUR'] for a in seq)/L,
        sum(a in GROUPS['POLAR'] for a in seq)/L,
        sum(a in GROUPS['ACIDIC'] for a in seq)/L,
        sum(a in GROUPS['GP'] for a in seq)/L,
        float(np.mean([
            HYDROPATHY.get(a,0.0)/4.5
            for a in seq
        ])),
    ],dtype=np.float32)

    return np.repeat(
        p[None,:],
        L,
        axis=0
    )


In [ ]:
# ============================================================
# 7. GENERATE / CACHE / ATTACH SMALL-SPECIFIC BLOCKS
# ============================================================
NEW_BLOCKS = {
    'SM_SEQCHEM32':32,
    'SM_POCKET3D44':44,
    'SM_LIGCHEM20':20,
    'SM_TOPO16':16,
    'SM_AROMSTACK12':12,
    'SM_PROTCTX8':8,
}

if RUN_FEATURE_GENERATION:
    for split,df in SPLIT_DF.items():
        outdir = FEATURE_CACHE/split
        outdir.mkdir(parents=True,exist_ok=True)

        for n,r in enumerate(df.itertuples(),start=1):
            fp = outdir/f'{r.prot_id}.npz'

            valid=False

            if fp.exists() and not FORCE_REBUILD_SM_FEATURES:
                try:
                    z=np.load(fp)

                    valid=all(
                        z[k].shape==(len(r.sequence),d)
                        for k,d in NEW_BLOCKS.items()
                    )
                except Exception:
                    valid=False

            if not valid:
                seqb=sequence_chem_block(r.sequence)

                pocket,ligchem,topo,arom=build_small_3d_blocks(
                    PDB_NORM_ROOT/split/f'{r.prot_id}.pdb',
                    r.sequence
                )

                prot=protein_context(r.sequence)

                np.savez_compressed(
                    fp,
                    SM_SEQCHEM32=seqb,
                    SM_POCKET3D44=pocket,
                    SM_LIGCHEM20=ligchem,
                    SM_TOPO16=topo,
                    SM_AROMSTACK12=arom,
                    SM_PROTCTX8=prot,
                )

            if n%250==0:
                print(split,n,'/',len(df))

for split,df in SPLIT_DF.items():
    store=ctx.stores[STORE_KEY[split]]
    outdir=FEATURE_CACHE/split

    for r in df.itertuples():
        z=np.load(outdir/f'{r.prot_id}.npz')

        for k,d in NEW_BLOCKS.items():
            a=z[k].astype(np.float32,copy=False)

            if a.shape!=(len(r.sequence),d):
                raise RuntimeError(
                    f'{split}/{r.prot_id}/{k}: {a.shape}'
                )

            set_block(
                store,
                r.prot_id,
                k,
                a
            )

print('New small-binding blocks attached.')

display(
    reporting.dimension_table(
        ctx.stores['train']
    )
)

audit_blocks = [
    'ESM',
    'ACCESS','ATOM','CHEM','TOPO','SOLV',
    'SPATIAL','LOCAL','SEQMOTIF',
    *NEW_BLOCKS.keys(),
]

if EVO_AVAILABLE:
    audit_blocks.append('EVO')

if ESMIF_AVAILABLE:
    audit_blocks.append('ESMIF')

display(
    reporting.validate_no_direct_label_features(
        ctx.stores,
        audit_blocks
    )
)


DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300
New small-binding blocks attached.


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
16,EVO,48
3,LOCAL,8


,split,block,missing_proteins,mean_finite_fraction
0,train,ESM,0,1.0
1,train,ACCESS,0,1.0
2,train,ATOM,0,1.0
3,train,CHEM,0,1.0
4,train,TOPO,0,1.0
5,train,SOLV,0,1.0
6,train,SPATIAL,0,1.0
7,train,LOCAL,0,1.0
8,train,SEQMOTIF,0,1.0
9,train,SM_SEQCHEM32,0,1.0


# Part B — Same-MLP feature/mechanism discovery

All representation experiments use the same GEAR MLP and the same official five protein folds.

This separates representation gain from architecture gain.


In [ ]:
# ============================================================
# 8. SINGLE-FAMILY MLP SCREEN — DEV ONLY
# ============================================================
FAMILY_GRID = {
    'ESM':['ESM'],

    # Existing audit-supported controls
    'ESM+ACCESS':['ESM','ACCESS'],
    'ESM+TOPO':['ESM','TOPO'],
    'ESM+ATOM':['ESM','ATOM'],
    'ESM+CHEM':['ESM','CHEM'],
    'ESM+SOLV':['ESM','SOLV'],
    'ESM+SEQMOTIF':['ESM','SEQMOTIF'],
    'ESM+Structure':['ESM','SPATIAL','LOCAL'],

    # New small-specific blocks
    'ESM+SM_SEQCHEM':['ESM','SM_SEQCHEM32'],
    'ESM+SM_POCKET3D':['ESM','SM_POCKET3D44'],
    'ESM+SM_LIGCHEM':['ESM','SM_LIGCHEM20'],
    'ESM+SM_TOPO':['ESM','SM_TOPO16'],
    'ESM+SM_AROMSTACK':['ESM','SM_AROMSTACK12'],
    'ESM+SM_PROTCTX':['ESM','SM_PROTCTX8'],
}

if EVO_AVAILABLE:
    FAMILY_GRID['ESM+EVO']=['ESM','EVO']

if ESMIF_AVAILABLE:
    FAMILY_GRID['ESM+ESMIF']=['ESM','ESMIF']

if RUN_FAMILY_SCREEN:
    family_cv,family_folds=suite.run_dev_cv_only(
        ctx,
        FAMILY_GRID,
        models=('MLP',),
        run_version='small_v1_01_family_mlp',
    )

    family_cv.to_csv(
        RUN_ROOT/'family_cv.csv',
        index=False
    )

    family_folds.to_csv(
        RUN_ROOT/'family_folds.csv',
        index=False
    )

else:
    family_cv=pd.read_csv(
        RUN_ROOT/'family_cv.csv'
    )

    family_folds=pd.read_csv(
        RUN_ROOT/'family_folds.csv'
    )

base_score=float(
    family_cv.loc[
        family_cv.representation.eq('ESM'),
        'CV_AUPRC'
    ].iloc[0]
)

family_rank=family_cv.copy()
family_rank['delta_vs_ESM']=family_rank.CV_AUPRC-base_score
family_rank=family_rank.sort_values(
    'CV_AUPRC',
    ascending=False
)

display(family_rank)

family_rank.to_csv(
    RUN_ROOT/'family_ranked.csv',
    index=False
)

print('ESM baseline:',base_score)


REUSE MLP ESM 1
REUSE MLP ESM 2
REUSE MLP ESM 3
REUSE MLP ESM 4
REUSE MLP ESM 5
REUSE MLP ESM+ACCESS 1
REUSE MLP ESM+ACCESS 2
REUSE MLP ESM+ACCESS 3
REUSE MLP ESM+ACCESS 4
REUSE MLP ESM+ACCESS 5
REUSE MLP ESM+TOPO 1
REUSE MLP ESM+TOPO 2
REUSE MLP ESM+TOPO 3
REUSE MLP ESM+TOPO 4
REUSE MLP ESM+TOPO 5
REUSE MLP ESM+ATOM 1
REUSE MLP ESM+ATOM 2
REUSE MLP ESM+ATOM 3
REUSE MLP ESM+ATOM 4
REUSE MLP ESM+ATOM 5
REUSE MLP ESM+CHEM 1
REUSE MLP ESM+CHEM 2
REUSE MLP ESM+CHEM 3
REUSE MLP ESM+CHEM 4
REUSE MLP ESM+CHEM 5
REUSE MLP ESM+SOLV 1
REUSE MLP ESM+SOLV 2
REUSE MLP ESM+SOLV 3
REUSE MLP ESM+SOLV 4
REUSE MLP ESM+SOLV 5
REUSE MLP ESM+SEQMOTIF 1
REUSE MLP ESM+SEQMOTIF 2
REUSE MLP ESM+SEQMOTIF 3
REUSE MLP ESM+SEQMOTIF 4
REUSE MLP ESM+SEQMOTIF 5
REUSE MLP ESM+Structure 1
REUSE MLP ESM+Structure 2
REUSE MLP ESM+Structure 3
REUSE MLP ESM+Structure 4
REUSE MLP ESM+Structure 5
REUSE MLP ESM+SM_SEQCHEM 1
REUSE MLP ESM+SM_SEQCHEM 2
REUSE MLP ESM+SM_SEQCHEM 3
REUSE MLP ESM+SM_SEQCHEM 4
REUSE MLP ESM+SM_SEQCH

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_ESM
8,MLP,ESM+SM_POCKET3D,0.366919,0.026637,0.849958,0.014745
13,MLP,ESM+Structure,0.362243,0.029628,0.846080,0.010070
1,MLP,ESM+ACCESS,0.361371,0.027588,0.851776,0.009197
14,MLP,ESM+TOPO,0.361230,0.031229,0.849949,0.009056
3,MLP,ESM+CHEM,0.360974,0.027618,0.848286,0.008800
9,MLP,ESM+SM_PROTCTX,0.358490,0.025384,0.848983,0.006316
7,MLP,ESM+SM_LIGCHEM,0.357130,0.024493,0.836049,0.004957
11,MLP,ESM+SM_TOPO,0.356375,0.026696,0.847231,0.004201
12,MLP,ESM+SOLV,0.356209,0.024483,0.845769,0.004035
5,MLP,ESM+SEQMOTIF,0.355184,0.020823,0.845086,0.003011


ESM baseline: 0.35217369108912633


In [ ]:
# ============================================================
# 9. MECHANISM COMBINATION LADDER — DEV ONLY
# ============================================================
MECHANISM_GRID = {
    'ESM':['ESM'],

    # Audit-derived existing compact control.
    'ESM+ACCESS+TOPO':
        ['ESM','ACCESS','TOPO'],

    # Pocket shape / accessibility.
    'ESM+POCKET_GEOMETRY':
        [
            'ESM',
            'ACCESS',
            'SM_POCKET3D44',
            'SM_TOPO16',
        ],

    # Aromatic + hydrophobic + sulfur microenvironment.
    'ESM+AROM_HYDRO':
        [
            'ESM',
            'SM_SEQCHEM32',
            'SM_AROMSTACK12',
        ],

    # Interaction chemistry / solvation.
    'ESM+LIGAND_CHEMISTRY':
        [
            'ESM',
            'CHEM',
            'SOLV',
            'SM_LIGCHEM20',
        ],

    # Main small-molecule hypothesis.
    'ESM+POCKET_CHEM':
        [
            'ESM',
            'ACCESS',
            'SM_POCKET3D44',
            'SM_TOPO16',
            'SM_SEQCHEM32',
            'SM_AROMSTACK12',
            'SM_LIGCHEM20',
        ],

    'ESM+POCKET_CHEM+SOLV':
        [
            'ESM',
            'ACCESS','SOLV',
            'SM_POCKET3D44','SM_TOPO16',
            'SM_SEQCHEM32','SM_AROMSTACK12',
            'SM_LIGCHEM20',
        ],

    # Existing ATOM/CHEM are transfer controls, not the main ontology.
    'ESM+POCKET_CHEM+ATOM_CHEM_CONTROL':
        [
            'ESM',
            'ACCESS','SOLV',
            'ATOM','CHEM',
            'SM_POCKET3D44','SM_TOPO16',
            'SM_SEQCHEM32','SM_AROMSTACK12',
            'SM_LIGCHEM20',
        ],
}

if EVO_AVAILABLE:
    MECHANISM_GRID['ESM+POCKET_CHEM+EVO_CONTROL']=[
        'ESM',
        'ACCESS',
        'SM_POCKET3D44','SM_TOPO16',
        'SM_SEQCHEM32','SM_AROMSTACK12',
        'SM_LIGCHEM20',
        'EVO',
    ]

if ESMIF_AVAILABLE:
    MECHANISM_GRID['ESM+POCKET_CHEM+ESMIF']=[
        'ESM',
        'ACCESS',
        'SM_POCKET3D44','SM_TOPO16',
        'SM_SEQCHEM32','SM_AROMSTACK12',
        'SM_LIGCHEM20',
        'ESMIF',
    ]

if RUN_COMBINATION_LADDER:
    combo_cv,combo_folds=suite.run_dev_cv_only(
        ctx,
        MECHANISM_GRID,
        models=('MLP',),
        run_version='small_v1_02_mechanism_combo_mlp',
    )

    combo_cv.to_csv(
        RUN_ROOT/'mechanism_combo_cv.csv',
        index=False
    )

    combo_folds.to_csv(
        RUN_ROOT/'mechanism_combo_folds.csv',
        index=False
    )

else:
    combo_cv=pd.read_csv(
        RUN_ROOT/'mechanism_combo_cv.csv'
    )

    combo_folds=pd.read_csv(
        RUN_ROOT/'mechanism_combo_folds.csv'
    )

combo_rank=combo_cv.sort_values(
    'CV_AUPRC',
    ascending=False
).reset_index(drop=True)

display(combo_rank)

BEST_REP_NAME=str(
    combo_rank.iloc[0].representation
)

BEST_REP_BLOCKS=list(
    MECHANISM_GRID[BEST_REP_NAME]
)

BEST_REP_SCORE=float(
    combo_rank.iloc[0].CV_AUPRC
)

print('LOCKED REPRESENTATION:',BEST_REP_NAME)
print('Blocks:',BEST_REP_BLOCKS)
print('Dev CV AUPRC:',BEST_REP_SCORE)

(RUN_ROOT/'locked_representation.json').write_text(
    json.dumps({
        'representation':BEST_REP_NAME,
        'blocks':BEST_REP_BLOCKS,
        'dev_cv_auprc':BEST_REP_SCORE,
        'selection_source':'DevSet1010 only',
    },indent=2)
)


REUSE MLP ESM 1
REUSE MLP ESM 2
REUSE MLP ESM 3
REUSE MLP ESM 4
REUSE MLP ESM 5
REUSE MLP ESM+ACCESS+TOPO 1
REUSE MLP ESM+ACCESS+TOPO 2
REUSE MLP ESM+ACCESS+TOPO 3
REUSE MLP ESM+ACCESS+TOPO 4
REUSE MLP ESM+ACCESS+TOPO 5
REUSE MLP ESM+POCKET_GEOMETRY 1
REUSE MLP ESM+POCKET_GEOMETRY 2
REUSE MLP ESM+POCKET_GEOMETRY 3
REUSE MLP ESM+POCKET_GEOMETRY 4
REUSE MLP ESM+POCKET_GEOMETRY 5
REUSE MLP ESM+AROM_HYDRO 1
REUSE MLP ESM+AROM_HYDRO 2
REUSE MLP ESM+AROM_HYDRO 3
REUSE MLP ESM+AROM_HYDRO 4
REUSE MLP ESM+AROM_HYDRO 5
REUSE MLP ESM+LIGAND_CHEMISTRY 1
REUSE MLP ESM+LIGAND_CHEMISTRY 2
REUSE MLP ESM+LIGAND_CHEMISTRY 3
REUSE MLP ESM+LIGAND_CHEMISTRY 4
REUSE MLP ESM+LIGAND_CHEMISTRY 5
REUSE MLP ESM+POCKET_CHEM 1
REUSE MLP ESM+POCKET_CHEM 2
REUSE MLP ESM+POCKET_CHEM 3
REUSE MLP ESM+POCKET_CHEM 4
REUSE MLP ESM+POCKET_CHEM 5
REUSE MLP ESM+POCKET_CHEM+SOLV 1
REUSE MLP ESM+POCKET_CHEM+SOLV 2
REUSE MLP ESM+POCKET_CHEM+SOLV 3
REUSE MLP ESM+POCKET_CHEM+SOLV 4
REUSE MLP ESM+POCKET_CHEM+SOLV 5
REUSE MLP ESM+P

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
0,MLP,ESM+POCKET_GEOMETRY,0.377708,0.033064,0.849750
1,MLP,ESM+POCKET_CHEM+EVO_CONTROL,0.374193,0.028098,0.856972
2,MLP,ESM+POCKET_CHEM+SOLV,0.374023,0.031997,0.854792
3,MLP,ESM+POCKET_CHEM+ATOM_CHEM_CONTROL,0.372913,0.028648,0.853947
4,MLP,ESM+ACCESS+TOPO,0.372839,0.030957,0.845852
5,MLP,ESM+POCKET_CHEM,0.371453,0.032185,0.852765
6,MLP,ESM+LIGAND_CHEMISTRY,0.362327,0.024956,0.843168
7,MLP,ESM+AROM_HYDRO,0.357675,0.024084,0.848024
8,MLP,ESM,0.352174,0.033973,0.844059


LOCKED REPRESENTATION: ESM+POCKET_GEOMETRY
Blocks: ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16']
Dev CV AUPRC: 0.3777080103628627


205

# Part C — Multimodal architecture screen

After the feature representation is frozen, the notebook tests whether the way modalities are fused matters.

Semantic modalities:

- **ESM** — sequence PLM
- **POCKET** — accessibility + pocket geometry + local topology
- **CHEM** — H-bond / solvation / explicit ligand-contact chemistry
- **AROM_HYDRO** — aromatic/hydrophobic/sulfur context
- **BIO** — optional EVO / ESM-IF / sequence motif / protein context

Architectures:

- ConcatMLP
- ProjectedConcat
- GatedFusion
- ModalityAttention
- PocketResidualFusion
- LatentLigandMoE

`LatentLigandMoE` tests the plausible heterogeneity of small-molecule pockets without requiring ligand subtype labels.


In [ ]:
# ============================================================
# 10. PACK LOCKED REPRESENTATION INTO SEMANTIC MODALITIES
# ============================================================
def modality_spec(blocks):
    blocks=list(blocks)
    spec={}

    if 'ESM' in blocks:
        spec['ESM']=['ESM']

    pocket=[
        b for b in [
            'ACCESS','TOPO',
            'SM_POCKET3D44','SM_TOPO16'
        ]
        if b in blocks
    ]
    if pocket:
        spec['POCKET']=pocket

    chem=[
        b for b in [
            'CHEM','SOLV','ATOM',
            'SM_LIGCHEM20'
        ]
        if b in blocks
    ]
    if chem:
        spec['CHEM']=chem

    arom=[
        b for b in [
            'SM_SEQCHEM32',
            'SM_AROMSTACK12'
        ]
        if b in blocks
    ]
    if arom:
        spec['AROM_HYDRO']=arom

    bio=[
        b for b in [
            'EVO','ESMIF','SEQMOTIF',
            'SM_PROTCTX8','SPATIAL','LOCAL'
        ]
        if b in blocks
    ]
    if bio:
        spec['BIO']=bio

    assigned={x for v in spec.values() for x in v}
    other=[b for b in blocks if b not in assigned]

    if other:
        spec['OTHER']=other

    return spec

MODALITY_SPEC=modality_spec(BEST_REP_BLOCKS)

print('MODALITY_SPEC =',MODALITY_SPEC)

def concat_blocks(store,pid,blocks):
    xs=[
        get_block(store,pid,b).astype(np.float32,copy=False)
        for b in blocks
    ]

    lens={x.shape[0] for x in xs}

    if len(lens)!=1:
        raise RuntimeError(f'{pid}: block length mismatch')

    return np.concatenate(xs,axis=1)

def build_pack(split):
    df=SPLIT_DF[split]
    store=ctx.stores[STORE_KEY[split]]

    mods={k:[] for k in MODALITY_SPEC}
    yy=[]
    pids=[]
    ridx=[]

    for r in df.itertuples():
        L=len(r.sequence)

        for m,blocks in MODALITY_SPEC.items():
            mods[m].append(
                concat_blocks(
                    store,
                    r.prot_id,
                    blocks
                )
            )

        yy.append(
            np.asarray(
                r.small_binding,
                dtype=np.float32
            )
        )

        pids.extend([str(r.prot_id)]*L)
        ridx.extend(range(L))

    mods={
        k:np.concatenate(v,axis=0).astype(
            np.float16 if k in {'ESM','BIO'} else np.float32,
            copy=False
        )
        for k,v in mods.items()
    }

    return {
        'mods':mods,
        'y':np.concatenate(yy).astype(np.float32),
        'pids':np.asarray(pids,dtype=object),
        'ridx':np.asarray(ridx,dtype=np.int32),
    }

PACKS={
    s:build_pack(s)
    for s in SPLIT_DF
}

DIMS={
    k:v.shape[1]
    for k,v in PACKS['DevSet1010']['mods'].items()
}

for split,p in PACKS.items():
    print(
        split,
        {k:v.shape for k,v in p['mods'].items()},
        'y=',p['y'].shape,
        'positive=',int(p['y'].sum())
    )

print('Modality dims:',DIMS)


MODALITY_SPEC = {'ESM': ['ESM'], 'POCKET': ['ACCESS', 'SM_POCKET3D44', 'SM_TOPO16']}
DevSet1010 {'ESM': (170181, 1280), 'POCKET': (170181, 70)} y= (170181,) positive= 9236
TestSet300 {'ESM': (62689, 1280), 'POCKET': (62689, 70)} y= (62689,) positive= 3906
TestSetNew46 {'ESM': (6227, 1280), 'POCKET': (6227, 70)} y= (6227,) positive= 425
Modality dims: {'ESM': 1280, 'POCKET': 70}


In [ ]:
# ============================================================
# 11. MULTIMODAL MODELS
# ============================================================
class ConcatMLP(nn.Module):
    def __init__(self,dims,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.norm=nn.ModuleDict({
            k:nn.LayerNorm(d)
            for k,d in dims.items()
        })

        d=sum(dims.values())

        self.head=nn.Sequential(
            nn.Linear(d,512),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(512,128),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(128,1),
        )

    def forward(self,x):
        z=torch.cat(
            [
                self.norm[k](x[k])
                for k in self.keys
            ],
            dim=-1
        )

        return self.head(z).squeeze(-1)

class ProjectedConcat(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.head=nn.Sequential(
            nn.Linear(
                latent*len(dims),
                256
            ),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(256,1)
        )

    def forward(self,x):
        z=torch.cat(
            [
                self.enc[k](x[k])
                for k in self.keys
            ],
            dim=-1
        )

        return self.head(z).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def encode(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.gate[k](z))

        Z=torch.stack(zz,dim=1)

        G=torch.softmax(
            torch.cat(gg,dim=1),
            dim=1
        ).unsqueeze(-1)

        return (Z*G).sum(dim=1),G.squeeze(-1)

    def forward(self,x):
        h,_=self.encode(x)
        return self.head(h).squeeze(-1)

class ModalityAttention(nn.Module):
    def __init__(self,dims,latent=192,heads=6,drop=0.2):
        super().__init__()

        assert latent%heads==0

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.attn=nn.MultiheadAttention(
            latent,
            heads,
            dropout=drop,
            batch_first=True
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        Z=torch.stack(
            [
                self.enc[k](x[k])
                for k in self.keys
            ],
            dim=1
        )

        A,_=self.attn(
            Z,Z,Z,
            need_weights=False
        )

        h=(Z+A).mean(dim=1)

        return self.head(h).squeeze(-1)

class PocketResidualFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        if 'POCKET' not in dims:
            raise ValueError(
                'PocketResidualFusion requires POCKET modality'
            )

        self.base_keys=sorted(
            [
                k for k in dims
                if k!='POCKET'
            ]
        )

        self.base_enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
            if k!='POCKET'
        })

        self.pocket_enc=nn.Sequential(
            nn.LayerNorm(dims['POCKET']),
            nn.Linear(dims['POCKET'],latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent),
        )

        self.base_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in self.base_keys
        })

        self.alpha_logit=nn.Parameter(
            torch.tensor(
                -2.944,
                dtype=torch.float32
            )
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.base_keys:
            z=self.base_enc[k](x[k])
            zz.append(z)
            gg.append(self.base_gate[k](z))

        Z=torch.stack(zz,dim=1)

        G=torch.softmax(
            torch.cat(gg,dim=1),
            dim=1
        ).unsqueeze(-1)

        h=(Z*G).sum(dim=1)

        pocket=self.pocket_enc(
            x['POCKET']
        )

        alpha=torch.sigmoid(
            self.alpha_logit
        )

        h=h+alpha*pocket

        return self.head(h).squeeze(-1)

class LatentLigandMoE(nn.Module):
    def __init__(self,dims,latent=192,experts=4,drop=0.2):
        super().__init__()

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.mod_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.shared=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,latent),
            nn.GELU()
        )

        self.experts=nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent,128),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(128,1)
            )
            for _ in range(experts)
        ])

        self.expert_gate=nn.Linear(
            latent,
            experts
        )

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.mod_gate[k](z))

        Z=torch.stack(zz,dim=1)

        G=torch.softmax(
            torch.cat(gg,dim=1),
            dim=1
        ).unsqueeze(-1)

        h=self.shared(
            (Z*G).sum(dim=1)
        )

        logits=torch.cat(
            [
                e(h)
                for e in self.experts
            ],
            dim=1
        )

        w=torch.softmax(
            self.expert_gate(h),
            dim=1
        )

        return (
            logits*w
        ).sum(dim=1)

ARCH_FACTORY = {
    'ConcatMLP':
        lambda dims:ConcatMLP(dims),

    'ProjectedConcat':
        lambda dims:ProjectedConcat(dims),

    'GatedFusion':
        lambda dims:GatedFusion(dims),

    'ModalityAttention':
        lambda dims:ModalityAttention(dims),

    'LatentLigandMoE':
        lambda dims:LatentLigandMoE(dims),
}

if 'POCKET' in DIMS:
    ARCH_FACTORY[
        'PocketResidualFusion'
    ]=lambda dims:PocketResidualFusion(dims)

print(
    'Architectures:',
    list(ARCH_FACTORY)
)


Architectures: ['ConcatMLP', 'ProjectedConcat', 'GatedFusion', 'ModalityAttention', 'LatentLigandMoE', 'PocketResidualFusion']


In [ ]:
# ============================================================
# 12. CUSTOM CV TRAINER — OFFICIAL FOLDS, RESUME SAFE
# ============================================================
class MultiArrayDataset(Dataset):
    def __init__(self,mods,y):
        self.mods=mods
        self.y=y
        self.keys=sorted(mods)

    def __len__(self):
        return len(self.y)

    def __getitem__(self,i):
        x={
            k:torch.from_numpy(
                np.asarray(
                    self.mods[k][i],
                    dtype=np.float32
                )
            )
            for k in self.keys
        }

        return (
            x,
            torch.tensor(
                self.y[i],
                dtype=torch.float32
            )
        )

def collate_multi(batch):
    keys=sorted(batch[0][0])

    x={
        k:torch.stack(
            [b[0][k] for b in batch],
            dim=0
        )
        for k in keys
    }

    y=torch.stack(
        [b[1] for b in batch]
    )

    return x,y

def ap(y,p):
    return float(
        average_precision_score(
            y,p
        )
    )

def predict_flat(
    model,
    dataset,
    indices=None,
    batch_size=8192
):
    ds=(
        dataset
        if indices is None
        else Subset(
            dataset,
            indices.tolist()
            if isinstance(indices,np.ndarray)
            else indices
        )
    )

    loader=DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    out=[]

    model.eval()

    with torch.no_grad():
        for x,_ in loader:
            x={
                k:v.to(
                    DEVICE,
                    non_blocking=True
                )
                for k,v in x.items()
            }

            out.append(
                torch.sigmoid(
                    model(x)
                )
                .detach()
                .cpu()
                .numpy()
            )

    return np.concatenate(out)

CUSTOM_STATE_ROOT=(
    RUN_ROOT/
    'custom_states'
)

CUSTOM_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

def train_one_arch_fold(
    arch,
    pack,
    train_idx,
    val_idx,
    fold
):
    torch.manual_seed(
        SEED+fold
    )
    np.random.seed(
        SEED+fold
    )

    ds=MultiArrayDataset(
        pack['mods'],
        pack['y']
    )

    train_ds=Subset(
        ds,
        train_idx.tolist()
    )

    model=ARCH_FACTORY[
        arch
    ](DIMS).to(DEVICE)

    ytr=pack['y'][train_idx]

    npos=max(
        float(ytr.sum()),
        1.0
    )

    nneg=max(
        float(len(ytr)-ytr.sum()),
        1.0
    )

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(nneg/npos),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=CUSTOM_LR,
        weight_decay=CUSTOM_WEIGHT_DECAY
    )

    loader=DataLoader(
        train_ds,
        batch_size=CUSTOM_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(
        1,
        CUSTOM_EPOCHS+1
    ):
        model.train()
        losses=[]

        for x,y in loader:
            x={
                k:v.to(
                    DEVICE,
                    non_blocking=True
                )
                for k,v in x.items()
            }

            y=y.to(
                DEVICE,
                non_blocking=True
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits=model(x)
            loss=criterion(
                logits,y
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            opt.step()

            losses.append(
                float(
                    loss.detach().cpu()
                )
            )

        vp=predict_flat(
            model,
            ds,
            val_idx
        )

        vy=pack['y'][val_idx]

        val_ap=ap(
            vy,vp
        )

        if val_ap>best_ap+1e-5:
            best_ap=val_ap

            best_state={
                k:v.detach()
                .cpu()
                .clone()
                for k,v
                in model.state_dict().items()
            }

            bad=0

        else:
            bad+=1

        if (
            epoch==1
            or epoch%5==0
        ):
            print(
                arch,
                'fold',fold,
                'epoch',epoch,
                'loss',
                round(
                    float(np.mean(losses)),
                    4
                ),
                'val_AP',
                round(val_ap,5),
                'best',
                round(best_ap,5)
            )

        if bad>=CUSTOM_PATIENCE:
            break

    model.load_state_dict(
        best_state
    )

    return model,best_state

def run_arch_cv(arch):
    pack=PACKS[
        'DevSet1010'
    ]

    pids=pack['pids']
    y=pack['y']

    ds=MultiArrayDataset(
        pack['mods'],
        pack['y']
    )

    oof=np.full(
        len(y),
        np.nan,
        dtype=np.float32
    )

    rows=[]
    states={}

    adir=(
        CUSTOM_STATE_ROOT/
        arch
    )

    adir.mkdir(
        parents=True,
        exist_ok=True
    )

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(
            val_ids
        )

        val_idx=np.flatnonzero(
            np.array([
                p in val_set
                for p in pids
            ])
        )

        train_idx=np.flatnonzero(
            np.array([
                p not in val_set
                for p in pids
            ])
        )

        state_fp=adir/f'fold{fold}.pt'
        pred_fp=adir/f'fold{fold}_val.npy'
        meta_fp=adir/f'fold{fold}.json'

        reuse=False

        if (
            state_fp.exists()
            and pred_fp.exists()
            and meta_fp.exists()
        ):
            try:
                meta=json.loads(
                    meta_fp.read_text()
                )

                pp=np.load(
                    pred_fp
                )

                reuse=(
                    meta.get(
                        'representation'
                    )
                    ==BEST_REP_NAME
                    and meta.get(
                        'n_val'
                    )
                    ==len(val_idx)
                    and pp.shape
                    ==(len(val_idx),)
                    and np.all(
                        np.isfinite(pp)
                    )
                )

            except Exception:
                reuse=False

        if reuse:
            print(
                arch,
                'fold',
                fold,
                'REUSE'
            )

            state=torch.load(
                state_fp,
                map_location='cpu'
            )

            pred=np.load(
                pred_fp
            ).astype(
                np.float32
            )

        else:
            print(
                arch,
                'fold',
                fold,
                'TRAIN'
            )

            model,state=(
                train_one_arch_fold(
                    arch,
                    pack,
                    train_idx,
                    val_idx,
                    fold
                )
            )

            pred=predict_flat(
                model,
                ds,
                val_idx
            ).astype(
                np.float32
            )

            torch.save(
                state,
                state_fp
            )

            np.save(
                pred_fp,
                pred
            )

            meta_fp.write_text(
                json.dumps({
                    'representation':
                        BEST_REP_NAME,
                    'blocks':
                        BEST_REP_BLOCKS,
                    'n_val':
                        int(len(val_idx)),
                },indent=2)
            )

            del model

            if DEVICE.type=='cuda':
                torch.cuda.empty_cache()

        states[fold]=state
        oof[val_idx]=pred

        rows.append({
            'architecture':arch,
            'fold':fold,
            'AUPRC':ap(
                y[val_idx],
                pred
            ),
            'n_val':
                len(val_idx),
            'n_positive':
                int(
                    y[val_idx].sum()
                ),
        })

    if not np.all(
        np.isfinite(oof)
    ):
        raise RuntimeError(
            f'{arch}: incomplete OOF'
        )

    f=pd.DataFrame(
        rows
    )

    return (
        {
            'architecture':
                arch,

            'CV_AUPRC_mean':
                float(
                    f.AUPRC.mean()
                ),

            'CV_AUPRC_std':
                float(
                    f.AUPRC.std(
                        ddof=1
                    )
                ),

            'OOF_AUPRC':
                ap(
                    y,
                    oof
                ),
        },
        f,
        oof,
        states
    )


In [ ]:
# ============================================================
# 13. RUN MULTIMODAL ARCHITECTURE SCREEN — DEV ONLY
# ============================================================
ARCH_RESULTS={}
ARCH_FOLDS=[]
ARCH_OOF={}
ARCH_STATES={}

if RUN_CUSTOM_ARCHITECTURES:
    for arch in ARCH_FACTORY:
        print(
            '\n'
            +'='*80
        )
        print(
            'ARCH:',
            arch
        )
        print(
            '='*80
        )

        s,f,o,st=run_arch_cv(
            arch
        )

        ARCH_RESULTS[arch]=s
        ARCH_FOLDS.append(f)
        ARCH_OOF[arch]=o
        ARCH_STATES[arch]=st

arch_summary=pd.DataFrame(
    ARCH_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(
    drop=True
)

arch_folds=pd.concat(
    ARCH_FOLDS,
    ignore_index=True
)

arch_summary.to_csv(
    RUN_ROOT/'architecture_cv.csv',
    index=False
)

arch_folds.to_csv(
    RUN_ROOT/'architecture_folds.csv',
    index=False
)

display(
    arch_summary
)

concat_score=float(
    arch_summary.loc[
        arch_summary.architecture.eq(
            'ConcatMLP'
        ),
        'CV_AUPRC_mean'
    ].iloc[0]
)

best_arch=str(
    arch_summary.iloc[0].architecture
)

best_arch_score=float(
    arch_summary.iloc[0].CV_AUPRC_mean
)

LOCKED_ARCH=(
    best_arch
    if (
        best_arch_score
        -concat_score
        >=MIN_MEAN_AUPRC_GAIN
    )
    else 'ConcatMLP'
)

LOCKED_ARCH_SCORE=float(
    arch_summary.loc[
        arch_summary.architecture.eq(
            LOCKED_ARCH
        ),
        'CV_AUPRC_mean'
    ].iloc[0]
)

print(
    'LOCKED ARCH:',
    LOCKED_ARCH,
    LOCKED_ARCH_SCORE
)

(RUN_ROOT/'locked_architecture.json').write_text(
    json.dumps({
        'architecture':
            LOCKED_ARCH,
        'dev_cv_auprc':
            LOCKED_ARCH_SCORE,
        'concat_cv_auprc':
            concat_score,
        'best_raw_architecture':
            best_arch,
        'best_raw_cv_auprc':
            best_arch_score,
    },indent=2)
)



ARCH: ConcatMLP
ConcatMLP fold 1 REUSE
ConcatMLP fold 2 REUSE
ConcatMLP fold 3 REUSE
ConcatMLP fold 4 REUSE
ConcatMLP fold 5 REUSE

ARCH: ProjectedConcat
ProjectedConcat fold 1 REUSE
ProjectedConcat fold 2 REUSE
ProjectedConcat fold 3 REUSE
ProjectedConcat fold 4 REUSE
ProjectedConcat fold 5 REUSE

ARCH: GatedFusion
GatedFusion fold 1 REUSE
GatedFusion fold 2 REUSE
GatedFusion fold 3 REUSE
GatedFusion fold 4 REUSE
GatedFusion fold 5 REUSE

ARCH: ModalityAttention
ModalityAttention fold 1 REUSE
ModalityAttention fold 2 REUSE
ModalityAttention fold 3 REUSE
ModalityAttention fold 4 REUSE
ModalityAttention fold 5 REUSE

ARCH: LatentLigandMoE
LatentLigandMoE fold 1 REUSE
LatentLigandMoE fold 2 REUSE
LatentLigandMoE fold 3 REUSE
LatentLigandMoE fold 4 REUSE
LatentLigandMoE fold 5 REUSE

ARCH: PocketResidualFusion
PocketResidualFusion fold 1 REUSE
PocketResidualFusion fold 2 REUSE
PocketResidualFusion fold 3 REUSE
PocketResidualFusion fold 4 REUSE
PocketResidualFusion fold 5 REUSE


,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,ConcatMLP,0.377218,0.026619,0.368111
1,PocketResidualFusion,0.373952,0.025417,0.371000
2,GatedFusion,0.373736,0.026525,0.370851
3,ProjectedConcat,0.373322,0.025479,0.370883
4,LatentLigandMoE,0.373089,0.022399,0.362111
5,ModalityAttention,0.369871,0.022821,0.368575


LOCKED ARCH: ConcatMLP 0.37721812239263486


198

# Part D — Small-molecule pocket hypergraph

The generic residue graph is not assumed to be optimal.

V1 constructs candidate pocket microenvironments around residues whose chemistry is consistently enriched in the audit:

```text
W / F / Y / H / C / M
```

All spatial neighbors within 8 Å become members of a candidate pocket.

The graph is **label-independent** and acts as a small residual correction to the strong local multimodal model.

This tests whether higher-order pocket organization adds signal beyond residue-local pocket descriptors.


In [ ]:
# ============================================================
# 14. BUILD / CACHE CANDIDATE POCKET HYPERGRAPHS
# ============================================================
POCKET_CACHE=(
    CACHE_ROOT/
    'pocket_hypergraph_v1'
)

POCKET_CACHE.mkdir(
    parents=True,
    exist_ok=True
)

POCKET_SEEDS=set(
    'WFYHCM'
)

def structure_coords(
    split,
    pid,
    seq
):
    path=(
        PDB_NORM_ROOT/
        split/
        f'{pid}.pdb'
    )

    st=PDBParser(
        QUIET=True
    ).get_structure(
        'p',
        str(path)
    )

    m=next(
        st.get_models()
    )

    rr=standard_residues(
        m['A']
    )

    if (
        residue_sequence(rr)
        !=seq
    ):
        raise RuntimeError(
            f'{split}/{pid}: '
            'normalized PDB mismatch'
        )

    return np.stack(
        [
            representative_coord(r)
            for r in rr
        ],
        axis=0
    ).astype(
        np.float32
    )

def pocket_static_features(
    seq,
    coords,
    members
):
    members=np.asarray(
        members,
        dtype=int
    )

    aa=[
        seq[i]
        for i in members
    ]

    cc=coords[members]

    n=len(
        members
    )

    centroid=cc.mean(
        axis=0
    )

    radial=np.linalg.norm(
        cc-centroid[None,:],
        axis=1
    )

    pair=[]

    if n>=2:
        for i in range(n):
            for j in range(i+1,n):
                pair.append(
                    float(
                        np.linalg.norm(
                            cc[i]-cc[j]
                        )
                    )
                )

    pair=np.asarray(
        pair
        if pair
        else [0.0],
        dtype=np.float32
    )

    hydro=np.asarray(
        [
            HYDROPATHY.get(a,0.0)/4.5
            for a in aa
        ],
        dtype=np.float32
    )

    comp=Counter(
        aa
    )

    probs=np.asarray(
        [
            x/n
            for x in comp.values()
        ],
        dtype=np.float32
    )

    entropy=float(
        -np.sum(
            probs*np.log(
                probs+1e-12
            )
        )
        / math.log(20)
    )

    f=np.zeros(
        22,
        dtype=np.float32
    )

    # Composition
    f[0]=min(n,40)/40.0

    f[1]=(
        sum(
            x in GROUPS['AROM']
            for x in aa
        )
        /n
    )

    f[2]=(
        sum(
            x in GROUPS['SULFUR']
            for x in aa
        )
        /n
    )

    f[3]=(
        sum(
            x in GROUPS['HYDRO']
            for x in aa
        )
        /n
    )

    f[4]=(
        sum(
            x in GROUPS['POLAR']
            for x in aa
        )
        /n
    )

    f[5]=(
        sum(
            x in GROUPS['ACIDIC']
            for x in aa
        )
        /n
    )

    f[6]=(
        sum(
            x in GROUPS['HBD']
            for x in aa
        )
        /n
    )

    f[7]=(
        sum(
            x in GROUPS['HBA']
            for x in aa
        )
        /n
    )

    f[8]=float(
        np.mean(hydro)
    )

    f[9]=(
        sum(
            x in GROUPS['GP']
            for x in aa
        )
        /n
    )

    # Geometry
    f[10]=min(
        float(
            radial.mean()
        )/12.0,
        1.0
    )

    f[11]=min(
        float(
            radial.std()
        )/8.0,
        1.0
    )

    f[12]=min(
        float(
            radial.max()
        )/15.0,
        1.0
    )

    f[13]=min(
        float(
            pair.mean()
        )/15.0,
        1.0
    )

    f[14]=min(
        float(
            pair.std()
        )/10.0,
        1.0
    )

    f[15]=min(
        (
            members.max()
            -members.min()
        )/100.0,
        1.0
    )

    f[16]=entropy

    # Tertiary-contact fraction
    if n>=2:
        seqsep=[]

        for i in range(n):
            for j in range(i+1,n):
                seqsep.append(
                    abs(
                        int(members[i])
                        -int(members[j])
                    )
                )

        seqsep=np.asarray(
            seqsep,
            dtype=float
        )

        f[17]=float(
            np.mean(
                seqsep>10
            )
        )

        f[18]=min(
            float(
                seqsep.mean()
            )/50.0,
            1.0
        )

    # PCA shape
    if n>=3:
        centered=(
            cc
            -cc.mean(
                axis=0,
                keepdims=True
            )
        )

        cov=(
            centered.T
            @centered
            /max(n-1,1)
        )

        eig=np.sort(
            np.clip(
                np.linalg.eigvalsh(
                    cov
                ),
                0,
                None
            )
        )[::-1]

        total=float(
            eig.sum()
            +1e-8
        )

        f[19]=np.clip(
            (
                eig[0]-eig[1]
            )/(eig[0]+1e-8),
            0,
            1
        )

        f[20]=np.clip(
            (
                eig[1]-eig[2]
            )/(eig[0]+1e-8),
            0,
            1
        )

        f[21]=np.clip(
            eig[2]/total,
            0,
            1
        )

    return f

def build_candidate_pockets(
    split,
    pid,
    seq
):
    coords=structure_coords(
        split,
        pid,
        seq
    )

    tree=cKDTree(
        coords
    )

    candidates={}

    for seed,a in enumerate(seq):
        if a not in POCKET_SEEDS:
            continue

        members=tuple(
            sorted(
                tree.query_ball_point(
                    coords[seed],
                    POCKET_RADIUS_A
                )
            )
        )

        if len(members)<4:
            continue

        feat=pocket_static_features(
            seq,
            coords,
            members
        )

        # Label-independent pocket plausibility:
        # enriched chemistry + compact local packing
        # with a mild penalty for strongly acidic environments.
        score=(
            1.4*feat[1]
            +0.8*feat[2]
            +0.8*feat[3]
            +0.5*feat[6]
            +0.5*feat[7]
            -0.5*feat[5]
            +0.25*feat[0]
            +0.15*feat[17]
        )

        old=candidates.get(
            members
        )

        if (
            old is None
            or score>old[0]
        ):
            candidates[
                members
            ]=(
                score,
                feat
            )

    ranked=sorted(
        candidates.items(),
        key=lambda kv:
            kv[1][0],
        reverse=True
    )[
        :POCKET_MAX_PER_PROTEIN
    ]

    edge_res=[]
    edge_pocket=[]
    pocket_feat=[]

    for pidx,(
        members,
        (_,feat)
    ) in enumerate(
        ranked
    ):
        pocket_feat.append(
            feat
        )

        for r in members:
            edge_res.append(
                int(r)
            )

            edge_pocket.append(
                int(pidx)
            )

    return (
        np.asarray(
            edge_res,
            dtype=np.int64
        ),
        np.asarray(
            edge_pocket,
            dtype=np.int64
        ),
        np.asarray(
            pocket_feat,
            dtype=np.float32
        ).reshape(
            -1,
            22
        )
    )

POCKET_RAM={}
POCKET_AUDIT=[]

for split,df in SPLIT_DF.items():
    outdir=(
        POCKET_CACHE/
        split
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    for n,r in enumerate(
        df.itertuples(),
        start=1
    ):
        fp=(
            outdir/
            f'{r.prot_id}.npz'
        )

        valid=False

        if (
            fp.exists()
            and not FORCE_REBUILD_POCKETS
        ):
            try:
                z=np.load(fp)

                valid=(
                    z['pocket_feat'].ndim
                    ==2
                    and z['pocket_feat'].shape[1]
                    ==22
                    and len(z['edge_res'])
                    ==len(z['edge_pocket'])
                )

            except Exception:
                valid=False

        if not valid:
            er,ep,pf=build_candidate_pockets(
                split,
                str(r.prot_id),
                r.sequence
            )

            np.savez_compressed(
                fp,
                edge_res=er,
                edge_pocket=ep,
                pocket_feat=pf
            )

        z=np.load(fp)

        er=z[
            'edge_res'
        ].astype(
            np.int64
        )

        ep=z[
            'edge_pocket'
        ].astype(
            np.int64
        )

        pf=z[
            'pocket_feat'
        ].astype(
            np.float32
        )

        POCKET_RAM[
            (
                split,
                str(r.prot_id)
            )
        ]=(
            er,
            ep,
            pf
        )

        covered=(
            len(
                np.unique(er)
            )
            if len(er)
            else 0
        )

        POCKET_AUDIT.append({
            'dataset':
                split,

            'prot_id':
                str(r.prot_id),

            'n_residues':
                len(r.sequence),

            'n_candidate_pockets':
                len(pf),

            'n_memberships':
                len(er),

            'covered_residue_fraction':
                covered
                /max(
                    len(r.sequence),
                    1
                ),
        })

        if n%250==0:
            print(
                split,
                n,
                '/',
                len(df)
            )

pocket_audit=pd.DataFrame(
    POCKET_AUDIT
)

pocket_audit.to_csv(
    RUN_ROOT/'pocket_audit.csv',
    index=False
)

display(
    pocket_audit.groupby(
        'dataset'
    ).agg(
        proteins=(
            'prot_id',
            'nunique'
        ),
        mean_pockets=(
            'n_candidate_pockets',
            'mean'
        ),
        median_pockets=(
            'n_candidate_pockets',
            'median'
        ),
        mean_memberships=(
            'n_memberships',
            'mean'
        ),
        mean_covered_fraction=(
            'covered_residue_fraction',
            'mean'
        ),
    ).reset_index()
)

print(
    'Pocket hypergraph cache: PASS'
)


DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300


,dataset,proteins,mean_pockets,median_pockets,mean_memberships,mean_covered_fraction
0,DevSet1010,1010,24.930693,22.0,240.274257,0.647951
1,TestSet300,300,32.196667,28.5,324.926667,0.686800
2,TestSetNew46,46,19.130435,17.0,180.673913,0.624250


Pocket hypergraph cache: PASS


In [ ]:
# ============================================================
# 15. POCKET-HYPERGRAPH DATASET / MODEL
# ============================================================
class PocketProteinDataset(
    Dataset
):
    def __init__(
        self,
        split,
        df
    ):
        self.split=split

        self.df=df.reset_index(
            drop=True
        )

        self.store=ctx.stores[
            STORE_KEY[split]
        ]

    def __len__(self):
        return len(
            self.df
        )

    def __getitem__(
        self,
        i
    ):
        r=self.df.iloc[
            i
        ]

        pid=str(
            r.prot_id
        )

        mods={
            m:concat_blocks(
                self.store,
                pid,
                blocks
            ).astype(
                np.float32,
                copy=False
            )
            for m,blocks
            in MODALITY_SPEC.items()
        }

        er,ep,pf=POCKET_RAM[
            (
                self.split,
                pid
            )
        ]

        return {
            'pid':
                pid,

            'mods':
                mods,

            'y':
                np.asarray(
                    r.small_binding,
                    dtype=np.float32
                ),

            'edge_res':
                er,

            'edge_pocket':
                ep,

            'pocket_feat':
                pf,
        }

def collate_pocket(
    batch
):
    keys=sorted(
        batch[0]['mods']
    )

    mods={
        k:torch.from_numpy(
            np.concatenate(
                [
                    b['mods'][k]
                    for b in batch
                ],
                axis=0
            )
        ).float()

        for k in keys
    }

    y=torch.from_numpy(
        np.concatenate(
            [
                b['y']
                for b in batch
            ]
        )
    ).float()

    pocket_feat=[]
    edge_res=[]
    edge_pocket=[]

    roff=0
    poff=0

    for b in batch:
        L=len(
            b['y']
        )

        P=len(
            b['pocket_feat']
        )

        if P:
            pocket_feat.append(
                torch.from_numpy(
                    b['pocket_feat']
                ).float()
            )

        if len(
            b['edge_res']
        ):
            edge_res.append(
                torch.from_numpy(
                    b['edge_res']
                ).long()
                +roff
            )

            edge_pocket.append(
                torch.from_numpy(
                    b['edge_pocket']
                ).long()
                +poff
            )

        roff+=L
        poff+=P

    pocket_feat=(
        torch.cat(
            pocket_feat,
            dim=0
        )
        if pocket_feat
        else torch.zeros(
            (0,22),
            dtype=torch.float32
        )
    )

    edge_res=(
        torch.cat(
            edge_res,
            dim=0
        )
        if edge_res
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    edge_pocket=(
        torch.cat(
            edge_pocket,
            dim=0
        )
        if edge_pocket
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    return {
        'mods':
            mods,

        'y':
            y,

        'edge_res':
            edge_res,

        'edge_pocket':
            edge_pocket,

        'pocket_feat':
            pocket_feat,
    }

class PocketResidueEncoder(
    nn.Module
):
    def __init__(
        self,
        dims,
        mode='gated',
        latent=192,
        drop=0.2
    ):
        super().__init__()

        self.keys=sorted(
            dims
        )

        self.mode=mode

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(
                    d,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                )
            )

            for k,d
            in dims.items()
        })

        if mode=='gated':
            self.gate=nn.ModuleDict({
                k:nn.Linear(
                    latent,
                    1
                )
                for k in dims
            })

        elif mode=='attention':
            self.attn=nn.MultiheadAttention(
                latent,
                6,
                dropout=drop,
                batch_first=True
            )

        else:
            raise ValueError(
                mode
            )

    def forward(
        self,
        x
    ):
        Z=torch.stack(
            [
                self.enc[k](
                    x[k]
                )
                for k in self.keys
            ],
            dim=1
        )

        if self.mode=='gated':
            G=torch.cat(
                [
                    self.gate[k](
                        Z[:,j,:]
                    )
                    for j,k
                    in enumerate(
                        self.keys
                    )
                ],
                dim=1
            )

            G=torch.softmax(
                G,
                dim=1
            ).unsqueeze(
                -1
            )

            return (
                Z*G
            ).sum(
                dim=1
            )

        A,_=self.attn(
            Z,
            Z,
            Z,
            need_weights=False
        )

        return (
            Z+A
        ).mean(
            dim=1
        )

class SmallPocketHypergraph(
    nn.Module
):
    def __init__(
        self,
        dims,
        mode='gated',
        latent=192,
        moe=False,
        experts=4,
        drop=0.2
    ):
        super().__init__()

        self.encoder=PocketResidueEncoder(
            dims,
            mode=mode,
            latent=latent,
            drop=drop
        )

        self.pocket_static=nn.Sequential(
            nn.LayerNorm(
                22
            ),
            nn.Linear(
                22,
                64
            ),
            nn.GELU()
        )

        self.pocket_mlp=nn.Sequential(
            nn.Linear(
                latent*2+64,
                latent
            ),
            nn.GELU(),
            nn.Dropout(
                drop
            ),
            nn.Linear(
                latent,
                latent
            ),
            nn.GELU(),
        )

        self.message_mlp=nn.Sequential(
            nn.LayerNorm(
                latent
            ),
            nn.Linear(
                latent,
                latent
            ),
            nn.GELU(),
            nn.Dropout(
                drop
            ),
            nn.Linear(
                latent,
                latent
            )
        )

        self.alpha_logit=nn.Parameter(
            torch.tensor(
                -2.944,
                dtype=torch.float32
            )
        )

        self.moe=moe

        if not moe:
            self.head=nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    128
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    128,
                    1
                )
            )

        else:
            self.experts=nn.ModuleList([
                nn.Sequential(
                    nn.LayerNorm(
                        latent
                    ),
                    nn.Linear(
                        latent,
                        128
                    ),
                    nn.GELU(),
                    nn.Dropout(
                        drop
                    ),
                    nn.Linear(
                        128,
                        1
                    )
                )

                for _ in range(
                    experts
                )
            ])

            self.expert_gate=nn.Linear(
                latent,
                experts
            )

    def index_mean(
        self,
        x,
        index,
        n
    ):
        out=x.new_zeros(
            (
                n,
                x.shape[-1]
            )
        )

        cnt=x.new_zeros(
            (
                n,
                1
            )
        )

        if len(index):
            out.index_add_(
                0,
                index,
                x
            )

            cnt.index_add_(
                0,
                index,
                x.new_ones(
                    (
                        len(index),
                        1
                    )
                )
            )

        return (
            out
            /cnt.clamp_min(
                1.0
            )
        )

    def index_max(
        self,
        x,
        index,
        n
    ):
        if n == 0:
            return x.new_zeros(
                (0, x.shape[-1])
            )

        base = x.new_full(
            (n, x.shape[-1]),
            -1e9
        )

        if len(index):
            idx = index[:, None].expand(
                -1,
                x.shape[-1]
            )

            # IMPORTANT:
            # out-of-place scatter_reduce for safe autograd
            out = base.scatter_reduce(
                0,
                idx,
                x,
                reduce='amax',
                include_self=True
            )
        else:
            out = base

        # IMPORTANT:
        # no inplace masking
        out = torch.where(
            out < -1e8,
            torch.zeros_like(out),
            out
        )

        return out

    def forward(
        self,
        mods,
        edge_res,
        edge_pocket,
        pocket_feat
    ):
        h=self.encoder(
            mods
        )

        nres=h.shape[0]
        npocket=pocket_feat.shape[0]

        alpha=torch.sigmoid(
            self.alpha_logit
        )

        if (
            npocket
            and edge_res.numel()
        ):
            mean=self.index_mean(
                h[edge_res],
                edge_pocket,
                npocket
            )

            mx=self.index_max(
                h[edge_res],
                edge_pocket,
                npocket
            )

            ps=self.pocket_static(
                pocket_feat
            )

            hp=self.pocket_mlp(
                torch.cat(
                    [
                        mean,
                        mx,
                        ps
                    ],
                    dim=-1
                )
            )

            msg=self.index_mean(
                hp[edge_pocket],
                edge_res,
                nres
            )

            incident=h.new_zeros(
                (
                    nres,
                    1
                )
            )

            incident.index_add_(
                0,
                edge_res,
                h.new_ones(
                    (
                        len(edge_res),
                        1
                    )
                )
            )

            delta=self.message_mlp(
                msg
            )

            delta=(
                delta
                *(
                    incident>0
                ).float()
            )

            h=(
                h
                +alpha*delta
            )

        if not self.moe:
            return self.head(
                h
            ).squeeze(
                -1
            )

        logits=torch.cat(
            [
                e(h)
                for e in self.experts
            ],
            dim=1
        )

        w=torch.softmax(
            self.expert_gate(
                h
            ),
            dim=1
        )

        return (
            logits*w
        ).sum(
            dim=1
        )

POCKET_FACTORY = {
    'SM_PocketHG_Gated':
        lambda dims:
            SmallPocketHypergraph(
                dims,
                mode='gated',
                latent=POCKET_LATENT,
                moe=False
            ),

    'SM_PocketHG_Attention':
        lambda dims:
            SmallPocketHypergraph(
                dims,
                mode='attention',
                latent=POCKET_LATENT,
                moe=False
            ),

    'SM_PocketHG_GatedMoE':
        lambda dims:
            SmallPocketHypergraph(
                dims,
                mode='gated',
                latent=POCKET_LATENT,
                moe=True,
                experts=4
            ),
}

print(
    'Pocket-hypergraph architectures:',
    list(
        POCKET_FACTORY
    )
)


Pocket-hypergraph architectures: ['SM_PocketHG_Gated', 'SM_PocketHG_Attention', 'SM_PocketHG_GatedMoE']


In [ ]:
# ============================================================
# 16. POCKET-HYPERGRAPH CV TRAINER — RESUME SAFE
# ============================================================
POCKET_STATE_ROOT=(
    RUN_ROOT/
    'pocket_states'
)

POCKET_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

def move_pocket_batch(
    b
):
    return {
        'mods':{
            k:v.to(
                DEVICE,
                non_blocking=True
            )
            for k,v
            in b['mods'].items()
        },

        'y':
            b['y'].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_res':
            b['edge_res'].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_pocket':
            b['edge_pocket'].to(
                DEVICE,
                non_blocking=True
            ),

        'pocket_feat':
            b['pocket_feat'].to(
                DEVICE,
                non_blocking=True
            ),
    }

def predict_pocket(
    model,
    dataset
):
    loader=DataLoader(
        dataset,
        batch_size=POCKET_PROTEIN_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_pocket
    )

    pp=[]
    yy=[]

    model.eval()

    with torch.no_grad():
        for b in loader:
            b=move_pocket_batch(
                b
            )

            p=torch.sigmoid(
                model(
                    b['mods'],
                    b['edge_res'],
                    b['edge_pocket'],
                    b['pocket_feat']
                )
            )

            pp.append(
                p.detach()
                .cpu()
                .numpy()
            )

            yy.append(
                b['y']
                .detach()
                .cpu()
                .numpy()
            )

    return (
        np.concatenate(
            pp
        ),
        np.concatenate(
            yy
        )
    )

def train_pocket_fold(
    arch,
    tr_ds,
    va_ds,
    fold
):
    torch.manual_seed(
        SEED+100+fold
    )

    np.random.seed(
        SEED+100+fold
    )

    model=POCKET_FACTORY[
        arch
    ](DIMS).to(
        DEVICE
    )

    ytr=np.concatenate(
        tr_ds.df.small_binding.apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        ).tolist()
    )

    npos=max(
        float(
            ytr.sum()
        ),
        1.0
    )

    nneg=max(
        float(
            len(ytr)-ytr.sum()
        ),
        1.0
    )

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(
                nneg/npos
            ),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=POCKET_LR,
        weight_decay=POCKET_WEIGHT_DECAY
    )

    loader=DataLoader(
        tr_ds,
        batch_size=POCKET_PROTEIN_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_pocket
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(
        1,
        POCKET_EPOCHS+1
    ):
        model.train()
        losses=[]

        for b in loader:
            b=move_pocket_batch(
                b
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits=model(
                b['mods'],
                b['edge_res'],
                b['edge_pocket'],
                b['pocket_feat']
            )

            loss=criterion(
                logits,
                b['y']
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            opt.step()

            losses.append(
                float(
                    loss.detach()
                    .cpu()
                )
            )

        vp,vy=predict_pocket(
            model,
            va_ds
        )

        val_ap=ap(
            vy,
            vp
        )

        if (
            val_ap
            >best_ap+1e-5
        ):
            best_ap=val_ap

            best_state={
                k:v.detach()
                .cpu()
                .clone()
                for k,v
                in model.state_dict().items()
            }

            bad=0

        else:
            bad+=1

        if (
            epoch==1
            or epoch%5==0
        ):
            print(
                arch,
                'fold',
                fold,
                'epoch',
                epoch,
                'loss',
                round(
                    float(
                        np.mean(
                            losses
                        )
                    ),
                    4
                ),
                'val_AP',
                round(
                    val_ap,
                    5
                ),
                'best',
                round(
                    best_ap,
                    5
                ),
                'alpha',
                round(
                    float(
                        torch.sigmoid(
                            model.alpha_logit
                        )
                        .detach()
                        .cpu()
                    ),
                    4
                )
            )

        if bad>=POCKET_PATIENCE:
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        best_state
    )

def run_pocket_cv(
    arch
):
    offsets={}
    cursor=0

    for r in train_df.itertuples():
        L=len(
            r.sequence
        )

        offsets[
            str(
                r.prot_id
            )
        ]=(
            cursor,
            cursor+L
        )

        cursor+=L

    oof=np.full(
        cursor,
        np.nan,
        dtype=np.float32
    )

    rows=[]
    states={}

    adir=(
        POCKET_STATE_ROOT/
        arch
    )

    adir.mkdir(
        parents=True,
        exist_ok=True
    )

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(
            val_ids
        )

        tr_df=train_df[
            ~train_df.prot_id.isin(
                val_set
            )
        ].copy()

        va_df=train_df[
            train_df.prot_id.isin(
                val_set
            )
        ].copy()

        tr_ds=PocketProteinDataset(
            'DevSet1010',
            tr_df
        )

        va_ds=PocketProteinDataset(
            'DevSet1010',
            va_df
        )

        state_fp=(
            adir/
            f'fold{fold}.pt'
        )

        pred_fp=(
            adir/
            f'fold{fold}_val.npy'
        )

        meta_fp=(
            adir/
            f'fold{fold}.json'
        )

        expected=int(
            va_df.sequence
            .str.len()
            .sum()
        )

        reuse=False

        if (
            state_fp.exists()
            and pred_fp.exists()
            and meta_fp.exists()
        ):
            try:
                meta=json.loads(
                    meta_fp.read_text()
                )

                pp=np.load(
                    pred_fp
                )

                reuse=(
                    meta.get(
                        'representation'
                    )
                    ==BEST_REP_NAME
                    and meta.get(
                        'n_val'
                    )
                    ==expected
                    and pp.shape
                    ==(expected,)
                    and np.all(
                        np.isfinite(
                            pp
                        )
                    )
                )

            except Exception:
                reuse=False

        if reuse:
            print(
                arch,
                'fold',
                fold,
                'REUSE'
            )

            state=torch.load(
                state_fp,
                map_location='cpu'
            )

            pred=np.load(
                pred_fp
            ).astype(
                np.float32
            )

        else:
            print(
                arch,
                'fold',
                fold,
                'TRAIN'
            )

            model,state=(
                train_pocket_fold(
                    arch,
                    tr_ds,
                    va_ds,
                    fold
                )
            )

            pred,_=predict_pocket(
                model,
                va_ds
            )

            pred=pred.astype(
                np.float32
            )

            torch.save(
                state,
                state_fp
            )

            np.save(
                pred_fp,
                pred
            )

            meta_fp.write_text(
                json.dumps({
                    'representation':
                        BEST_REP_NAME,

                    'blocks':
                        BEST_REP_BLOCKS,

                    'n_val':
                        expected,
                },indent=2)
            )

            del model

            if DEVICE.type=='cuda':
                torch.cuda.empty_cache()

        states[
            fold
        ]=state

        pos=0

        for r in va_df.itertuples():
            L=len(
                r.sequence
            )

            a,b=offsets[
                str(
                    r.prot_id
                )
            ]

            oof[
                a:b
            ]=pred[
                pos:
                pos+L
            ]

            pos+=L

        yv=np.concatenate(
            va_df.small_binding.apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.float32
                    )
            ).tolist()
        )

        rows.append({
            'architecture':
                arch,

            'fold':
                fold,

            'AUPRC':
                ap(
                    yv,
                    pred
                ),

            'n_val':
                len(
                    pred
                ),

            'n_positive':
                int(
                    yv.sum()
                ),
        })

    if not np.all(
        np.isfinite(
            oof
        )
    ):
        raise RuntimeError(
            f'{arch}: incomplete pocket OOF'
        )

    f=pd.DataFrame(
        rows
    )

    yall=np.concatenate(
        train_df.small_binding.apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        ).tolist()
    )

    return (
        {
            'architecture':
                arch,

            'CV_AUPRC_mean':
                float(
                    f.AUPRC.mean()
                ),

            'CV_AUPRC_std':
                float(
                    f.AUPRC.std(
                        ddof=1
                    )
                ),

            'OOF_AUPRC':
                ap(
                    yall,
                    oof
                ),
        },
        f,
        oof,
        states
    )


In [ ]:
# ============================================================
# 17. RUN POCKET-HYPERGRAPH SCREEN — DEV ONLY
# ============================================================
POCKET_RESULTS={}
POCKET_FOLDS=[]
POCKET_OOF={}
POCKET_STATES={}

if RUN_POCKET_HYPERGRAPH:
    for arch in POCKET_FACTORY:
        print(
            '\n'
            +'='*80
        )

        print(
            'POCKET HYPERGRAPH:',
            arch
        )

        print(
            '='*80
        )

        s,f,o,st=run_pocket_cv(
            arch
        )

        POCKET_RESULTS[
            arch
        ]=s

        POCKET_FOLDS.append(
            f
        )

        POCKET_OOF[
            arch
        ]=o

        POCKET_STATES[
            arch
        ]=st

pocket_summary=pd.DataFrame(
    POCKET_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(
    drop=True
)

pocket_folds=pd.concat(
    POCKET_FOLDS,
    ignore_index=True
)

pocket_summary.to_csv(
    RUN_ROOT/
    'pocket_hypergraph_cv.csv',
    index=False
)

pocket_folds.to_csv(
    RUN_ROOT/
    'pocket_hypergraph_folds.csv',
    index=False
)

display(
    pocket_summary
)

BEST_POCKET=str(
    pocket_summary.iloc[0]
    .architecture
)

BEST_POCKET_SCORE=float(
    pocket_summary.iloc[0]
    .CV_AUPRC_mean
)

alphas=[]

for fold,state in POCKET_STATES[
    BEST_POCKET
].items():
    if (
        'alpha_logit'
        in state
    ):
        alphas.append(
            float(
                torch.sigmoid(
                    state[
                        'alpha_logit'
                    ]
                ).item()
            )
        )

print(
    'BEST POCKET MODEL:',
    BEST_POCKET
)

print(
    'Dev CV AUPRC:',
    BEST_POCKET_SCORE
)

print(
    'Residual alpha:',
    alphas
)

print(
    'Mean alpha:',
    float(
        np.mean(
            alphas
        )
    )
    if alphas
    else None
)

(RUN_ROOT/'locked_pocket_model.json').write_text(
    json.dumps({
        'architecture':
            BEST_POCKET,

        'dev_cv_auprc':
            BEST_POCKET_SCORE,

        'alpha_by_fold':
            alphas,

        'alpha_mean':
            float(
                np.mean(
                    alphas
                )
            )
            if alphas
            else None,
    },indent=2)
)



POCKET HYPERGRAPH: SM_PocketHG_Gated
SM_PocketHG_Gated fold 1 REUSE
SM_PocketHG_Gated fold 2 REUSE
SM_PocketHG_Gated fold 3 TRAIN
SM_PocketHG_Gated fold 3 epoch 1 loss 0.4727 val_AP 0.29881 best 0.29881 alpha 0.05
SM_PocketHG_Gated fold 3 epoch 5 loss 0.2459 val_AP 0.35825 best 0.35978 alpha 0.0494
SM_PocketHG_Gated fold 3 epoch 10 loss 0.0673 val_AP 0.31201 best 0.35978 alpha 0.0488
SM_PocketHG_Gated fold 4 TRAIN
SM_PocketHG_Gated fold 4 epoch 1 loss 0.4677 val_AP 0.29657 best 0.29657 alpha 0.0499
SM_PocketHG_Gated fold 4 epoch 5 loss 0.25 val_AP 0.35431 best 0.36063 alpha 0.0493
SM_PocketHG_Gated fold 4 epoch 10 loss 0.0664 val_AP 0.32036 best 0.36063 alpha 0.0489
SM_PocketHG_Gated fold 5 TRAIN
SM_PocketHG_Gated fold 5 epoch 1 loss 0.4734 val_AP 0.32893 best 0.32893 alpha 0.0499
SM_PocketHG_Gated fold 5 epoch 5 loss 0.2349 val_AP 0.41406 best 0.41458 alpha 0.0497
SM_PocketHG_Gated fold 5 epoch 10 loss 0.065 val_AP 0.38727 best 0.41458 alpha 0.049

POCKET HYPERGRAPH: SM_PocketHG_Atte

,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,SM_PocketHG_Attention,0.397647,0.040608,0.376346
1,SM_PocketHG_Gated,0.394450,0.038383,0.385554
2,SM_PocketHG_GatedMoE,0.391164,0.028219,0.372161


BEST POCKET MODEL: SM_PocketHG_Attention
Dev CV AUPRC: 0.39764655317751607
Residual alpha: [0.04977594316005707, 0.04997347295284271, 0.04953764006495476, 0.05000319331884384, 0.04984140023589134]
Mean alpha: 0.04982632994651794


271

# Part E — Global Dev-only lock

The final headline family is selected before external evaluation.

The more complex pocket-hypergraph replaces the best local multimodal model only if it improves mean Dev CV AUPRC by at least `GLOBAL_COMPLEXITY_MARGIN`.


In [ ]:
# ============================================================
# 18. GLOBAL DEV-ONLY LOCK
# ============================================================
candidates=pd.DataFrame([
    {
        'family':
            'LOCAL_MULTIMODAL',

        'model':
            LOCKED_ARCH,

        'Dev_CV_AUPRC':
            LOCKED_ARCH_SCORE,

        'complexity_rank':
            1,
    },

    {
        'family':
            'POCKET_HYPERGRAPH',

        'model':
            BEST_POCKET,

        'Dev_CV_AUPRC':
            BEST_POCKET_SCORE,

        'complexity_rank':
            2,
    },
])

winner=(
    candidates
    .sort_values(
        'complexity_rank'
    )
    .iloc[0]
    .to_dict()
)

for _,r in (
    candidates
    .sort_values(
        'complexity_rank'
    )
    .iloc[1:]
    .iterrows()
):
    if (
        float(
            r.Dev_CV_AUPRC
        )
        -float(
            winner[
                'Dev_CV_AUPRC'
            ]
        )
        >=GLOBAL_COMPLEXITY_MARGIN
    ):
        winner=r.to_dict()

GLOBAL_FAMILY=str(
    winner[
        'family'
    ]
)

GLOBAL_MODEL=str(
    winner[
        'model'
    ]
)

GLOBAL_SCORE=float(
    winner[
        'Dev_CV_AUPRC'
    ]
)

display(
    candidates
)

print(
    'GLOBAL LOCK:',
    GLOBAL_FAMILY,
    GLOBAL_MODEL,
    GLOBAL_SCORE
)

(RUN_ROOT/'GLOBAL_LOCK.json').write_text(
    json.dumps({
        'family':
            GLOBAL_FAMILY,

        'model':
            GLOBAL_MODEL,

        'dev_cv_auprc':
            GLOBAL_SCORE,

        'complexity_margin':
            GLOBAL_COMPLEXITY_MARGIN,
    },indent=2)
)


,family,model,Dev_CV_AUPRC,complexity_rank
0,LOCAL_MULTIMODAL,ConcatMLP,0.377218,1
1,POCKET_HYPERGRAPH,SM_PocketHG_Attention,0.397647,2


GLOBAL LOCK: POCKET_HYPERGRAPH SM_PocketHG_Attention 0.39764655317751607


140

In [ ]:
# ============================================================
# 19. METRIC HELPERS + DEV OOF THRESHOLDS
# ============================================================
def binary_metrics(
    y,
    p,
    threshold
):
    y=np.asarray(
        y,
        dtype=int
    )

    p=np.asarray(
        p,
        dtype=float
    )

    pred=(
        p>=threshold
    ).astype(
        int
    )

    return {
        'AUPRC':
            float(
                average_precision_score(
                    y,p
                )
            ),

        'AUROC':
            float(
                roc_auc_score(
                    y,p
                )
            ),

        'precision':
            float(
                precision_score(
                    y,pred,
                    zero_division=0
                )
            ),

        'recall':
            float(
                recall_score(
                    y,pred,
                    zero_division=0
                )
            ),

        'F1':
            float(
                f1_score(
                    y,pred,
                    zero_division=0
                )
            ),

        'MCC':
            float(
                matthews_corrcoef(
                    y,pred
                )
            ),
    }

def best_mcc_threshold(
    y,
    p
):
    best=(-999,None)

    for t in np.linspace(
        0.01,
        0.99,
        199
    ):
        m=matthews_corrcoef(
            y,
            (
                p>=t
            ).astype(
                int
            )
        )

        if m>best[0]:
            best=(
                m,
                float(t)
            )

    return (
        best[1],
        best[0]
    )

dev_y=PACKS[
    'DevSet1010'
]['y'].astype(
    int
)

LOCAL_OOF=ARCH_OOF[
    LOCKED_ARCH
]

LOCAL_THR,LOCAL_OOF_MCC=(
    best_mcc_threshold(
        dev_y,
        LOCAL_OOF
    )
)

POCKET_OOF_LOCK=POCKET_OOF[
    BEST_POCKET
]

POCKET_THR,POCKET_OOF_MCC=(
    best_mcc_threshold(
        dev_y,
        POCKET_OOF_LOCK
    )
)

print(
    'Local threshold:',
    LOCAL_THR,
    'OOF MCC:',
    LOCAL_OOF_MCC
)

print(
    'Pocket threshold:',
    POCKET_THR,
    'OOF MCC:',
    POCKET_OOF_MCC
)


Local threshold: 0.5593939393939393 OOF MCC: 0.3750286199352274
Pocket threshold: 0.6088888888888889 OOF MCC: 0.3799406947088853


In [ ]:
# ============================================================
# 20. FINAL EXTERNAL EVALUATION — LOCKED FAMILY WINNERS
# ============================================================
def predict_local_external(
    split
):
    pack=PACKS[
        split
    ]

    ds=MultiArrayDataset(
        pack['mods'],
        pack['y']
    )

    preds=[]

    for fold,state in ARCH_STATES[
        LOCKED_ARCH
    ].items():
        model=ARCH_FACTORY[
            LOCKED_ARCH
        ](DIMS).to(
            DEVICE
        )

        model.load_state_dict(
            state
        )

        preds.append(
            predict_flat(
                model,
                ds
            )
        )

        del model

        if DEVICE.type=='cuda':
            torch.cuda.empty_cache()

    return np.mean(
        np.stack(
            preds
        ),
        axis=0
    )

def predict_pocket_external(
    split
):
    ds=PocketProteinDataset(
        split,
        SPLIT_DF[
            split
        ]
    )

    preds=[]

    for fold,state in POCKET_STATES[
        BEST_POCKET
    ].items():
        model=POCKET_FACTORY[
            BEST_POCKET
        ](DIMS).to(
            DEVICE
        )

        model.load_state_dict(
            state
        )

        p,_=predict_pocket(
            model,
            ds
        )

        preds.append(
            p
        )

        del model

        if DEVICE.type=='cuda':
            torch.cuda.empty_cache()

    return np.mean(
        np.stack(
            preds
        ),
        axis=0
    )

rows=[]

if RUN_FINAL_EXTERNAL:
    for split in [
        'TestSet300',
        'TestSetNew46'
    ]:
        y=PACKS[
            split
        ]['y'].astype(
            int
        )

        lp=predict_local_external(
            split
        )

        pp=predict_pocket_external(
            split
        )

        for (
            family,
            model,
            p,
            thr,
            devscore
        ) in [
            (
                'LOCAL_MULTIMODAL',
                LOCKED_ARCH,
                lp,
                LOCAL_THR,
                LOCKED_ARCH_SCORE
            ),
            (
                'POCKET_HYPERGRAPH',
                BEST_POCKET,
                pp,
                POCKET_THR,
                BEST_POCKET_SCORE
            ),
        ]:
            m05=binary_metrics(
                y,
                p,
                0.5
            )

            moo=binary_metrics(
                y,
                p,
                thr
            )

            row={
                'family':
                    family,

                'model':
                    model,

                'dataset':
                    split,

                'Dev_CV_AUPRC':
                    devscore,

                'Dev_OOF_MCC_threshold':
                    thr,
            }

            for k,v in m05.items():
                row[
                    f'{k}_fixed0.5'
                ]=v

            for k,v in moo.items():
                row[
                    f'{k}_OOFthr'
                ]=v

            rows.append(
                row
            )

        pd.DataFrame({
            'prot_id':
                PACKS[split]['pids'],

            'residue_index':
                PACKS[split]['ridx'],

            'y_true':
                y,

            'local_probability':
                lp,

            'pocket_probability':
                pp,
        }).to_csv(
            RUN_ROOT/
            f'predictions_{split}.csv.gz',
            index=False
        )

external_df=pd.DataFrame(
    rows
)

external_df.to_csv(
    RUN_ROOT/
    'external_family_winners.csv',
    index=False
)

display(
    external_df
)

headline=external_df[
    external_df.family.eq(
        GLOBAL_FAMILY
    )
    &
    external_df.model.eq(
        GLOBAL_MODEL
    )
].copy()

headline.to_csv(
    RUN_ROOT/
    'FINAL_HEADLINE_EXTERNAL.csv',
    index=False
)

print(
    'HEADLINE:'
)

display(
    headline
)


,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
0,LOCAL_MULTIMODAL,ConcatMLP,TestSet300,0.377218,0.559394,0.432469,0.862458,0.514911,0.397849,0.448873,0.421109,0.432469,0.862458,0.546205,0.355607,0.430764,0.411645
1,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSet300,0.397647,0.608889,0.440255,0.867647,0.454417,0.495136,0.473903,0.437803,0.440255,0.867647,0.509835,0.418075,0.459418,0.429625
2,LOCAL_MULTIMODAL,ConcatMLP,TestSetNew46,0.377218,0.559394,0.394729,0.857986,0.455108,0.345882,0.393048,0.358818,0.394729,0.857986,0.473118,0.310588,0.375000,0.347716
3,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSetNew46,0.397647,0.608889,0.424158,0.865581,0.451852,0.430588,0.440964,0.401205,0.424158,0.865581,0.487500,0.367059,0.418792,0.386954


HEADLINE:


,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
1,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSet300,0.397647,0.608889,0.440255,0.867647,0.454417,0.495136,0.473903,0.437803,0.440255,0.867647,0.509835,0.418075,0.459418,0.429625
3,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSetNew46,0.397647,0.608889,0.424158,0.865581,0.451852,0.430588,0.440964,0.401205,0.424158,0.865581,0.487500,0.367059,0.418792,0.386954


# Part F — Read-only baselines and historical controls

No baseline inference is rerun.

The notebook reads:
- reproduced bindEmbed21DL / bindNode24 small-binding rows;
- prior GEAR small-binding controls when available.


In [ ]:
# ============================================================
# 21. BASELINE / HISTORICAL CONTROL TABLES
# ============================================================
baseline_frames=[]

fp=(
    BASELINE_WORK/
    'results'/
    'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
)

if fp.exists():
    z=pd.read_csv(
        fp
    )

    if 'label' in z.columns:
        z=z[
            z.label.eq(
                'small_binding'
            )
        ].copy()

    if 'operating_point' in z.columns:
        z=z[
            z.operating_point.eq(
                'fixed_0.5'
            )
        ].copy()

    baseline_frames.append(
        z
    )

baselines=(
    pd.concat(
        baseline_frames,
        ignore_index=True,
        sort=False
    )
    if baseline_frames
    else pd.DataFrame()
)

if len(
    baselines
):
    cols=[
        c
        for c in [
            'method',
            'dataset',
            'label',
            'AP_AUPRC',
            'AUPRC',
            'AUROC',
            'global_positive_F1',
            'global_MCC',
            'protein_mean_F1',
            'protein_mean_MCC',
        ]
        if c in baselines.columns
    ]

    print(
        'LOCKED REPRODUCTION BASELINES'
    )

    display(
        baselines[
            cols
        ]
    )

    baselines.to_csv(
        RUN_ROOT/
        'locked_small_baselines.csv',
        index=False
    )

historic_runs=[
    'ultimate_other_generic_core_v1',
    'ultimate_other_standardized13_transfer_v1',
    'ultimate_other_metal_transfer_v1',
    'ultimate_other_metal_mechanistic_transfer_v1',
    'ultimate_other_evo_full_v1',
    'ultimate_other_generic_graph_v1',
    'ultimate_other_evo_graph_v1',
]

historic=[]

for run in historic_runs:
    root=(
        BASE/
        'results'/
        run/
        'small_binding'
    )

    fp=(
        root/
        'cv_summary.csv'
    )

    if fp.exists():
        z=pd.read_csv(
            fp
        )

        z.insert(
            0,
            'run_version',
            run
        )

        historic.append(
            z
        )

historic_cv=(
    pd.concat(
        historic,
        ignore_index=True,
        sort=False
    )
    if historic
    else pd.DataFrame()
)

if len(
    historic_cv
):
    print(
        'PRIOR SMALL DEV CONTROLS'
    )

    display(
        historic_cv.sort_values(
            'CV_AUPRC',
            ascending=False
        ).head(
            40
        )
    )

    historic_cv.to_csv(
        RUN_ROOT/
        'prior_small_cv_controls.csv',
        index=False
    )


LOCKED REPRODUCTION BASELINES


,method,dataset,label,AP_AUPRC,AUROC,global_positive_F1,global_MCC,protein_mean_F1,protein_mean_MCC
0,bindEmbed21DL,TestSet300,small_binding,0.394041,0.864302,0.435777,0.396702,0.331199,0.307604
1,bindEmbed21DL,TestSetNew46,small_binding,0.315583,0.857311,0.374857,0.327813,0.285816,0.253609
2,bindNode24,TestSet300,small_binding,0.431709,0.869662,0.468245,0.432216,0.353203,0.337026
3,bindNode24,TestSetNew46,small_binding,0.380681,0.872435,0.407911,0.367430,0.300463,0.266746


PRIOR SMALL DEV CONTROLS


,run_version,model,representation,folds,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,CV_F1_at_0_5,CV_MCC_at_0_5,mean_best_epoch,n_params,mean_seconds,fold_AUPRC,OOF_threshold,mean_alpha
24,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+ALL9,5.0,0.371896,0.031957,0.855650,0.368474,0.351015,9.4,378497.0,35.031736,NaN,NaN,NaN
17,ultimate_other_metal_transfer_v1,MLP,ESM+Atomic+Chem+MULTI2,5.0,0.371304,0.029691,0.854266,0.360273,0.345432,9.0,369793.0,33.015180,NaN,NaN,NaN
25,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+MULTI2,5.0,0.371304,0.029691,0.854266,0.360273,0.345432,9.0,369793.0,33.142819,NaN,NaN,NaN
26,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+MECH1,5.0,0.371042,0.028609,0.852730,0.368351,0.348973,10.0,363649.0,34.838930,NaN,NaN,NaN
27,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Topology,5.0,0.369239,0.032497,0.850695,0.361020,0.344716,9.6,358017.0,33.481025,NaN,NaN,NaN
28,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Access,5.0,0.368116,0.025843,0.853586,0.360011,0.343871,8.8,357505.0,31.871501,NaN,NaN,NaN
4,ultimate_other_standardized13_transfer_v1,MLP,ESM+Structure+Atomic+Chem,5.0,0.367558,0.029767,0.853827,0.359113,0.346258,8.0,385665.0,31.767710,NaN,NaN,NaN
29,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Shell2,5.0,0.365533,0.029936,0.845022,0.370564,0.348568,11.4,358017.0,36.997176,NaN,NaN,NaN
30,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Conf,5.0,0.364430,0.032103,0.845702,0.378972,0.352813,11.8,355969.0,37.379578,NaN,NaN,NaN
31,ultimate_other_metal_mechanistic_transfer_v1,MLP,BASE+Elec3D,5.0,0.364409,0.027228,0.849843,0.355596,0.339932,9.2,356993.0,32.479378,NaN,NaN,NaN


# Part G — Automatic insight report

The final section decomposes gain from:

- accessibility;
- pocket geometry/topology;
- aromatic/hydrophobic chemistry;
- ligand-contact chemistry / solvation;
- multimodal fusion;
- latent ligand experts;
- full pocket-hypergraph reasoning.

It explicitly reports when extra complexity is not justified.


In [ ]:
# ============================================================
# 22. GAIN DECOMPOSITION / HYPOTHESIS VERDICTS
# ============================================================
def score_rep(
    df,
    name
):
    x=df.loc[
        df.representation.eq(
            name
        ),
        'CV_AUPRC'
    ]

    return (
        float(
            x.iloc[0]
        )
        if len(x)
        else np.nan
    )

mechanism_rows=[]

for name in MECHANISM_GRID:
    s=score_rep(
        combo_cv,
        name
    )

    mechanism_rows.append({
        'representation':
            name,

        'Dev_CV_AUPRC':
            s,

        'delta_vs_ESM':
            s-base_score,
    })

mechanism_gain=pd.DataFrame(
    mechanism_rows
).sort_values(
    'Dev_CV_AUPRC',
    ascending=False
)

display(
    mechanism_gain
)

mechanism_gain.to_csv(
    RUN_ROOT/
    'INSIGHT_mechanism_gain.csv',
    index=False
)

arch_gain=(
    LOCKED_ARCH_SCORE
    -concat_score
)

pocket_gain=(
    BEST_POCKET_SCORE
    -LOCKED_ARCH_SCORE
)

def verdict(
    gain
):
    if gain>=0.01:
        return 'STRONG'

    if gain>=0.005:
        return 'CLEAR'

    if gain>=MIN_MEAN_AUPRC_GAIN:
        return 'MODEST_BUT_REAL'

    if gain>0:
        return 'WEAK'

    return 'NO_GAIN'

summary={
    'ESM_baseline':
        base_score,

    'best_representation':
        BEST_REP_NAME,

    'best_representation_dev_auprc':
        BEST_REP_SCORE,

    'best_representation_gain_vs_ESM':
        BEST_REP_SCORE-base_score,

    'locked_multimodal_architecture':
        LOCKED_ARCH,

    'locked_multimodal_dev_auprc':
        LOCKED_ARCH_SCORE,

    'architecture_gain_vs_concat':
        arch_gain,

    'architecture_verdict':
        verdict(
            arch_gain
        ),

    'best_pocket_hypergraph':
        BEST_POCKET,

    'best_pocket_dev_auprc':
        BEST_POCKET_SCORE,

    'pocket_gain_vs_locked_local':
        pocket_gain,

    'pocket_verdict':
        verdict(
            pocket_gain
        ),

    'pocket_alpha_mean':
        float(
            np.mean(
                alphas
            )
        )
        if alphas
        else None,

    'global_family':
        GLOBAL_FAMILY,

    'global_model':
        GLOBAL_MODEL,

    'global_dev_auprc':
        GLOBAL_SCORE,

    'EVO_available':
        EVO_AVAILABLE,

    'ESMIF_available':
        ESMIF_AVAILABLE,
}

(RUN_ROOT/'INSIGHT_summary.json').write_text(
    json.dumps(
        summary,
        indent=2
    )
)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print(
    '\n'
    +'='*96
)

print(
    'SMALL ULTIMATE V1 — INTERPRETATION'
)

print(
    '='*96
)

for key in [
    'ESM+ACCESS+TOPO',
    'ESM+POCKET_GEOMETRY',
    'ESM+AROM_HYDRO',
    'ESM+LIGAND_CHEMISTRY',
    'ESM+POCKET_CHEM',
    'ESM+POCKET_CHEM+SOLV',
    'ESM+POCKET_CHEM+ATOM_CHEM_CONTROL',
    'ESM+POCKET_CHEM+EVO_CONTROL',
]:
    if key in set(
        combo_cv.representation
    ):
        s=score_rep(
            combo_cv,
            key
        )

        print(
            f'{key:44s} '
            f'{s:.5f} '
            f'({s-base_score:+.5f} vs ESM)'
        )

print(
    '\nArchitecture:'
)

print(
    LOCKED_ARCH,
    f'{LOCKED_ARCH_SCORE:.5f}',
    'gain vs concat =',
    f'{arch_gain:+.5f}',
    verdict(
        arch_gain
    )
)

print(
    '\nPocket hypergraph:'
)

print(
    BEST_POCKET,
    f'{BEST_POCKET_SCORE:.5f}',
    'gain vs local =',
    f'{pocket_gain:+.5f}',
    verdict(
        pocket_gain
    )
)

if alphas:
    print(
        'Learned pocket residual alpha:',
        [
            round(
                x,
                4
            )
            for x in alphas
        ],
        'mean=',
        round(
            float(
                np.mean(
                    alphas
                )
            ),
            4
        )
    )

print(
    '\nGLOBAL DEV-LOCKED WINNER:'
)

print(
    GLOBAL_FAMILY,
    '/',
    GLOBAL_MODEL,
    '/',
    f'{GLOBAL_SCORE:.5f}'
)

print(
    '\nExternal headline:'
)

display(
    headline
)

print(
    '\nPocket coverage:'
)

display(
    pocket_audit.groupby(
        'dataset'
    ).agg(
        mean_candidate_pockets=(
            'n_candidate_pockets',
            'mean'
        ),
        median_candidate_pockets=(
            'n_candidate_pockets',
            'median'
        ),
        mean_covered_fraction=(
            'covered_residue_fraction',
            'mean'
        ),
    ).reset_index()
)

print(
    '\nArtifacts:'
)

print(
    RUN_ROOT
)

print(
    '='*96
)


,representation,Dev_CV_AUPRC,delta_vs_ESM
2,ESM+POCKET_GEOMETRY,0.377708,0.025534
8,ESM+POCKET_CHEM+EVO_CONTROL,0.374193,0.022019
6,ESM+POCKET_CHEM+SOLV,0.374023,0.021849
7,ESM+POCKET_CHEM+ATOM_CHEM_CONTROL,0.372913,0.020739
1,ESM+ACCESS+TOPO,0.372839,0.020665
5,ESM+POCKET_CHEM,0.371453,0.019279
4,ESM+LIGAND_CHEMISTRY,0.362327,0.010154
3,ESM+AROM_HYDRO,0.357675,0.005501
0,ESM,0.352174,0.000000


{
  "ESM_baseline": 0.35217369108912633,
  "best_representation": "ESM+POCKET_GEOMETRY",
  "best_representation_dev_auprc": 0.3777080103628627,
  "best_representation_gain_vs_ESM": 0.025534319273736394,
  "locked_multimodal_architecture": "ConcatMLP",
  "locked_multimodal_dev_auprc": 0.37721812239263486,
  "architecture_gain_vs_concat": 0.0,
  "architecture_verdict": "NO_GAIN",
  "best_pocket_hypergraph": "SM_PocketHG_Attention",
  "best_pocket_dev_auprc": 0.39764655317751607,
  "pocket_gain_vs_locked_local": 0.020428430784881213,
  "pocket_verdict": "STRONG",
  "pocket_alpha_mean": 0.04982632994651794,
  "global_family": "POCKET_HYPERGRAPH",
  "global_model": "SM_PocketHG_Attention",
  "global_dev_auprc": 0.39764655317751607,
  "EVO_available": true,
  "ESMIF_available": false
}

SMALL ULTIMATE V1 — INTERPRETATION
ESM+ACCESS+TOPO                              0.37284 (+0.02066 vs ESM)
ESM+POCKET_GEOMETRY                          0.37771 (+0.02553 vs ESM)
ESM+AROM_HYDRO                 

,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
1,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSet300,0.397647,0.608889,0.440255,0.867647,0.454417,0.495136,0.473903,0.437803,0.440255,0.867647,0.509835,0.418075,0.459418,0.429625
3,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSetNew46,0.397647,0.608889,0.424158,0.865581,0.451852,0.430588,0.440964,0.401205,0.424158,0.865581,0.487500,0.367059,0.418792,0.386954



Pocket coverage:


,dataset,mean_candidate_pockets,median_candidate_pockets,mean_covered_fraction
0,DevSet1010,24.930693,22.0,0.647951
1,TestSet300,32.196667,28.5,0.686800
2,TestSetNew46,19.130435,17.0,0.624250



Artifacts:
/content/drive/MyDrive/Protein-binding-Phi/results/SMALL_ULTIMATE_V1_DATA_DRIVEN_POCKET


In [ ]:
# ============================================================
# 23. FINAL MANIFEST
# ============================================================
manifest={
    'gear_version':
        GEAR_VERSION,

    'python':
        platform.python_version(),

    'torch':
        torch.__version__,

    'device':
        str(DEVICE),

    'task':
        'small_binding',

    'development_set':
        'DevSet1010 official bindNode24 5-fold split',

    'external_sets':[
        'TestSet300',
        'TestSetNew46',
    ],

    'selection_metric':
        'residue-level Dev CV AUPRC',

    'audit_driven_design':{
        'core_existing':[
            'ACCESS',
            'TOPO',
            'CHEM',
            'SOLV',
        ],

        'new_blocks':
            NEW_BLOCKS,

        'core_mechanism':
            'pocket accessibility + geometry/topology + aromatic/hydrophobic/sulfur chemistry',

        'electrostatics_role':
            'secondary/contextual, not core',

        'EVO_role':
            'optional control because prior small-binding EVO was not consistently beneficial',

        'pocket_reasoning':
            'soft residual pocket hypergraph',
    },

    'best_representation':
        BEST_REP_NAME,

    'best_representation_blocks':
        BEST_REP_BLOCKS,

    'locked_architecture':
        LOCKED_ARCH,

    'best_pocket_model':
        BEST_POCKET,

    'global_family':
        GLOBAL_FAMILY,

    'global_model':
        GLOBAL_MODEL,

    'global_dev_auprc':
        GLOBAL_SCORE,

    'evo_available':
        EVO_AVAILABLE,

    'esmif_available':
        ESMIF_AVAILABLE,

    'summary':
        summary,
}

(RUN_ROOT/'FINAL_MANIFEST.json').write_text(
    json.dumps(
        manifest,
        indent=2
    )
)

print(
    json.dumps(
        manifest,
        indent=2
    )
)

print(
    '\n'
    +'='*96
)

print(
    'SMALL ULTIMATE V1 COMPLETE'
)

print(
    '='*96
)

print(
    'Headline:',
    GLOBAL_FAMILY,
    '/',
    GLOBAL_MODEL
)

print(
    'Dev CV AUPRC:',
    GLOBAL_SCORE
)

print(
    'Saved to:',
    RUN_ROOT
)

print(
    '='*96
)


{
  "gear_version": "2026.08-other-bindings-v1",
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "device": "cuda",
  "task": "small_binding",
  "development_set": "DevSet1010 official bindNode24 5-fold split",
  "external_sets": [
    "TestSet300",
    "TestSetNew46"
  ],
  "selection_metric": "residue-level Dev CV AUPRC",
  "audit_driven_design": {
    "core_existing": [
      "ACCESS",
      "TOPO",
      "CHEM",
      "SOLV"
    ],
    "new_blocks": {
      "SM_SEQCHEM32": 32,
      "SM_POCKET3D44": 44,
      "SM_LIGCHEM20": 20,
      "SM_TOPO16": 16,
      "SM_AROMSTACK12": 12,
      "SM_PROTCTX8": 8
    },
    "core_mechanism": "pocket accessibility + geometry/topology + aromatic/hydrophobic/sulfur chemistry",
    "electrostatics_role": "secondary/contextual, not core",
    "EVO_role": "optional control because prior small-binding EVO was not consistently beneficial",
    "pocket_reasoning": "soft residual pocket hypergraph"
  },
  "best_representation": "ESM+POCKET_GEOMETRY",

In [ ]:
# ============================================================
# SMALL V2-A.
# Does the current label-free pocket generator actually enrich
# small-binding residues? DEV ONLY.
# ============================================================

SMV2_RUN_ROOT = (
    BASE
    / 'results'
    / 'SMALL_V2_MATCHED_POCKET_REASONING'
)

SMV2_RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

rows = []

split = 'DevSet1010'

df = SPLIT_DF[split]


all_y = []
all_cov = []

per_protein = []


for r in df.itertuples():

    pid = str(
        r.prot_id
    )

    y = np.asarray(
        r.small_binding,
        dtype=np.int8
    )

    er, ep, pf = (
        POCKET_RAM[
            (
                split,
                pid
            )
        ]
    )

    covered = np.zeros(
        len(y),
        dtype=bool
    )

    if len(er):

        covered[
            np.unique(
                er
            )
        ] = True


    all_y.append(
        y
    )

    all_cov.append(
        covered
    )


    if y.sum() > 0:

        per_protein.append(
            {
                'prot_id':
                    pid,

                'n_positive':
                    int(
                        y.sum()
                    ),

                'positive_recall':
                    float(
                        covered[
                            y == 1
                        ].mean()
                    ),

                'overall_coverage':
                    float(
                        covered.mean()
                    ),
            }
        )


y = np.concatenate(
    all_y
)

cov = np.concatenate(
    all_cov
)


prevalence = float(
    y.mean()
)

covered_prevalence = float(
    y[
        cov
    ].mean()
)

positive_coverage = float(
    cov[
        y == 1
    ].mean()
)

negative_coverage = float(
    cov[
        y == 0
    ].mean()
)


audit = pd.DataFrame(
    [
        {
            'dataset':
                'DevSet1010',

            'global_prevalence':
                prevalence,

            'overall_residue_coverage':
                float(
                    cov.mean()
                ),

            'positive_residue_coverage':
                positive_coverage,

            'negative_residue_coverage':
                negative_coverage,

            'P_positive_given_covered':
                covered_prevalence,

            'covered_enrichment_vs_global':
                covered_prevalence
                /
                max(
                    prevalence,
                    1e-12
                ),

            'coverage_odds_ratio_pos_vs_neg':
                (
                    positive_coverage
                    /
                    max(
                        1-positive_coverage,
                        1e-12
                    )
                )
                /
                (
                    negative_coverage
                    /
                    max(
                        1-negative_coverage,
                        1e-12
                    )
                ),
        }
    ]
)


positive_protein_audit = pd.DataFrame(
    per_protein
)


print(
    'CURRENT POCKET CANDIDATE AUDIT'
)

display(
    audit
)


print(
    'Positive-protein pocket recall:'
)

display(
    positive_protein_audit[
        [
            'positive_recall',
            'overall_coverage'
        ]
    ].describe()
)


audit.to_csv(
    SMV2_RUN_ROOT
    / 'current_pocket_candidate_audit.csv',
    index=False
)


positive_protein_audit.to_csv(
    SMV2_RUN_ROOT
    / 'current_pocket_positive_protein_audit.csv',
    index=False
)

CURRENT POCKET CANDIDATE AUDIT


,dataset,global_prevalence,overall_residue_coverage,positive_residue_coverage,negative_residue_coverage,P_positive_given_covered,covered_enrichment_vs_global,coverage_odds_ratio_pos_vs_neg
0,DevSet1010,0.054272,0.659551,0.838133,0.649303,0.068966,1.270764,2.796671


Positive-protein pocket recall:


,positive_recall,overall_coverage
count,603.000000,603.000000
mean,0.816954,0.671121
std,0.228645,0.123167
min,0.000000,0.000000
25%,0.750000,0.610773
50%,0.888889,0.692015
75%,1.000000,0.755954
max,1.000000,0.930233


In [ ]:
# ============================================================
# SMALL V2-B.
# SAME residue encoder/head for:
#   NONE pocket message
#   MEAN pocket message
#   CROSS-ATTENTION pocket message
# ============================================================

import math


class SMV2ResidueEncoder(
    nn.Module
):

    def __init__(
        self,
        dims,
        latent=192,
        heads=6,
        drop=0.20
    ):

        super().__init__()


        assert (
            latent
            % heads
            == 0
        )


        self.keys = sorted(
            dims
        )


        self.enc = nn.ModuleDict(
            {
                k:
                    nn.Sequential(
                        nn.LayerNorm(
                            d
                        ),
                        nn.Linear(
                            d,
                            latent
                        ),
                        nn.GELU(),
                        nn.Dropout(
                            drop
                        )
                    )

                for k, d
                in dims.items()
            }
        )


        self.attn = (
            nn.MultiheadAttention(
                latent,
                heads,
                dropout=drop,
                batch_first=True
            )
        )


    def forward(
        self,
        x
    ):

        z = torch.stack(
            [
                self.enc[k](
                    x[k]
                )

                for k
                in self.keys
            ],
            dim=1
        )


        a, _ = self.attn(
            z,
            z,
            z,
            need_weights=False
        )


        return (
            z + a
        ).mean(
            dim=1
        )


class SmallPocketV2(
    nn.Module
):

    def __init__(
        self,
        dims,
        message_mode='none',
        latent=192,
        heads=6,
        drop=0.20
    ):

        super().__init__()


        assert message_mode in {
            'none',
            'mean',
            'cross'
        }


        self.message_mode = (
            message_mode
        )

        self.latent = (
            latent
        )


        self.residue_encoder = (
            SMV2ResidueEncoder(
                dims,
                latent=latent,
                heads=heads,
                drop=drop
            )
        )


        # Current V1 pocket static features are 22-D.
        self.pocket_static = (
            nn.Sequential(
                nn.LayerNorm(
                    22
                ),
                nn.Linear(
                    22,
                    64
                ),
                nn.GELU()
            )
        )


        self.pocket_encoder = (
            nn.Sequential(
                nn.Linear(
                    latent * 2
                    + 64,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    latent,
                    latent
                ),
                nn.GELU()
            )
        )


        # cross-attention projections
        self.q = nn.Linear(
            latent,
            latent,
            bias=False
        )

        self.k = nn.Linear(
            latent,
            latent,
            bias=False
        )

        self.v = nn.Linear(
            latent,
            latent,
            bias=False
        )


        self.delta = (
            nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    latent,
                    latent
                )
            )
        )


        # Residue-specific message gate.
        # Unlike V1, this can learn different patch reliance
        # for different residues.
        self.gate = (
            nn.Sequential(
                nn.LayerNorm(
                    latent * 2
                ),
                nn.Linear(
                    latent * 2,
                    1
                )
            )
        )


        nn.init.zeros_(
            self.gate[
                -1
            ].weight
        )

        # sigmoid(-1.5) ~= 0.18
        nn.init.constant_(
            self.gate[
                -1
            ].bias,
            -1.5
        )


        self.head = (
            nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    128
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    128,
                    1
                )
            )
        )


    def index_mean(
        self,
        x,
        index,
        n
    ):

        out = x.new_zeros(
            (
                n,
                x.shape[-1]
            )
        )

        cnt = x.new_zeros(
            (
                n,
                1
            )
        )


        if index.numel():

            out = out.index_add(
                0,
                index,
                x
            )

            cnt = cnt.index_add(
                0,
                index,
                x.new_ones(
                    (
                        len(index),
                        1
                    )
                )
            )


        return (
            out
            /
            cnt.clamp_min(
                1.0
            )
        )


    def index_max(
        self,
        x,
        index,
        n
    ):

        if n == 0:

            return x.new_zeros(
                (
                    0,
                    x.shape[-1]
                )
            )


        base = x.new_full(
            (
                n,
                x.shape[-1]
            ),
            -1e9
        )


        if index.numel():

            idx = index[
                :,
                None
            ].expand(
                -1,
                x.shape[-1]
            )


            out = (
                base.scatter_reduce(
                    0,
                    idx,
                    x,
                    reduce='amax',
                    include_self=True
                )
            )

        else:

            out = base


        return torch.where(
            out < -1e8,
            torch.zeros_like(
                out
            ),
            out
        )


    def forward(
        self,
        mods,
        edge_res,
        edge_pocket,
        pocket_feat,
        return_aux=False
    ):

        h = self.residue_encoder(
            mods
        )


        gate_mean = (
            h.new_tensor(
                0.0
            )
        )


        # ----------------------------------
        # NO POCKET MESSAGE CONTROL
        # ----------------------------------

        if (
            self.message_mode
            != 'none'
            and
            pocket_feat.shape[0]
            > 0
            and
            edge_res.numel()
            > 0
        ):

            nres = h.shape[0]

            npocket = (
                pocket_feat.shape[0]
            )


            # residue -> pocket
            mean = self.index_mean(
                h[
                    edge_res
                ],
                edge_pocket,
                npocket
            )


            mx = self.index_max(
                h[
                    edge_res
                ],
                edge_pocket,
                npocket
            )


            stat = (
                self.pocket_static(
                    pocket_feat
                )
            )


            hp = (
                self.pocket_encoder(
                    torch.cat(
                        [
                            mean,
                            mx,
                            stat
                        ],
                        dim=-1
                    )
                )
            )


            # ----------------------------------
            # MEAN pocket -> residue
            # ----------------------------------

            if (
                self.message_mode
                == 'mean'
            ):

                msg = self.index_mean(
                    hp[
                        edge_pocket
                    ],
                    edge_res,
                    nres
                )


            # ----------------------------------
            # CROSS-ATTENTION pocket -> residue
            # ----------------------------------

            else:

                q = self.q(
                    h[
                        edge_res
                    ]
                )


                k = self.k(
                    hp[
                        edge_pocket
                    ]
                )


                score = (
                    (
                        q * k
                    )
                    .sum(
                        dim=-1
                    )
                    /
                    math.sqrt(
                        self.latent
                    )
                )


                base = score.new_full(
                    (
                        nres,
                    ),
                    -1e9
                )


                max_r = (
                    base.scatter_reduce(
                        0,
                        edge_res,
                        score,
                        reduce='amax',
                        include_self=True
                    )
                )


                max_r = torch.where(
                    max_r < -1e8,
                    torch.zeros_like(
                        max_r
                    ),
                    max_r
                )


                w = torch.exp(
                    score
                    -
                    max_r[
                        edge_res
                    ]
                )


                denom = (
                    w.new_zeros(
                        (
                            nres,
                            1
                        )
                    )
                    .index_add(
                        0,
                        edge_res,
                        w[
                            :,
                            None
                        ]
                    )
                )


                weighted = (
                    self.v(
                        hp[
                            edge_pocket
                        ]
                    )
                    *
                    w[
                        :,
                        None
                    ]
                )


                msg = (
                    h.new_zeros(
                        (
                            nres,
                            self.latent
                        )
                    )
                    .index_add(
                        0,
                        edge_res,
                        weighted
                    )
                )


                msg = (
                    msg
                    /
                    denom.clamp_min(
                        1e-8
                    )
                )


            incident = (
                h.new_zeros(
                    (
                        nres,
                        1
                    )
                )
                .index_add(
                    0,
                    edge_res,
                    h.new_ones(
                        (
                            len(
                                edge_res
                            ),
                            1
                        )
                    )
                )
            )


            g = torch.sigmoid(
                self.gate(
                    torch.cat(
                        [
                            h,
                            msg
                        ],
                        dim=-1
                    )
                )
            )


            g = (
                g
                *
                (
                    incident > 0
                ).float()
            )


            h = (
                h
                +
                g
                *
                self.delta(
                    msg
                )
            )


            active = (
                incident
                .squeeze(
                    -1
                )
                > 0
            )


            if active.any():

                gate_mean = (
                    g[
                        active
                    ]
                    .mean()
                )


        logits = (
            self.head(
                h
            )
            .squeeze(
                -1
            )
        )


        if return_aux:

            return (
                logits,
                {
                    'gate_mean':
                        gate_mean
                }
            )


        return logits

In [ ]:
# ============================================================
# SMALL V2-C.
# Dataset + protein-balanced BCE + hard-negative ranking
# ============================================================

class SMV2Dataset(
    Dataset
):

    def __init__(
        self,
        split,
        df
    ):

        self.split = split

        self.df = df.reset_index(
            drop=True
        )

        self.store = ctx.stores[
            STORE_KEY[
                split
            ]
        ]


    def __len__(
        self
    ):

        return len(
            self.df
        )


    def __getitem__(
        self,
        i
    ):

        r = self.df.iloc[
            i
        ]


        pid = str(
            r.prot_id
        )


        mods = {

            m:
                concat_blocks(
                    self.store,
                    pid,
                    blocks
                )
                .astype(
                    np.float32,
                    copy=False
                )

            for m, blocks
            in MODALITY_SPEC.items()
        }


        er, ep, pf = (
            POCKET_RAM[
                (
                    self.split,
                    pid
                )
            ]
        )


        return {

            'pid':
                pid,

            'mods':
                mods,

            'y':
                np.asarray(
                    r.small_binding,
                    dtype=np.float32
                ),

            'edge_res':
                er,

            'edge_pocket':
                ep,

            'pocket_feat':
                pf,
        }


def smv2_collate(
    batch
):

    keys = sorted(
        batch[0][
            'mods'
        ]
    )


    mods = {

        k:
            torch.from_numpy(
                np.concatenate(
                    [
                        b[
                            'mods'
                        ][k]

                        for b
                        in batch
                    ],
                    axis=0
                )
            )
            .float()

        for k
        in keys
    }


    y = torch.from_numpy(
        np.concatenate(
            [
                b['y']
                for b in batch
            ]
        )
    ).float()


    protein_index = []


    pf_all = []
    er_all = []
    ep_all = []


    roff = 0
    poff = 0


    for pidx, b in enumerate(
        batch
    ):

        L = len(
            b['y']
        )

        P = len(
            b[
                'pocket_feat'
            ]
        )


        protein_index.append(
            torch.full(
                (
                    L,
                ),
                pidx,
                dtype=torch.long
            )
        )


        if P:

            pf_all.append(
                torch.from_numpy(
                    b[
                        'pocket_feat'
                    ]
                )
                .float()
            )


        if len(
            b[
                'edge_res'
            ]
        ):

            er_all.append(
                torch.from_numpy(
                    b[
                        'edge_res'
                    ]
                )
                .long()
                +
                roff
            )


            ep_all.append(
                torch.from_numpy(
                    b[
                        'edge_pocket'
                    ]
                )
                .long()
                +
                poff
            )


        roff += L
        poff += P


    return {

        'mods':
            mods,

        'y':
            y,

        'protein_index':
            torch.cat(
                protein_index,
                dim=0
            ),

        'edge_res':

            (
                torch.cat(
                    er_all,
                    dim=0
                )

                if er_all

                else torch.zeros(
                    (
                        0,
                    ),
                    dtype=torch.long
                )
            ),

        'edge_pocket':

            (
                torch.cat(
                    ep_all,
                    dim=0
                )

                if ep_all

                else torch.zeros(
                    (
                        0,
                    ),
                    dtype=torch.long
                )
            ),

        'pocket_feat':

            (
                torch.cat(
                    pf_all,
                    dim=0
                )

                if pf_all

                else torch.zeros(
                    (
                        0,
                        22
                    ),
                    dtype=torch.float32
                )
            ),
    }


def smv2_move(
    b
):

    return {

        'mods': {

            k:
                v.to(
                    DEVICE,
                    non_blocking=True
                )

            for k, v
            in b[
                'mods'
            ].items()
        },

        'y':
            b[
                'y'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'protein_index':
            b[
                'protein_index'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_res':
            b[
                'edge_res'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_pocket':
            b[
                'edge_pocket'
            ].to(
                DEVICE,
                non_blocking=True
            ),

        'pocket_feat':
            b[
                'pocket_feat'
            ].to(
                DEVICE,
                non_blocking=True
            ),
    }


SMV2_HARDNEG_LAMBDA = 0.15

SMV2_HARDNEG_MARGIN = 0.50


def smv2_loss(
    logits,
    y,
    protein_index,
    pos_weight,
    hardneg
):

    # --------------------------------------------------------
    # Protein-balanced weighted BCE.
    # Prevents very long proteins from dominating training.
    # --------------------------------------------------------

    raw = (
        F.binary_cross_entropy_with_logits(
            logits,
            y,
            pos_weight=pos_weight,
            reduction='none'
        )
    )


    nprot = (
        int(
            protein_index
            .max()
            .item()
        )
        +
        1
    )


    sums = (
        raw.new_zeros(
            (
                nprot,
            )
        )
        .index_add(
            0,
            protein_index,
            raw
        )
    )


    counts = (
        raw.new_zeros(
            (
                nprot,
            )
        )
        .index_add(
            0,
            protein_index,
            raw.new_ones(
                len(
                    raw
                )
            )
        )
    )


    base = (
        sums
        /
        counts.clamp_min(
            1.0
        )
    ).mean()


    if not hardneg:

        return (
            base,
            raw.new_tensor(
                0.0
            )
        )


    # --------------------------------------------------------
    # Within-protein hard-negative ranking.
    #
    # Focuses learning on the non-binding residues that the
    # model currently thinks look most pocket-like.
    # --------------------------------------------------------

    terms = []


    for p in range(
        nprot
    ):

        mask = (
            protein_index
            == p
        )


        lp = logits[
            mask
        ]


        yp = (
            y[
                mask
            ]
            >
            0.5
        )


        if (
            not yp.any()
            or
            (
                ~yp
            ).sum()
            == 0
        ):

            continue


        pos_logits = (
            lp[
                yp
            ]
        )


        neg_logits = (
            lp[
                ~yp
            ]
        )


        k = min(
            max(
                8,
                int(
                    2
                    *
                    len(
                        pos_logits
                    )
                )
            ),
            48,
            len(
                neg_logits
            )
        )


        hard = (
            torch.topk(
                neg_logits,
                k=k
            )
            .values
        )


        # Interface-level positive anchor.
        pos_anchor = (
            pos_logits.mean()
        )


        terms.append(
            F.softplus(
                hard
                -
                pos_anchor
                +
                SMV2_HARDNEG_MARGIN
            )
            .mean()
        )


    rank = (

        torch.stack(
            terms
        )
        .mean()

        if terms

        else base.new_tensor(
            0.0
        )
    )


    return (

        base
        +
        SMV2_HARDNEG_LAMBDA
        *
        rank,

        rank
    )

In [ ]:
# ============================================================
# SMALL V2-D.
# 3 message modes x 2 losses = CLEAN MATCHED EXPERIMENT
# ============================================================

SMV2_EPOCHS = 36

SMV2_PATIENCE = 6

SMV2_BATCH = 4

SMV2_LR = 3e-4

SMV2_WD = 1e-4

SMV2_CODE_VERSION = (
    'small_v2_matched_pocket_reasoning_r1'
)


SMV2_VARIANTS = {

    'NoPocket_BCE': {
        'mode':
            'none',

        'hardneg':
            False
    },

    'NoPocket_HardNeg': {
        'mode':
            'none',

        'hardneg':
            True
    },

    'MeanPocket_BCE': {
        'mode':
            'mean',

        'hardneg':
            False
    },

    'MeanPocket_HardNeg': {
        'mode':
            'mean',

        'hardneg':
            True
    },

    'CrossPocket_BCE': {
        'mode':
            'cross',

        'hardneg':
            False
    },

    'CrossPocket_HardNeg': {
        'mode':
            'cross',

        'hardneg':
            True
    },
}


SMV2_STATE_ROOT = (
    SMV2_RUN_ROOT
    / 'states'
)

SMV2_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def smv2_predict(
    model,
    ds
):

    loader = DataLoader(
        ds,
        batch_size=SMV2_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == 'cuda'
        ),
        collate_fn=smv2_collate
    )


    pp = []
    yy = []
    gg = []


    model.eval()


    with torch.no_grad():

        for b in loader:

            b = smv2_move(
                b
            )


            logits, aux = (
                model(
                    b[
                        'mods'
                    ],
                    b[
                        'edge_res'
                    ],
                    b[
                        'edge_pocket'
                    ],
                    b[
                        'pocket_feat'
                    ],
                    return_aux=True
                )
            )


            pp.append(
                torch.sigmoid(
                    logits
                )
                .cpu()
                .numpy()
            )


            yy.append(
                b[
                    'y'
                ]
                .cpu()
                .numpy()
            )


            gg.append(
                float(
                    aux[
                        'gate_mean'
                    ]
                    .detach()
                    .cpu()
                )
            )


    return (
        np.concatenate(
            pp
        ),
        np.concatenate(
            yy
        ),
        float(
            np.mean(
                gg
            )
        )
    )


def smv2_train_fold(
    variant,
    tr_ds,
    va_ds,
    fold
):

    cfg = (
        SMV2_VARIANTS[
            variant
        ]
    )


    torch.manual_seed(
        SEED
        +
        900
        +
        fold
    )


    np.random.seed(
        SEED
        +
        900
        +
        fold
    )


    model = SmallPocketV2(
        DIMS,
        message_mode=cfg[
            'mode'
        ],
        latent=192,
        heads=6,
        drop=0.20
    ).to(
        DEVICE
    )


    ytr = np.concatenate(
        tr_ds
        .df
        .small_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        )
        .tolist()
    )


    npos = max(
        float(
            ytr.sum()
        ),
        1.0
    )


    nneg = max(
        float(
            len(
                ytr
            )
            -
            ytr.sum()
        ),
        1.0
    )


    pos_weight = torch.tensor(
        np.sqrt(
            nneg
            /
            npos
        ),
        dtype=torch.float32,
        device=DEVICE
    )


    opt = torch.optim.AdamW(
        model.parameters(),
        lr=SMV2_LR,
        weight_decay=SMV2_WD
    )


    loader = DataLoader(
        tr_ds,
        batch_size=SMV2_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=(
            DEVICE.type
            == 'cuda'
        ),
        collate_fn=smv2_collate
    )


    best_ap = -1.0

    best_state = None

    bad = 0


    for epoch in range(
        1,
        SMV2_EPOCHS
        +
        1
    ):

        model.train()

        losses = []
        ranks = []


        for b in loader:

            b = smv2_move(
                b
            )


            opt.zero_grad(
                set_to_none=True
            )


            logits = model(
                b[
                    'mods'
                ],
                b[
                    'edge_res'
                ],
                b[
                    'edge_pocket'
                ],
                b[
                    'pocket_feat'
                ]
            )


            loss, rank = (
                smv2_loss(
                    logits,
                    b[
                        'y'
                    ],
                    b[
                        'protein_index'
                    ],
                    pos_weight,
                    cfg[
                        'hardneg'
                    ]
                )
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )


            opt.step()


            losses.append(
                float(
                    loss
                    .detach()
                    .cpu()
                )
            )


            ranks.append(
                float(
                    rank
                    .detach()
                    .cpu()
                )
            )


        vp, vy, gate = (
            smv2_predict(
                model,
                va_ds
            )
        )


        vap = float(
            average_precision_score(
                vy,
                vp
            )
        )


        if (
            vap
            >
            best_ap
            +
            1e-5
        ):

            best_ap = (
                vap
            )


            best_state = {

                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model
                .state_dict()
                .items()
            }


            bad = 0


        else:

            bad += 1


        if (
            epoch == 1
            or
            epoch % 5 == 0
        ):

            print(
                variant,
                'fold',
                fold,
                'epoch',
                epoch,
                'loss',
                round(
                    float(
                        np.mean(
                            losses
                        )
                    ),
                    4
                ),
                'rank',
                round(
                    float(
                        np.mean(
                            ranks
                        )
                    ),
                    4
                ),
                'val_AP',
                round(
                    vap,
                    5
                ),
                'gate',
                round(
                    gate,
                    4
                )
            )


        if (
            bad
            >=
            SMV2_PATIENCE
        ):

            break


    model.load_state_dict(
        best_state
    )


    return (
        model,
        best_state
    )


def smv2_run_cv(
    variant
):

    offsets = {}

    cursor = 0


    for r in (
        train_df.itertuples()
    ):

        L = len(
            r.sequence
        )


        offsets[
            str(
                r.prot_id
            )
        ] = (
            cursor,
            cursor + L
        )


        cursor += L


    oof = np.full(
        cursor,
        np.nan,
        dtype=np.float32
    )


    rows = []
    states = {}


    adir = (
        SMV2_STATE_ROOT
        /
        variant
    )


    adir.mkdir(
        parents=True,
        exist_ok=True
    )


    for fold, val_ids in (
        OFFICIAL_FOLDS.items()
    ):

        val_set = set(
            val_ids
        )


        tr_df = train_df[
            ~train_df
            .prot_id
            .isin(
                val_set
            )
        ].copy()


        va_df = train_df[
            train_df
            .prot_id
            .isin(
                val_set
            )
        ].copy()


        tr_ds = SMV2Dataset(
            'DevSet1010',
            tr_df
        )


        va_ds = SMV2Dataset(
            'DevSet1010',
            va_df
        )


        state_fp = (
            adir
            /
            f'fold{fold}.pt'
        )


        pred_fp = (
            adir
            /
            f'fold{fold}_val.npy'
        )


        meta_fp = (
            adir
            /
            f'fold{fold}.json'
        )


        expected = int(
            va_df
            .sequence
            .str
            .len()
            .sum()
        )


        reuse = False


        if (
            state_fp.exists()
            and
            pred_fp.exists()
            and
            meta_fp.exists()
        ):

            try:

                meta = json.loads(
                    meta_fp
                    .read_text()
                )


                pred = np.load(
                    pred_fp
                )


                reuse = (

                    meta.get(
                        'code_version'
                    )
                    ==
                    SMV2_CODE_VERSION

                    and

                    meta.get(
                        'n_val'
                    )
                    ==
                    expected

                    and

                    pred.shape
                    ==
                    (
                        expected,
                    )

                    and

                    np.all(
                        np.isfinite(
                            pred
                        )
                    )
                )


            except Exception:

                reuse = False


        if reuse:

            print(
                variant,
                'fold',
                fold,
                'REUSE'
            )


            state = torch.load(
                state_fp,
                map_location='cpu'
            )


            pred = np.load(
                pred_fp
            ).astype(
                np.float32
            )


            model = SmallPocketV2(
                DIMS,
                message_mode=SMV2_VARIANTS[
                    variant
                ][
                    'mode'
                ]
            ).to(
                DEVICE
            )


            model.load_state_dict(
                state
            )


            _, _, gate = (
                smv2_predict(
                    model,
                    va_ds
                )
            )


            del model


        else:

            print(
                variant,
                'fold',
                fold,
                'TRAIN'
            )


            model, state = (
                smv2_train_fold(
                    variant,
                    tr_ds,
                    va_ds,
                    fold
                )
            )


            pred, _, gate = (
                smv2_predict(
                    model,
                    va_ds
                )
            )


            torch.save(
                state,
                state_fp
            )


            np.save(
                pred_fp,
                pred.astype(
                    np.float32
                )
            )


            meta_fp.write_text(
                json.dumps(
                    {
                        'code_version':
                            SMV2_CODE_VERSION,

                        'n_val':
                            expected,

                        'message_mode':
                            SMV2_VARIANTS[
                                variant
                            ][
                                'mode'
                            ],

                        'hardneg':
                            SMV2_VARIANTS[
                                variant
                            ][
                                'hardneg'
                            ],
                    },
                    indent=2
                )
            )


            del model


            if (
                DEVICE.type
                ==
                'cuda'
            ):

                torch.cuda.empty_cache()


        states[
            fold
        ] = (
            state
        )


        pos = 0


        for r in (
            va_df.itertuples()
        ):

            L = len(
                r.sequence
            )


            a, b = offsets[
                str(
                    r.prot_id
                )
            ]


            oof[
                a:b
            ] = pred[
                pos:
                pos + L
            ]


            pos += L


        yv = np.concatenate(
            va_df
            .small_binding
            .apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.float32
                    )
            )
            .tolist()
        )


        rows.append(
            {
                'variant':
                    variant,

                'fold':
                    fold,

                'AUPRC':
                    float(
                        average_precision_score(
                            yv,
                            pred
                        )
                    ),

                'n_positive':
                    int(
                        yv.sum()
                    ),

                'gate_mean':
                    gate,
            }
        )


    if not np.all(
        np.isfinite(
            oof
        )
    ):

        raise RuntimeError(
            f'{variant}: incomplete OOF'
        )


    f = pd.DataFrame(
        rows
    )


    yall = np.concatenate(
        train_df
        .small_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        )
        .tolist()
    )


    return (

        {
            'variant':
                variant,

            'CV_AUPRC_mean':
                float(
                    f.AUPRC.mean()
                ),

            'CV_AUPRC_std':
                float(
                    f.AUPRC.std(
                        ddof=1
                    )
                ),

            'OOF_AUPRC':
                float(
                    average_precision_score(
                        yall,
                        oof
                    )
                ),

            'gate_mean':
                float(
                    f.gate_mean.mean()
                ),
        },

        f,

        oof,

        states
    )


SMV2_RESULTS = {}

SMV2_FOLDS = []

SMV2_OOF = {}

SMV2_STATES = {}


for variant in (
    SMV2_VARIANTS
):

    print(
        '\n'
        +
        '=' * 88
    )

    print(
        'SMALL V2:',
        variant
    )

    print(
        '=' * 88
    )


    s, f, o, st = (
        smv2_run_cv(
            variant
        )
    )


    SMV2_RESULTS[
        variant
    ] = s


    SMV2_FOLDS.append(
        f
    )


    SMV2_OOF[
        variant
    ] = o


    SMV2_STATES[
        variant
    ] = st


smv2_summary = (
    pd.DataFrame(
        SMV2_RESULTS.values()
    )
    .sort_values(
        'CV_AUPRC_mean',
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


smv2_folds = (
    pd.concat(
        SMV2_FOLDS,
        ignore_index=True
    )
)


display(
    smv2_summary
)


smv2_summary.to_csv(
    SMV2_RUN_ROOT
    / 'v2_cv_summary.csv',
    index=False
)


smv2_folds.to_csv(
    SMV2_RUN_ROOT
    / 'v2_cv_folds.csv',
    index=False
)


def smv2score(
    name
):

    return float(
        smv2_summary.loc[
            smv2_summary
            .variant
            .eq(
                name
            ),
            'CV_AUPRC_mean'
        ]
        .iloc[0]
    )


# ============================================================
# FACTORIAL INTERPRETATION
# ============================================================

matched = pd.DataFrame(
    [

        {
            'comparison':
                'Mean pocket effect under BCE',

            'delta_AUPRC':
                smv2score(
                    'MeanPocket_BCE'
                )
                -
                smv2score(
                    'NoPocket_BCE'
                )
        },

        {
            'comparison':
                'Cross-attention pocket effect under BCE',

            'delta_AUPRC':
                smv2score(
                    'CrossPocket_BCE'
                )
                -
                smv2score(
                    'NoPocket_BCE'
                )
        },

        {
            'comparison':
                'Hard-negative effect without pocket',

            'delta_AUPRC':
                smv2score(
                    'NoPocket_HardNeg'
                )
                -
                smv2score(
                    'NoPocket_BCE'
                )
        },

        {
            'comparison':
                'Hard-negative effect with mean pocket',

            'delta_AUPRC':
                smv2score(
                    'MeanPocket_HardNeg'
                )
                -
                smv2score(
                    'MeanPocket_BCE'
                )
        },

        {
            'comparison':
                'Hard-negative effect with cross pocket',

            'delta_AUPRC':
                smv2score(
                    'CrossPocket_HardNeg'
                )
                -
                smv2score(
                    'CrossPocket_BCE'
                )
        },

        {
            'comparison':
                'Cross vs mean under BCE',

            'delta_AUPRC':
                smv2score(
                    'CrossPocket_BCE'
                )
                -
                smv2score(
                    'MeanPocket_BCE'
                )
        },

        {
            'comparison':
                'Cross vs mean under HardNeg',

            'delta_AUPRC':
                smv2score(
                    'CrossPocket_HardNeg'
                )
                -
                smv2score(
                    'MeanPocket_HardNeg'
                )
        },
    ]
)


display(
    matched
)


matched.to_csv(
    SMV2_RUN_ROOT
    / 'matched_ablation.csv',
    index=False
)


# ============================================================
# DEV-ONLY complexity-aware lock
# ============================================================

complexity = {

    'NoPocket_BCE':
        0,

    'NoPocket_HardNeg':
        1,

    'MeanPocket_BCE':
        2,

    'MeanPocket_HardNeg':
        3,

    'CrossPocket_BCE':
        4,

    'CrossPocket_HardNeg':
        5,
}


ordered = sorted(
    SMV2_VARIANTS,
    key=lambda x:
        complexity[x]
)


winner = (
    ordered[0]
)


for name in (
    ordered[1:]
):

    if (
        smv2score(
            name
        )
        -
        smv2score(
            winner
        )
        >=
        0.002
    ):

        winner = name


SMV2_LOCKED = (
    winner
)


SMV2_LOCKED_SCORE = (
    smv2score(
        winner
    )
)


print(
    '\nSMALL V2 LOCKED:',
    SMV2_LOCKED,
    SMV2_LOCKED_SCORE
)


SMALL V2: NoPocket_BCE
NoPocket_BCE fold 1 TRAIN
NoPocket_BCE fold 1 epoch 1 loss 0.4937 rank 0.0 val_AP 0.3442 gate 0.0
NoPocket_BCE fold 1 epoch 5 loss 0.2614 rank 0.0 val_AP 0.3827 gate 0.0
NoPocket_BCE fold 1 epoch 10 loss 0.0814 rank 0.0 val_AP 0.34608 gate 0.0
NoPocket_BCE fold 2 TRAIN
NoPocket_BCE fold 2 epoch 1 loss 0.5076 rank 0.0 val_AP 0.33229 gate 0.0
NoPocket_BCE fold 2 epoch 5 loss 0.2762 rank 0.0 val_AP 0.42006 gate 0.0
NoPocket_BCE fold 2 epoch 10 loss 0.0897 rank 0.0 val_AP 0.40616 gate 0.0
NoPocket_BCE fold 3 TRAIN
NoPocket_BCE fold 3 epoch 1 loss 0.4933 rank 0.0 val_AP 0.29158 gate 0.0
NoPocket_BCE fold 3 epoch 5 loss 0.2615 rank 0.0 val_AP 0.34057 gate 0.0
NoPocket_BCE fold 3 epoch 10 loss 0.0806 rank 0.0 val_AP 0.3166 gate 0.0
NoPocket_BCE fold 4 TRAIN
NoPocket_BCE fold 4 epoch 1 loss 0.4912 rank 0.0 val_AP 0.28675 gate 0.0
NoPocket_BCE fold 4 epoch 5 loss 0.2654 rank 0.0 val_AP 0.34923 gate 0.0
NoPocket_BCE fold 4 epoch 10 loss 0.0841 rank 0.0 val_AP 0.32345 gate

,variant,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,gate_mean
0,CrossPocket_BCE,0.394885,0.036723,0.383378,0.090126
1,MeanPocket_BCE,0.394698,0.037089,0.381720,0.092045
2,MeanPocket_HardNeg,0.382200,0.035177,0.374096,0.092272
3,CrossPocket_HardNeg,0.379733,0.035740,0.374419,0.090339
4,NoPocket_BCE,0.378191,0.032748,0.367651,0.000000
5,NoPocket_HardNeg,0.365820,0.031550,0.355170,0.000000


,comparison,delta_AUPRC
0,Mean pocket effect under BCE,0.016507
1,Cross-attention pocket effect under BCE,0.016694
2,Hard-negative effect without pocket,-0.012372
3,Hard-negative effect with mean pocket,-0.012498
4,Hard-negative effect with cross pocket,-0.015152
5,Cross vs mean under BCE,0.000187
6,Cross vs mean under HardNeg,-0.002468



SMALL V2 LOCKED: MeanPocket_BCE 0.39469813862633585


In [ ]:
# ============================================================
# SMALL V2-E.
# External confirmation of DEV-LOCKED winner
# ============================================================

def smv2_best_mcc_threshold(
    y,
    p
):

    best = (
        -999,
        None
    )


    for t in np.linspace(
        0.01,
        0.99,
        199
    ):

        pred = (
            p >= t
        ).astype(
            int
        )


        m = matthews_corrcoef(
            y,
            pred
        )


        if m > best[0]:

            best = (
                m,
                float(t)
            )


    return best[1]


dev_y_smv2 = np.concatenate(
    train_df
    .small_binding
    .apply(
        lambda x:
            np.asarray(
                x,
                dtype=np.int8
            )
    )
    .tolist()
)


SMV2_THR = (
    smv2_best_mcc_threshold(
        dev_y_smv2,
        SMV2_OOF[
            SMV2_LOCKED
        ]
    )
)


def smv2_predict_external(
    split
):

    ds = SMV2Dataset(
        split,
        SPLIT_DF[
            split
        ]
    )


    pp = []


    for fold, state in (
        SMV2_STATES[
            SMV2_LOCKED
        ].items()
    ):

        model = SmallPocketV2(
            DIMS,
            message_mode=SMV2_VARIANTS[
                SMV2_LOCKED
            ][
                'mode'
            ],
            latent=192,
            heads=6,
            drop=0.20
        ).to(
            DEVICE
        )


        model.load_state_dict(
            state
        )


        p, _, _ = (
            smv2_predict(
                model,
                ds
            )
        )


        pp.append(
            p
        )


        del model


        if (
            DEVICE.type
            ==
            'cuda'
        ):

            torch.cuda.empty_cache()


    return np.mean(
        np.stack(
            pp
        ),
        axis=0
    )


def smv2_metrics(
    y,
    p,
    threshold
):

    pred = (
        p
        >=
        threshold
    ).astype(
        int
    )


    return {

        'AUPRC':
            float(
                average_precision_score(
                    y,
                    p
                )
            ),

        'AUROC':
            float(
                roc_auc_score(
                    y,
                    p
                )
            ),

        'precision':
            float(
                precision_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'recall':
            float(
                recall_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'F1':
            float(
                f1_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        'MCC':
            float(
                matthews_corrcoef(
                    y,
                    pred
                )
            ),
    }


ext = []


for split in [
    'TestSet300',
    'TestSetNew46'
]:

    y = np.concatenate(
        SPLIT_DF[
            split
        ]
        .small_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.int8
                )
        )
        .tolist()
    )


    p = (
        smv2_predict_external(
            split
        )
    )


    m05 = (
        smv2_metrics(
            y,
            p,
            0.5
        )
    )


    mdev = (
        smv2_metrics(
            y,
            p,
            SMV2_THR
        )
    )


    row = {

        'dataset':
            split,

        'variant':
            SMV2_LOCKED,

        'Dev_CV_AUPRC':
            SMV2_LOCKED_SCORE,

        'Dev_threshold':
            SMV2_THR,
    }


    row.update(
        {
            f'{k}_fixed0.5':
                v

            for k, v
            in m05.items()
        }
    )


    row.update(
        {
            f'{k}_OOFthr':
                v

            for k, v
            in mdev.items()
        }
    )


    ext.append(
        row
    )


smv2_external = pd.DataFrame(
    ext
)


display(
    smv2_external
)


smv2_external.to_csv(
    SMV2_RUN_ROOT
    / 'FINAL_V2_EXTERNAL.csv',
    index=False
)


# ============================================================
# Compare V2 vs V1 vs reproduced baselines
# ============================================================

compare = []


for _, r in headline.iterrows():

    compare.append(
        {
            'dataset': r['dataset'],

            'method': f'V1 {GLOBAL_MODEL}',

            'AUPRC': float(
                r['AUPRC_fixed0.5']
            ),

            'AUROC': float(
                r['AUROC_fixed0.5']
            ),

            'F1': float(
                r['F1_fixed0.5']
            ),

            'MCC': float(
                r['MCC_fixed0.5']
            ),
        }
    )


# V1
for _, r in headline.iterrows():

    compare.append(
        {
            'dataset': r['dataset'],

            'method': f'V1 {GLOBAL_MODEL}',

            'AUPRC': float(
                r['AUPRC_fixed0.5']
            ),

            'AUROC': float(
                r['AUROC_fixed0.5']
            ),

            'F1': float(
                r['F1_fixed0.5']
            ),

            'MCC': float(
                r['MCC_fixed0.5']
            ),
        }
    )


# reproduced direct baselines
baseline_fp = (
    BASELINE_WORK
    / 'results'
    / 'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
)


if baseline_fp.exists():

    bb = pd.read_csv(
        baseline_fp
    )


    if 'label' in bb.columns:

        bb = bb[
            bb.label.eq(
                'small_binding'
            )
        ].copy()


    if (
        'operating_point'
        in bb.columns
    ):

        bb = bb[
            bb.operating_point.eq(
                'fixed_0.5'
            )
        ].copy()


    for _, r in (
        bb.iterrows()
    ):

        if str(
            r.method
        ) not in {
            'bindNode24',
            'bindEmbed21DL'
        }:

            continue


        apcol = (

            'AP_AUPRC'

            if (
                'AP_AUPRC'
                in bb.columns
            )

            else 'AUPRC'
        )


        compare.append(
            {

                'dataset':
                    str(
                        r.dataset
                    ),

                'method':
                    str(
                        r.method
                    ),

                'AUPRC':
                    float(
                        r[
                            apcol
                        ]
                    ),

                'AUROC':
                    float(
                        r.AUROC
                    ),

                'F1':
                    float(
                        r.global_positive_F1
                    ),

                'MCC':
                    float(
                        r.global_MCC
                    ),
            }
        )


smv2_compare = (
    pd.DataFrame(
        compare
    )
    .sort_values(
        [
            'dataset',
            'AUPRC'
        ],
        ascending=[
            True,
            False
        ]
    )
)


display(
    smv2_compare
)


smv2_compare.to_csv(
    SMV2_RUN_ROOT
    / 'FINAL_V2_COMPARISON.csv',
    index=False
)


print(
    '\nSaved:',
    SMV2_RUN_ROOT
)

,dataset,variant,Dev_CV_AUPRC,Dev_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
0,TestSet300,MeanPocket_BCE,0.394698,0.618788,0.428841,0.863689,0.441561,0.492320,0.465561,0.428709,0.428841,0.863689,0.503226,0.399386,0.445333,0.416059
1,TestSetNew46,MeanPocket_BCE,0.394698,0.618788,0.415326,0.862839,0.426887,0.425882,0.426384,0.384420,0.415326,0.862839,0.473016,0.350588,0.402703,0.370498


,dataset,method,AUPRC,AUROC,F1,MCC
0,TestSet300,V1 SM_PocketHG_Attention,0.440255,0.867647,0.473903,0.437803
2,TestSet300,V1 SM_PocketHG_Attention,0.440255,0.867647,0.473903,0.437803
6,TestSet300,bindNode24,0.431709,0.869662,0.468245,0.432216
4,TestSet300,bindEmbed21DL,0.394041,0.864302,0.435777,0.396702
1,TestSetNew46,V1 SM_PocketHG_Attention,0.424158,0.865581,0.440964,0.401205
3,TestSetNew46,V1 SM_PocketHG_Attention,0.424158,0.865581,0.440964,0.401205
7,TestSetNew46,bindNode24,0.380681,0.872435,0.407911,0.367430
5,TestSetNew46,bindEmbed21DL,0.315583,0.857311,0.374857,0.327813



Saved: /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_V2_MATCHED_POCKET_REASONING


In [ ]:
# ============================================================
# SMALL FINAL EXPERIMENT
# V1 Attention architecture + multi-scale pocket bank
# One-cell: build cache -> Dev CV -> external comparison
# ============================================================
from pathlib import Path
from collections import Counter
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.spatial import cKDTree


SMMS_ROOT = (
    BASE
    / 'results'
    / 'SMALL_V1_ATTENTION_MULTISCALE_POCKETS'
)

SMMS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


SMMS_CACHE = (
    BASE
    / 'cache'
    / 'small_multiscale_pockets_v1'
)

SMMS_CACHE.mkdir(
    parents=True,
    exist_ok=True
)


SMMS_RADII = (
    6.0,
    8.0,
    10.0,
    12.0
)

SMMS_MAX = 64

SMMS_NMS = 0.65

SMMS_FORCE = False


def _jaccard(
    a,
    b
):

    a = set(a)
    b = set(b)

    return (
        len(
            a & b
        )
        /
        max(
            len(
                a | b
            ),
            1
        )
    )


def _build_multiscale_pockets(
    split,
    pid,
    seq
):

    coords = structure_coords(
        split,
        pid,
        seq
    )

    tree = cKDTree(
        coords
    )

    candidates = []


    for seed, aa in enumerate(
        seq
    ):

        if aa not in POCKET_SEEDS:
            continue


        for radius in SMMS_RADII:

            members = tuple(
                sorted(
                    tree.query_ball_point(
                        coords[
                            seed
                        ],
                        radius
                    )
                )
            )


            if len(
                members
            ) < 4:
                continue


            feat22 = (
                pocket_static_features(
                    seq,
                    coords,
                    members
                )
            )


            # Append radius explicitly.
            feat23 = np.concatenate(
                [
                    feat22.astype(
                        np.float32
                    ),

                    np.array(
                        [
                            radius
                            / 12.0
                        ],
                        dtype=np.float32
                    )
                ]
            )


            # Original V1 pocket score
            # + very mild preference against
            # unnecessarily large radius.
            score = (

                1.4
                * feat22[1]

                + 0.8
                * feat22[2]

                + 0.8
                * feat22[3]

                + 0.5
                * feat22[6]

                + 0.5
                * feat22[7]

                - 0.5
                * feat22[5]

                + 0.25
                * feat22[0]

                + 0.15
                * feat22[17]

                - 0.05
                * (
                    radius
                    / 12.0
                )
            )


            candidates.append(
                (
                    float(
                        score
                    ),
                    members,
                    feat23
                )
            )


    # ========================================================
    # Exact duplicate removal
    # ========================================================

    unique = {}


    for (
        score,
        members,
        feat
    ) in candidates:

        if (
            members not in unique
            or
            score
            >
            unique[
                members
            ][0]
        ):

            unique[
                members
            ] = (
                score,
                feat
            )


    ranked = sorted(
        [
            (
                s,
                m,
                f
            )

            for m, (
                s,
                f
            )
            in unique.items()
        ],

        key=lambda x:
            x[0],

        reverse=True
    )


    # ========================================================
    # Jaccard NMS
    # ========================================================

    selected = []


    for item in ranked:

        (
            score,
            members,
            feat
        ) = item


        if any(
            _jaccard(
                members,
                old_members
            )
            >=
            SMMS_NMS

            for _,
            old_members,
            _
            in selected
        ):
            continue


        selected.append(
            item
        )


        if len(
            selected
        ) >= SMMS_MAX:
            break


    edge_res = []
    edge_pocket = []
    pocket_feat = []


    for pidx, (
        _,
        members,
        feat
    ) in enumerate(
        selected
    ):

        pocket_feat.append(
            feat
        )


        for rid in members:

            edge_res.append(
                int(
                    rid
                )
            )

            edge_pocket.append(
                int(
                    pidx
                )
            )


    return (

        np.asarray(
            edge_res,
            dtype=np.int64
        ),

        np.asarray(
            edge_pocket,
            dtype=np.int64
        ),

        np.asarray(
            pocket_feat,
            dtype=np.float32
        ).reshape(
            -1,
            23
        )
    )


# ============================================================
# Build/resume pocket bank
# ============================================================

SMMS_RAM = {}

audit_rows = []


for split, df in (
    SPLIT_DF.items()
):

    od = (
        SMMS_CACHE
        / split
    )

    od.mkdir(
        parents=True,
        exist_ok=True
    )


    for n, r in enumerate(
        df.itertuples(),
        start=1
    ):

        fp = (
            od
            / f'{r.prot_id}.npz'
        )


        good = False


        if (
            fp.exists()
            and
            not SMMS_FORCE
        ):

            try:

                z = np.load(
                    fp
                )


                good = (

                    z[
                        'pocket_feat'
                    ].ndim
                    == 2

                    and

                    z[
                        'pocket_feat'
                    ].shape[1]
                    == 23

                    and

                    len(
                        z[
                            'edge_res'
                        ]
                    )
                    ==
                    len(
                        z[
                            'edge_pocket'
                        ]
                    )
                )


            except Exception:

                good = False


        if not good:

            (
                er,
                ep,
                pf
            ) = (
                _build_multiscale_pockets(
                    split,
                    str(
                        r.prot_id
                    ),
                    r.sequence
                )
            )


            np.savez_compressed(
                fp,
                edge_res=er,
                edge_pocket=ep,
                pocket_feat=pf
            )


        z = np.load(
            fp
        )


        er = z[
            'edge_res'
        ].astype(
            np.int64
        )


        ep = z[
            'edge_pocket'
        ].astype(
            np.int64
        )


        pf = z[
            'pocket_feat'
        ].astype(
            np.float32
        )


        SMMS_RAM[
            (
                split,
                str(
                    r.prot_id
                )
            )
        ] = (
            er,
            ep,
            pf
        )


        covered = (
            len(
                np.unique(
                    er
                )
            )
            if len(er)
            else 0
        )


        audit_rows.append(
            {
                'dataset':
                    split,

                'prot_id':
                    str(
                        r.prot_id
                    ),

                'n_pockets':
                    len(
                        pf
                    ),

                'coverage':
                    covered
                    /
                    max(
                        len(
                            r.sequence
                        ),
                        1
                    ),

                'memberships':
                    len(
                        er
                    ),
            }
        )


        if n % 250 == 0:

            print(
                split,
                n,
                '/',
                len(df)
            )


smms_audit = pd.DataFrame(
    audit_rows
)


print(
    "\nMULTI-SCALE POCKET AUDIT"
)


display(
    smms_audit
    .groupby(
        'dataset'
    )
    .agg(
        mean_pockets=(
            'n_pockets',
            'mean'
        ),

        median_pockets=(
            'n_pockets',
            'median'
        ),

        mean_coverage=(
            'coverage',
            'mean'
        ),

        mean_memberships=(
            'memberships',
            'mean'
        ),
    )
    .reset_index()
)


smms_audit.to_csv(
    SMMS_ROOT
    / 'pocket_audit.csv',
    index=False
)


# ============================================================
# Same V1 Attention architecture.
# Only pocket static dimension 22 -> 23.
# ============================================================

class SmallPocketHypergraphMS(
    SmallPocketHypergraph
):

    def __init__(
        self,
        dims,
        mode='attention',
        latent=192,
        drop=0.2
    ):

        super().__init__(
            dims,
            mode=mode,
            latent=latent,
            moe=False,
            drop=drop
        )


        self.pocket_static = (
            nn.Sequential(
                nn.LayerNorm(
                    23
                ),
                nn.Linear(
                    23,
                    64
                ),
                nn.GELU()
            )
        )


def collate_pocket_ms(
    batch
):

    keys = sorted(
        batch[0][
            'mods'
        ]
    )


    mods = {

        k:
            torch.from_numpy(
                np.concatenate(
                    [
                        b[
                            'mods'
                        ][k]

                        for b
                        in batch
                    ],
                    axis=0
                )
            )
            .float()

        for k
        in keys
    }


    y = torch.from_numpy(
        np.concatenate(
            [
                b[
                    'y'
                ]
                for b
                in batch
            ]
        )
    ).float()


    pf_all = []
    er_all = []
    ep_all = []


    roff = 0
    poff = 0


    for b in batch:

        L = len(
            b[
                'y'
            ]
        )


        P = len(
            b[
                'pocket_feat'
            ]
        )


        if P:

            pf_all.append(
                torch.from_numpy(
                    b[
                        'pocket_feat'
                    ]
                )
                .float()
            )


        if len(
            b[
                'edge_res'
            ]
        ):

            er_all.append(
                torch.from_numpy(
                    b[
                        'edge_res'
                    ]
                )
                .long()
                +
                roff
            )


            ep_all.append(
                torch.from_numpy(
                    b[
                        'edge_pocket'
                    ]
                )
                .long()
                +
                poff
            )


        roff += L
        poff += P


    return {

        'mods':
            mods,

        'y':
            y,

        'edge_res':

            (
                torch.cat(
                    er_all,
                    0
                )

                if er_all

                else torch.zeros(
                    (
                        0,
                    ),
                    dtype=torch.long
                )
            ),

        'edge_pocket':

            (
                torch.cat(
                    ep_all,
                    0
                )

                if ep_all

                else torch.zeros(
                    (
                        0,
                    ),
                    dtype=torch.long
                )
            ),

        'pocket_feat':

            (
                torch.cat(
                    pf_all,
                    0
                )

                if pf_all

                else torch.zeros(
                    (
                        0,
                        23
                    ),
                    dtype=torch.float32
                )
            ),
    }


# ============================================================
# Swap ONLY pocket bank/factory temporarily.
# ============================================================

_OLD_POCKET_RAM = (
    POCKET_RAM
)

_OLD_POCKET_FACTORY = (
    POCKET_FACTORY
)

_OLD_POCKET_STATE_ROOT = (
    POCKET_STATE_ROOT
)

_OLD_COLLATE_POCKET = (
    collate_pocket
)


POCKET_RAM = (
    SMMS_RAM
)


POCKET_FACTORY = {

    'SM_PocketHG_Attention_MS':

        lambda dims:
            SmallPocketHypergraphMS(
                dims,
                mode='attention',
                latent=POCKET_LATENT,
                drop=0.2
            )
}


POCKET_STATE_ROOT = (
    SMMS_ROOT
    / 'states'
)


POCKET_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


collate_pocket = (
    collate_pocket_ms
)


try:

    print(
        "=" * 100
    )

    print(
        "SMALL: V1 ATTENTION "
        "+ MULTI-SCALE POCKET BANK"
    )

    print(
        "=" * 100
    )


    (
        ms_summary,
        ms_folds,
        ms_oof,
        ms_states
    ) = run_pocket_cv(
        'SM_PocketHG_Attention_MS'
    )


    ms_summary_df = pd.DataFrame(
        [
            ms_summary
        ]
    )


    display(
        ms_summary_df
    )


    display(
        ms_folds
    )


    ms_summary_df.to_csv(
        SMMS_ROOT
        / 'dev_cv_summary.csv',
        index=False
    )


    ms_folds.to_csv(
        SMMS_ROOT
        / 'dev_cv_folds.csv',
        index=False
    )


    MS_DEV = float(
        ms_summary[
            'CV_AUPRC_mean'
        ]
    )


    V1_DEV = float(
        pocket_summary.loc[
            pocket_summary
            .architecture
            .eq(
                'SM_PocketHG_Attention'
            ),
            'CV_AUPRC_mean'
        ].iloc[0]
    )


    print(
        f"\nV1 Attention Dev mean AUPRC : "
        f"{V1_DEV:.6f}"
    )


    print(
        f"MS Attention Dev mean AUPRC : "
        f"{MS_DEV:.6f}"
    )


    print(
        f"Delta MS - V1               : "
        f"{MS_DEV - V1_DEV:+.6f}"
    )


    MS_WINS = (
        MS_DEV
        -
        V1_DEV
        >=
        0.002
    )


    print(
        "MS becomes new headline?     :",
        MS_WINS
    )


    # ========================================================
    # Frozen Dev threshold
    # ========================================================

    dev_y = np.concatenate(
        train_df
        .small_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.int8
                )
        )
        .tolist()
    )


    (
        ms_thr,
        ms_oof_mcc
    ) = best_mcc_threshold(
        dev_y,
        ms_oof
    )


    # ========================================================
    # External prediction
    # ========================================================

    def _predict_ms_external(
        split
    ):

        ds = PocketProteinDataset(
            split,
            SPLIT_DF[
                split
            ]
        )


        preds = []


        for fold, state in (
            ms_states.items()
        ):

            model = (
                POCKET_FACTORY[
                    'SM_PocketHG_Attention_MS'
                ](
                    DIMS
                )
                .to(
                    DEVICE
                )
            )


            model.load_state_dict(
                state
            )


            p, _ = predict_pocket(
                model,
                ds
            )


            preds.append(
                p
            )


            del model


            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()


        return np.mean(
            np.stack(
                preds
            ),
            axis=0
        )


    ext_rows = []


    for split in [
        'TestSet300',
        'TestSetNew46'
    ]:

        y = np.concatenate(
            SPLIT_DF[
                split
            ]
            .small_binding
            .apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.int8
                    )
            )
            .tolist()
        )


        p = _predict_ms_external(
            split
        )


        m05 = binary_metrics(
            y,
            p,
            0.5
        )


        mdev = binary_metrics(
            y,
            p,
            ms_thr
        )


        row = {

            'dataset':
                split,

            'method':
                'SM_PocketHG_Attention_MS',

            'Dev_CV_AUPRC':
                MS_DEV,

            'Dev_OOF_MCC_threshold':
                ms_thr,
        }


        row.update(
            {
                f'{k}_fixed0.5':
                    v

                for k, v
                in m05.items()
            }
        )


        row.update(
            {
                f'{k}_OOFthr':
                    v

                for k, v
                in mdev.items()
            }
        )


        ext_rows.append(
            row
        )


    ms_external = pd.DataFrame(
        ext_rows
    )


    display(
        ms_external
    )


    ms_external.to_csv(
        SMMS_ROOT
        / 'external_metrics.csv',
        index=False
    )


    # ========================================================
    # Correct comparison:
    # new MS + V1 + bind24 + bind21
    # ========================================================

    cmp_rows = []


    for _, r in (
        ms_external.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r[
                        'dataset'
                    ],

                'method':
                    'MS PocketHG Attention',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    # V1 current headline
    for _, r in (
        headline.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r[
                        'dataset'
                    ],

                'method':
                    f'V1 {GLOBAL_MODEL}',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    baseline_fp = (
        BASELINE_WORK
        / 'results'
        / 'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
    )


    if baseline_fp.exists():

        bb = pd.read_csv(
            baseline_fp
        )


        if 'label' in bb.columns:

            bb = bb[
                bb.label.eq(
                    'small_binding'
                )
            ].copy()


        if (
            'operating_point'
            in bb.columns
        ):

            bb = bb[
                bb.operating_point.eq(
                    'fixed_0.5'
                )
            ].copy()


        for _, r in (
            bb.iterrows()
        ):

            if str(
                r.method
            ) not in {
                'bindNode24',
                'bindEmbed21DL'
            }:

                continue


            apcol = (
                'AP_AUPRC'
                if (
                    'AP_AUPRC'
                    in bb.columns
                )
                else 'AUPRC'
            )


            cmp_rows.append(
                {
                    'dataset':
                        str(
                            r.dataset
                        ),

                    'method':
                        str(
                            r.method
                        ),

                    'AUPRC':
                        float(
                            r[
                                apcol
                            ]
                        ),

                    'AUROC':
                        float(
                            r.AUROC
                        ),

                    'F1':
                        float(
                            r.global_positive_F1
                        ),

                    'MCC':
                        float(
                            r.global_MCC
                        ),
                }
            )


    ms_compare = (
        pd.DataFrame(
            cmp_rows
        )
        .sort_values(
            [
                'dataset',
                'AUPRC'
            ],
            ascending=[
                True,
                False
            ]
        )
    )


    print(
        "\nFINAL SMALL COMPARISON"
    )


    display(
        ms_compare
    )


    ms_compare.to_csv(
        SMMS_ROOT
        / 'FINAL_COMPARISON.csv',
        index=False
    )


finally:

    POCKET_RAM = (
        _OLD_POCKET_RAM
    )

    POCKET_FACTORY = (
        _OLD_POCKET_FACTORY
    )

    POCKET_STATE_ROOT = (
        _OLD_POCKET_STATE_ROOT
    )

    collate_pocket = (
        _OLD_COLLATE_POCKET
    )


print(
    "\nSaved:",
    SMMS_ROOT
)

DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300

MULTI-SCALE POCKET AUDIT


,dataset,mean_pockets,median_pockets,mean_coverage,mean_memberships
0,DevSet1010,52.470297,64.0,0.834215,811.611881
1,TestSet300,57.490000,64.0,0.802372,939.083333
2,TestSetNew46,45.586957,53.0,0.848493,687.173913


SMALL: V1 ATTENTION + MULTI-SCALE POCKET BANK
SM_PocketHG_Attention_MS fold 1 TRAIN
SM_PocketHG_Attention_MS fold 1 epoch 1 loss 0.4653 val_AP 0.34943 best 0.34943 alpha 0.0498
SM_PocketHG_Attention_MS fold 1 epoch 5 loss 0.2515 val_AP 0.38301 best 0.4029 alpha 0.0496
SM_PocketHG_Attention_MS fold 1 epoch 10 loss 0.0741 val_AP 0.38112 best 0.4029 alpha 0.0497
SM_PocketHG_Attention_MS fold 2 TRAIN
SM_PocketHG_Attention_MS fold 2 epoch 1 loss 0.4682 val_AP 0.3282 best 0.3282 alpha 0.0499
SM_PocketHG_Attention_MS fold 2 epoch 5 loss 0.2541 val_AP 0.42011 best 0.42105 alpha 0.0498
SM_PocketHG_Attention_MS fold 2 epoch 10 loss 0.0697 val_AP 0.40156 best 0.42216 alpha 0.0498
SM_PocketHG_Attention_MS fold 3 TRAIN
SM_PocketHG_Attention_MS fold 3 epoch 1 loss 0.4616 val_AP 0.30804 best 0.30804 alpha 0.0499
SM_PocketHG_Attention_MS fold 3 epoch 5 loss 0.2562 val_AP 0.3618 best 0.3618 alpha 0.0496
SM_PocketHG_Attention_MS fold 3 epoch 10 loss 0.075 val_AP 0.32257 best 0.3618 alpha 0.0496
SM_Pocke

,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,SM_PocketHG_Attention_MS,0.392769,0.029363,0.372801


,architecture,fold,AUPRC,n_val,n_positive
0,SM_PocketHG_Attention_MS,1,0.402902,32977,2069
1,SM_PocketHG_Attention_MS,2,0.422164,34703,1838
2,SM_PocketHG_Attention_MS,3,0.361797,34034,1629
3,SM_PocketHG_Attention_MS,4,0.361232,33622,1688
4,SM_PocketHG_Attention_MS,5,0.415750,34845,2012



V1 Attention Dev mean AUPRC : 0.397647
MS Attention Dev mean AUPRC : 0.392769
Delta MS - V1               : -0.004878
MS becomes new headline?     : False


,dataset,method,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
0,TestSet300,SM_PocketHG_Attention_MS,0.392769,0.465354,0.441991,0.863503,0.511378,0.425755,0.464655,0.434571,0.441991,0.863503,0.489739,0.452125,0.470181,0.436865
1,TestSetNew46,SM_PocketHG_Attention_MS,0.392769,0.465354,0.419632,0.857929,0.452381,0.357647,0.399474,0.363788,0.419632,0.857929,0.453039,0.385882,0.416773,0.379085



FINAL SMALL COMPARISON


,dataset,method,AUPRC,AUROC,F1,MCC
0,TestSet300,MS PocketHG Attention,0.441991,0.863503,0.464655,0.434571
2,TestSet300,V1 SM_PocketHG_Attention,0.440255,0.867647,0.473903,0.437803
6,TestSet300,bindNode24,0.431709,0.869662,0.468245,0.432216
4,TestSet300,bindEmbed21DL,0.394041,0.864302,0.435777,0.396702
3,TestSetNew46,V1 SM_PocketHG_Attention,0.424158,0.865581,0.440964,0.401205
1,TestSetNew46,MS PocketHG Attention,0.419632,0.857929,0.399474,0.363788
7,TestSetNew46,bindNode24,0.380681,0.872435,0.407911,0.367430
5,TestSetNew46,bindEmbed21DL,0.315583,0.857311,0.374857,0.327813



Saved: /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_V1_ATTENTION_MULTISCALE_POCKETS


In [ ]:
# ============================================================
# SMALL FINAL ONE-CELL
# Dev-lock V1 + Multi-scale ensemble, then report ALL metrics
# including protein-mean F1/MCC and bindEmbed21DL/bindNode24.
#
# Requires completed Small V1 + multi-scale cells:
# POCKET_OOF, POCKET_STATES, SMMS_RAM, ms_oof, ms_states,
# SmallPocketHypergraphMS, collate_pocket_ms, etc.
# ============================================================
from pathlib import Path
import json, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    matthews_corrcoef, precision_score, recall_score
)

REQ = [
    "POCKET_OOF","POCKET_STATES","SMMS_RAM","ms_oof","ms_states",
    "SmallPocketHypergraphMS","collate_pocket_ms","PocketProteinDataset",
    "predict_pocket","POCKET_FACTORY","POCKET_RAM","collate_pocket",
    "train_df","SPLIT_DF","OFFICIAL_FOLDS","BASELINE_WORK"
]
missing = [x for x in REQ if x not in globals()]
if missing:
    raise RuntimeError(
        "Run the completed Small V1 + MS cells first. Missing: "
        + ", ".join(missing)
    )

OUT = BASE/'results'/'SMALL_FINAL_ONECELL'
OUT.mkdir(parents=True, exist_ok=True)

V1_NAME = 'SM_PocketHG_Attention'
v1_oof = np.asarray(POCKET_OOF[V1_NAME],dtype=np.float32)
ms_oof_ = np.asarray(ms_oof,dtype=np.float32)

y_dev=np.concatenate(
    train_df.small_binding.apply(
        lambda x:np.asarray(x,dtype=np.int8)
    ).tolist()
)

assert v1_oof.shape == ms_oof_.shape == y_dev.shape

# ------------------------------------------------------------
# 1. Official fold residue indices
# ------------------------------------------------------------
offsets,cur={},0
for r in train_df.itertuples():
    L=len(r.sequence)
    offsets[str(r.prot_id)]=(cur,cur+L)
    cur+=L

fold_idx={}
for fold,ids in OFFICIAL_FOLDS.items():
    z=[]
    for pid in ids:
        a,b=offsets[str(pid)]
        z.extend(range(a,b))
    fold_idx[fold]=np.asarray(z,dtype=np.int64)

# ------------------------------------------------------------
# 2. DEV-only weight search
# ------------------------------------------------------------
rows=[]
for w in np.linspace(0,1,21):
    p=w*v1_oof+(1-w)*ms_oof_
    aps=[
        average_precision_score(y_dev[idx],p[idx])
        for _,idx in sorted(fold_idx.items())
    ]
    rows.append({
        'w_V1':float(w),
        'w_MS':float(1-w),
        'CV_AUPRC_mean':float(np.mean(aps)),
        'CV_AUPRC_std':float(np.std(aps,ddof=1)),
        'OOF_AUPRC':float(average_precision_score(y_dev,p)),
        **{f'fold{i+1}_AUPRC':float(v) for i,v in enumerate(aps)}
    })

weight_search=pd.DataFrame(rows).sort_values(
    'CV_AUPRC_mean',ascending=False
).reset_index(drop=True)

display(weight_search.head(10))
weight_search.to_csv(OUT/'DEV_WEIGHT_SEARCH.csv',index=False)

best=weight_search.iloc[0]
w1,wm=float(best.w_V1),float(best.w_MS)
ens_dev=float(best.CV_AUPRC_mean)

v1_dev=float(
    pocket_summary.loc[
        pocket_summary.architecture.eq(V1_NAME),
        'CV_AUPRC_mean'
    ].iloc[0]
)

PROMOTE=(ens_dev-v1_dev)>=0.002
if not PROMOTE:
    w1,wm=1.0,0.0
    ens_dev=v1_dev

ens_oof=w1*v1_oof+wm*ms_oof_

def best_mcc_thr(y,p):
    best=(-999,.5)
    for t in np.linspace(.01,.99,199):
        m=matthews_corrcoef(y,(p>=t).astype(int))
        if m>best[0]:
            best=(m,float(t))
    return best[1],best[0]

ens_thr,ens_oof_mcc=best_mcc_thr(y_dev,ens_oof)

print("\nDEV FINAL")
print("w_V1 =",w1,"w_MS =",wm)
print("V1 Dev =",v1_dev)
print("Final Dev =",ens_dev)
print("Delta =",ens_dev-v1_dev)
print("Promote ensemble =",PROMOTE)
print("Frozen Dev threshold =",ens_thr)
print("OOF MCC =",ens_oof_mcc)

# ------------------------------------------------------------
# 3. External predictors
# ------------------------------------------------------------
def predict_v1(split):
    ds=PocketProteinDataset(split,SPLIT_DF[split])
    pred=[]
    for fold,state in POCKET_STATES[V1_NAME].items():
        model=POCKET_FACTORY[V1_NAME](DIMS).to(DEVICE)
        model.load_state_dict(state)
        p,_=predict_pocket(model,ds)
        pred.append(p)
        del model
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return np.mean(np.stack(pred),axis=0)

def predict_ms(split):
    global POCKET_RAM,POCKET_FACTORY,collate_pocket
    old_ram,old_fac,old_col=POCKET_RAM,POCKET_FACTORY,collate_pocket
    try:
        POCKET_RAM=SMMS_RAM
        POCKET_FACTORY={
            'SM_PocketHG_Attention_MS':
                lambda dims:SmallPocketHypergraphMS(
                    dims,mode='attention',
                    latent=POCKET_LATENT,drop=.2
                )
        }
        collate_pocket=collate_pocket_ms
        ds=PocketProteinDataset(split,SPLIT_DF[split])
        pred=[]
        for fold,state in ms_states.items():
            model=POCKET_FACTORY['SM_PocketHG_Attention_MS'](DIMS).to(DEVICE)
            model.load_state_dict(state)
            p,_=predict_pocket(model,ds)
            pred.append(p)
            del model
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
        return np.mean(np.stack(pred),axis=0)
    finally:
        POCKET_RAM,POCKET_FACTORY,collate_pocket=old_ram,old_fac,old_col

# ------------------------------------------------------------
# 4. Metrics
# ------------------------------------------------------------
def global_metrics(y,p,t=.5):
    pr=(p>=t).astype(int)
    return {
        'AUPRC':float(average_precision_score(y,p)),
        'AUROC':float(roc_auc_score(y,p)),
        'precision':float(precision_score(y,pr,zero_division=0)),
        'recall':float(recall_score(y,pr,zero_division=0)),
        'F1':float(f1_score(y,pr,zero_division=0)),
        'MCC':float(matthews_corrcoef(y,pr)),
    }

def protein_metrics(df,p,t=.5):
    f1s,mccs=[],[]
    pos=0
    for r in df.itertuples():
        L=len(r.sequence)
        y=np.asarray(r.small_binding,dtype=int)
        pr=(p[pos:pos+L]>=t).astype(int)
        f1s.append(f1_score(y,pr,zero_division=0))
        mccs.append(matthews_corrcoef(y,pr))
        pos+=L
    return {
        'protein_mean_F1':float(np.mean(f1s)),
        'protein_mean_MCC':float(np.mean(mccs)),
    }

our_rows=[]

for split in ['TestSet300','TestSetNew46']:
    df=SPLIT_DF[split]
    y=np.concatenate(df.small_binding.apply(
        lambda x:np.asarray(x,dtype=int)
    ).tolist())

    pv1=predict_v1(split)
    pms=predict_ms(split)
    pens=w1*pv1+wm*pms

    for name,p in [
        ('V1_Attention',pv1),
        ('MS_Attention',pms),
        ('FINAL_DEV_LOCKED',pens),
    ]:
        g=global_metrics(y,p,.5)
        pm=protein_metrics(df,p,.5)
        our_rows.append({
            'dataset':split,'method':name,
            **g,**pm
        })

    pids=[];ridx=[]
    for r in df.itertuples():
        pids.extend([str(r.prot_id)]*len(r.sequence))
        ridx.extend(range(len(r.sequence)))

    pd.DataFrame({
        'prot_id':pids,
        'residue_index':ridx,
        'y_true':y,
        'probability':pens
    }).to_csv(
        OUT/f'FINAL_predictions_{split}.csv.gz',
        index=False
    )

ours=pd.DataFrame(our_rows)

# ------------------------------------------------------------
# 5. Direct reproduced baselines
# ------------------------------------------------------------
bfp=BASELINE_WORK/'results'/'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
bb=pd.read_csv(bfp)
bb=bb[bb.label.eq('small_binding')].copy()
if 'operating_point' in bb.columns:
    bb=bb[bb.operating_point.eq('fixed_0.5')].copy()

base_rows=[]
for _,r in bb.iterrows():
    if str(r.method) not in {'bindNode24','bindEmbed21DL'}:
        continue
    apcol='AP_AUPRC' if 'AP_AUPRC' in bb.columns else 'AUPRC'
    base_rows.append({
        'dataset':str(r.dataset),
        'method':str(r.method),
        'AUPRC':float(r[apcol]),
        'AUROC':float(r.AUROC),
        'precision':np.nan,
        'recall':np.nan,
        'F1':float(r.global_positive_F1),
        'MCC':float(r.global_MCC),
        'protein_mean_F1':float(r.protein_mean_F1),
        'protein_mean_MCC':float(r.protein_mean_MCC),
    })

final=pd.concat(
    [ours,pd.DataFrame(base_rows)],
    ignore_index=True
).sort_values(
    ['dataset','AUPRC'],
    ascending=[True,False]
)

print("\nFINAL SMALL — ALL AVAILABLE METRICS")
display(final)

final.to_csv(OUT/'FINAL_ALL_METRICS.csv',index=False)

decision={
    'w_V1':w1,'w_MS':wm,
    'V1_Dev_mean_AUPRC':v1_dev,
    'Final_Dev_mean_AUPRC':ens_dev,
    'Dev_gain':ens_dev-v1_dev,
    'Promoted':bool(PROMOTE),
    'Frozen_Dev_MCC_threshold':ens_thr,
}
(OUT/'FINAL_DECISION.json').write_text(json.dumps(decision,indent=2))
print("\nDecision:",decision)
print("Saved:",OUT)

,w_V1,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
0,0.60,0.40,0.409283,0.035424,0.395044,0.412509,0.458149,0.374628,0.375385,0.425746
1,0.55,0.45,0.409256,0.035038,0.395299,0.412701,0.457238,0.374688,0.375733,0.425922
2,0.65,0.35,0.409065,0.035797,0.394464,0.412071,0.458758,0.374274,0.374847,0.425376
3,0.50,0.50,0.409037,0.034590,0.395260,0.412780,0.456011,0.374847,0.375676,0.425870
4,0.45,0.55,0.408678,0.034145,0.394942,0.412877,0.454558,0.374607,0.375656,0.425691
5,0.70,0.30,0.408658,0.036168,0.393617,0.411329,0.459137,0.373738,0.374131,0.424957
6,0.75,0.25,0.408066,0.036586,0.392425,0.410514,0.459329,0.372861,0.373225,0.424402
7,0.40,0.60,0.408055,0.033690,0.394334,0.412767,0.452773,0.374213,0.375246,0.425273
8,0.35,0.65,0.407274,0.033119,0.393408,0.412503,0.450695,0.373863,0.374732,0.424580
9,0.80,0.20,0.407190,0.037073,0.390902,0.409508,0.459382,0.371795,0.371819,0.423447



DEV FINAL
w_V1 = 0.6000000000000001 w_MS = 0.3999999999999999
V1 Dev = 0.39764655317751607
Final Dev = 0.4092833208152622
Delta = 0.011636767637746126
Promote ensemble = True
Frozen Dev threshold = 0.5148484848484849
OOF MCC = 0.3929680466162692


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use th


FINAL SMALL — ALL AVAILABLE METRICS


,dataset,method,AUPRC,AUROC,precision,recall,F1,MCC,protein_mean_F1,protein_mean_MCC
2,TestSet300,FINAL_DEV_LOCKED,0.446015,0.868490,0.478455,0.466206,0.472251,0.437710,0.321952,0.310649
1,TestSet300,MS_Attention,0.441991,0.863503,0.511378,0.425755,0.464655,0.434571,0.314404,0.306961
0,TestSet300,V1_Attention,0.440255,0.867647,0.454417,0.495136,0.473903,0.437803,0.324798,0.311897
8,TestSet300,bindNode24,0.431709,0.869662,NaN,NaN,0.468245,0.432216,0.353203,0.337026
6,TestSet300,bindEmbed21DL,0.394041,0.864302,NaN,NaN,0.435777,0.396702,0.331199,0.307604
5,TestSetNew46,FINAL_DEV_LOCKED,0.427383,0.865695,0.461957,0.400000,0.428752,0.391272,0.221530,0.199216
3,TestSetNew46,V1_Attention,0.424158,0.865581,0.451852,0.430588,0.440964,0.401205,0.223885,0.199532
4,TestSetNew46,MS_Attention,0.419632,0.857929,0.452381,0.357647,0.399474,0.363788,0.191811,0.170538
9,TestSetNew46,bindNode24,0.380681,0.872435,NaN,NaN,0.407911,0.367430,0.300463,0.266746
7,TestSetNew46,bindEmbed21DL,0.315583,0.857311,NaN,NaN,0.374857,0.327813,0.285816,0.253609



Decision: {'w_V1': 0.6000000000000001, 'w_MS': 0.3999999999999999, 'V1_Dev_mean_AUPRC': 0.39764655317751607, 'Final_Dev_mean_AUPRC': 0.4092833208152622, 'Dev_gain': 0.011636767637746126, 'Promoted': True, 'Frozen_Dev_MCC_threshold': 0.5148484848484849}
Saved: /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_ONECELL


In [ ]:
# ============================================================
# SMALL FINAL PAPER-READY REPORT — ONE CELL
#
# Run AFTER completed V1 + multi-scale training cells.
#
# What this cell does:
# 1) Dev-only V1/MS ensemble weight lock (official 5 folds)
# 2) Cross-fitted ensemble audit (selection-bias sanity check)
# 3) Dev OOF threshold locks for EACH model:
#       - fixed 0.5
#       - MCC-optimal
#       - positive-class F1-optimal
#       - Youden-J-optimal
# 4) External Test300/Test46 metrics at ALL locked thresholds
# 5) Direct fixed-0.5 comparison to bindEmbed21DL / bindNode24
# 6) Correct metric names:
#       - global residue-level positive-class F1
#       - global residue-level MCC
#       - formal micro-F1 across both classes (= accuracy here)
#       - protein-mean F1/MCC using baseline convention:
#         include protein iff TP+FP+FN > 0; exclude TN-only
# 7) Saves paper-ready CSV/JSON and per-residue predictions.
# ============================================================

from pathlib import Path
import json, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    accuracy_score, balanced_accuracy_score
)

TASK_NAME = "SMALL"
LABEL_COL = "small_binding"
V1_NAME = "SM_PocketHG_Attention"
MS_NAME = "SM_PocketHG_Attention_MS"
FINAL_NAME = "FINAL_DEV_LOCKED_ENSEMBLE"
MIN_DEV_GAIN = 0.002
WEIGHT_GRID = np.linspace(0.0, 1.0, 21)

OUT = BASE / "results" / "SMALL_FINAL_PAPER_READY"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# A. Required globals
# ------------------------------------------------------------
REQ = [
    "POCKET_OOF", "POCKET_STATES", "SMMS_RAM",
    "ms_oof", "ms_states",
    "SmallPocketHypergraphMS", "collate_pocket_ms",
    "PocketProteinDataset", "predict_pocket",
    "POCKET_FACTORY", "POCKET_RAM", "collate_pocket",
    "train_df", "SPLIT_DF", "OFFICIAL_FOLDS",
    "BASELINE_WORK", "pocket_summary",
    "DIMS", "DEVICE", "POCKET_LATENT"
]
missing = [x for x in REQ if x not in globals()]
if missing:
    raise RuntimeError(
        "Run the completed V1 + multi-scale cells first. Missing: "
        + ", ".join(missing)
    )

# ------------------------------------------------------------
# B. Metric helpers
# ------------------------------------------------------------
def _counts(y, pred):
    y = np.asarray(y, dtype=np.int8)
    pred = np.asarray(pred, dtype=np.int8)
    tp = int(np.sum((y == 1) & (pred == 1)))
    tn = int(np.sum((y == 0) & (pred == 0)))
    fp = int(np.sum((y == 0) & (pred == 1)))
    fn = int(np.sum((y == 1) & (pred == 0)))
    return tp, tn, fp, fn

def _mcc_counts(tp, tn, fp, fn):
    den = ((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)) ** 0.5
    return 0.0 if den == 0 else float((tp*tn - fp*fn) / den)

def _safe_div(a, b):
    return 0.0 if b == 0 else float(a / b)

def full_global_metrics(y, p, threshold):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)
    pred = (p >= float(threshold)).astype(np.int8)
    tp, tn, fp, fn = _counts(y, pred)

    precision_pos = _safe_div(tp, tp+fp)
    recall_pos = _safe_div(tp, tp+fn)
    specificity = _safe_div(tn, tn+fp)
    npv = _safe_div(tn, tn+fn)
    f1_pos = _safe_div(2*tp, 2*tp+fp+fn)

    # Formal micro-F1 across both classes.
    # For binary single-label prediction this equals accuracy.
    micro_f1 = float(f1_score(y, pred, average="micro", zero_division=0))
    macro_f1 = float(f1_score(y, pred, average="macro", zero_division=0))

    return {
        "n_residues": int(len(y)),
        "n_positive": int(np.sum(y == 1)),
        "prevalence": float(np.mean(y)),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,

        # Threshold-free pooled/global residue-level ranking metrics
        "AUPRC_global_residue": float(average_precision_score(y, p)),
        "AUROC_global_residue": float(roc_auc_score(y, p)),

        # Threshold-dependent pooled/global residue-level metrics
        "Accuracy_global_residue": float(accuracy_score(y, pred)),
        "Balanced_accuracy_global_residue": float(
            balanced_accuracy_score(y, pred)
        ),
        "Precision_positive_global_residue": precision_pos,
        "Recall_positive_global_residue": recall_pos,
        "Sensitivity_global_residue": recall_pos,
        "Specificity_global_residue": specificity,
        "NPV_global_residue": npv,
        "F1_positive_global_residue": f1_pos,
        "MCC_global_residue": _mcc_counts(tp, tn, fp, fn),

        # Formal class-averaged F1 metrics
        "F1_micro_all_classes": micro_f1,
        "F1_macro_two_classes": macro_f1,
    }

def protein_mean_metrics_baseline_convention(df, p, threshold):
    """
    Same aggregation convention used by the reproduced direct baseline:
    calculate each protein separately and INCLUDE it iff TP+FP+FN > 0.
    Therefore completely TN-only proteins are excluded.
    """
    p = np.asarray(p, dtype=np.float64)
    pos = 0
    f1s, mccs = [], []
    n_total = 0
    n_included = 0
    n_excluded_tn_only = 0

    for r in df.itertuples():
        L = len(r.sequence)
        y = np.asarray(getattr(r, LABEL_COL), dtype=np.int8)
        pred = (p[pos:pos+L] >= float(threshold)).astype(np.int8)
        pos += L
        n_total += 1

        tp, tn, fp, fn = _counts(y, pred)

        if (tp + fp + fn) == 0:
            n_excluded_tn_only += 1
            continue

        n_included += 1
        f1s.append(_safe_div(2*tp, 2*tp+fp+fn))
        mccs.append(_mcc_counts(tp, tn, fp, fn))

    assert pos == len(p)

    return {
        "protein_mean_F1_baseline_convention":
            float(np.mean(f1s)) if f1s else np.nan,
        "protein_mean_MCC_baseline_convention":
            float(np.mean(mccs)) if mccs else np.nan,
        "protein_n_total": int(n_total),
        "protein_n_included_baseline_convention": int(n_included),
        "protein_n_excluded_TN_only": int(n_excluded_tn_only),
    }

def optimize_thresholds_exact(y, p):
    """
    Exact search at unique OOF probability cutoffs for pred = (p >= threshold).
    Returns thresholds optimized ONLY on Dev OOF predictions.
    """
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)

    order = np.argsort(-p, kind="mergesort")
    ys = y[order]
    ps = p[order]

    tp_c = np.cumsum(ys == 1)
    fp_c = np.cumsum(ys == 0)

    # End index of each equal-score group so >= threshold includes ties.
    ends = np.r_[np.where(ps[:-1] != ps[1:])[0], len(ps)-1]

    tp = tp_c[ends].astype(np.float64)
    fp = fp_c[ends].astype(np.float64)
    P = float(np.sum(y == 1))
    N = float(np.sum(y == 0))
    fn = P - tp
    tn = N - fp
    thr = ps[ends]

    den = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc = np.divide(
        tp*tn - fp*fn,
        den,
        out=np.zeros_like(tp),
        where=den > 0
    )

    f1 = np.divide(
        2*tp,
        2*tp + fp + fn,
        out=np.zeros_like(tp),
        where=(2*tp + fp + fn) > 0
    )

    tpr = np.divide(tp, P, out=np.zeros_like(tp), where=P > 0)
    tnr = np.divide(tn, N, out=np.zeros_like(tn), where=N > 0)
    youden = tpr + tnr - 1.0

    i_mcc = int(np.nanargmax(mcc))
    i_f1 = int(np.nanargmax(f1))
    i_y = int(np.nanargmax(youden))

    return {
        "fixed_0.5": 0.5,
        "dev_oof_mcc_opt": float(thr[i_mcc]),
        "dev_oof_positive_f1_opt": float(thr[i_f1]),
        "dev_oof_youden_j_opt": float(thr[i_y]),
    }

# ------------------------------------------------------------
# C. Dev OOF + official-fold indices
# ------------------------------------------------------------
V1_OOF = np.asarray(POCKET_OOF[V1_NAME], dtype=np.float64)
MS_OOF = np.asarray(ms_oof, dtype=np.float64)

Y_DEV = np.concatenate(
    train_df[LABEL_COL].apply(
        lambda x: np.asarray(x, dtype=np.int8)
    ).tolist()
)

assert V1_OOF.shape == MS_OOF.shape == Y_DEV.shape
assert np.all(np.isfinite(V1_OOF))
assert np.all(np.isfinite(MS_OOF))

offsets, cur = {}, 0
for r in train_df.itertuples():
    L = len(r.sequence)
    offsets[str(r.prot_id)] = (cur, cur+L)
    cur += L

FOLD_IDX = {}
for fold, ids in OFFICIAL_FOLDS.items():
    z = []
    for pid in ids:
        a, b = offsets[str(pid)]
        z.extend(range(a, b))
    FOLD_IDX[fold] = np.asarray(z, dtype=np.int64)

# ------------------------------------------------------------
# D. Dev-only ensemble weight lock
# ------------------------------------------------------------
weight_rows = []

for w in WEIGHT_GRID:
    p = w*V1_OOF + (1.0-w)*MS_OOF
    fold_ap = []

    for fold in sorted(FOLD_IDX):
        idx = FOLD_IDX[fold]
        fold_ap.append(
            float(average_precision_score(Y_DEV[idx], p[idx]))
        )

    weight_rows.append({
        "w_V1": float(w),
        "w_MS": float(1.0-w),
        "CV_AUPRC_mean": float(np.mean(fold_ap)),
        "CV_AUPRC_std": float(np.std(fold_ap, ddof=1)),
        "OOF_AUPRC": float(average_precision_score(Y_DEV, p)),
        **{
            f"fold{i+1}_AUPRC": float(v)
            for i, v in enumerate(fold_ap)
        }
    })

weight_search = pd.DataFrame(weight_rows).sort_values(
    ["CV_AUPRC_mean", "w_V1"],
    ascending=[False, False]
).reset_index(drop=True)

best = weight_search.iloc[0]
W_V1_CAND = float(best.w_V1)
W_MS_CAND = float(best.w_MS)
ENS_DEV_CAND = float(best.CV_AUPRC_mean)

V1_DEV = float(
    pocket_summary.loc[
        pocket_summary.architecture.eq(V1_NAME),
        "CV_AUPRC_mean"
    ].iloc[0]
)

PROMOTE = (ENS_DEV_CAND - V1_DEV) >= MIN_DEV_GAIN

if PROMOTE:
    W_V1, W_MS = W_V1_CAND, W_MS_CAND
    FINAL_DEV = ENS_DEV_CAND
else:
    W_V1, W_MS = 1.0, 0.0
    FINAL_DEV = V1_DEV

FINAL_OOF = W_V1*V1_OOF + W_MS*MS_OOF

print("="*100)
print(f"{TASK_NAME} DEV ENSEMBLE LOCK")
print("="*100)
display(weight_search.head(10))
print("Candidate weights:", W_V1_CAND, W_MS_CAND)
print("Candidate Dev mean AUPRC:", ENS_DEV_CAND)
print("V1 Dev mean AUPRC:", V1_DEV)
print("Candidate gain:", ENS_DEV_CAND - V1_DEV)
print("Promote by >=0.002 rule:", PROMOTE)
print("FINAL weights:", W_V1, W_MS)
print("FINAL Dev mean AUPRC:", FINAL_DEV)

weight_search.to_csv(
    OUT / "DEV_ENSEMBLE_WEIGHT_SEARCH.csv",
    index=False
)

# ------------------------------------------------------------
# E. Cross-fitted ensemble audit
#    For each held fold, choose weight using only other 4 folds.
# ------------------------------------------------------------
cf_rows = []

for held_fold in sorted(FOLD_IDX):
    train_folds = [f for f in sorted(FOLD_IDX) if f != held_fold]

    cand = []
    for w in WEIGHT_GRID:
        p = w*V1_OOF + (1.0-w)*MS_OOF
        aps = [
            average_precision_score(
                Y_DEV[FOLD_IDX[f]],
                p[FOLD_IDX[f]]
            )
            for f in train_folds
        ]
        cand.append((float(np.mean(aps)), float(w)))

    # Higher mean AP first; on exact tie prefer more V1.
    cand.sort(key=lambda x: (x[0], x[1]), reverse=True)
    _, wcf = cand[0]
    pm = 1.0 - wcf

    idx = FOLD_IDX[held_fold]
    pcf = wcf*V1_OOF[idx] + pm*MS_OOF[idx]

    cf_rows.append({
        "held_fold": held_fold,
        "selected_w_V1_from_other_4_folds": wcf,
        "selected_w_MS_from_other_4_folds": pm,
        "held_fold_AUPRC_crossfit_ensemble":
            float(average_precision_score(Y_DEV[idx], pcf)),
        "held_fold_AUPRC_V1":
            float(average_precision_score(Y_DEV[idx], V1_OOF[idx])),
        "held_fold_AUPRC_MS":
            float(average_precision_score(Y_DEV[idx], MS_OOF[idx])),
    })

crossfit = pd.DataFrame(cf_rows)
crossfit.to_csv(
    OUT / "DEV_CROSSFIT_ENSEMBLE_AUDIT.csv",
    index=False
)

print("\nCross-fitted ensemble audit:")
display(crossfit)
print(
    "Crossfit mean AUPRC ensemble =",
    crossfit.held_fold_AUPRC_crossfit_ensemble.mean()
)
print(
    "Crossfit mean AUPRC V1       =",
    crossfit.held_fold_AUPRC_V1.mean()
)

# ------------------------------------------------------------
# F. Lock thresholds independently for V1, MS, FINAL from Dev OOF
# ------------------------------------------------------------
DEV_PROBS = {
    "V1_Attention": V1_OOF,
    "MS_Attention": MS_OOF,
    FINAL_NAME: FINAL_OOF,
}

THRESHOLDS = {}
threshold_rows = []
dev_metric_rows = []

for method, p in DEV_PROBS.items():
    THRESHOLDS[method] = optimize_thresholds_exact(Y_DEV, p)

    for rule, thr in THRESHOLDS[method].items():
        threshold_rows.append({
            "method": method,
            "threshold_rule": rule,
            "threshold": float(thr),
            "threshold_selected_on": (
                "fixed" if rule == "fixed_0.5" else "DevSet1010 OOF only"
            ),
        })

        m = full_global_metrics(Y_DEV, p, thr)
        pm = protein_mean_metrics_baseline_convention(
            train_df, p, thr
        )

        dev_metric_rows.append({
            "dataset": "DevSet1010_OOF",
            "method": method,
            "threshold_rule": rule,
            "threshold": float(thr),
            **m, **pm
        })

threshold_df = pd.DataFrame(threshold_rows)
dev_metrics = pd.DataFrame(dev_metric_rows)

threshold_df.to_csv(
    OUT / "DEV_THRESHOLD_LOCK.csv",
    index=False
)
dev_metrics.to_csv(
    OUT / "DEV_OOF_FULL_METRICS.csv",
    index=False
)

print("\nDEV threshold locks:")
display(threshold_df)

# ------------------------------------------------------------
# G. External prediction helpers with prediction cache
# ------------------------------------------------------------
PRED_CACHE = OUT / "prediction_cache"
PRED_CACHE.mkdir(parents=True, exist_ok=True)

def _load_cached(method, split, expected_len):
    fp = PRED_CACHE / f"{method}__{split}.npy"
    if fp.exists():
        try:
            p = np.load(fp)
            if p.shape == (expected_len,) and np.all(np.isfinite(p)):
                return p.astype(np.float64)
        except Exception:
            pass
    return None

def _save_cached(method, split, p):
    np.save(
        PRED_CACHE / f"{method}__{split}.npy",
        np.asarray(p, dtype=np.float32)
    )

def predict_v1_external(split):
    expected = int(sum(SPLIT_DF[split].sequence.str.len()))
    cached = _load_cached("V1_Attention", split, expected)
    if cached is not None:
        return cached

    ds = PocketProteinDataset(split, SPLIT_DF[split])
    preds = []

    for fold, state in POCKET_STATES[V1_NAME].items():
        model = POCKET_FACTORY[V1_NAME](DIMS).to(DEVICE)
        model.load_state_dict(state)
        p, _ = predict_pocket(model, ds)
        preds.append(p)
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    p = np.mean(np.stack(preds), axis=0)
    _save_cached("V1_Attention", split, p)
    return p.astype(np.float64)

def predict_ms_external(split):
    expected = int(sum(SPLIT_DF[split].sequence.str.len()))
    cached = _load_cached("MS_Attention", split, expected)
    if cached is not None:
        return cached

    global POCKET_RAM, POCKET_FACTORY, collate_pocket

    old_ram = POCKET_RAM
    old_factory = POCKET_FACTORY
    old_collate = collate_pocket

    try:
        POCKET_RAM = SMMS_RAM
        POCKET_FACTORY = {
            MS_NAME:
                lambda dims: SmallPocketHypergraphMS(
                    dims,
                    mode="attention",
                    latent=POCKET_LATENT,
                    drop=0.2
                )
        }
        collate_pocket = collate_pocket_ms

        ds = PocketProteinDataset(split, SPLIT_DF[split])
        preds = []

        for fold, state in ms_states.items():
            model = POCKET_FACTORY[MS_NAME](DIMS).to(DEVICE)
            model.load_state_dict(state)
            p, _ = predict_pocket(model, ds)
            preds.append(p)
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        p = np.mean(np.stack(preds), axis=0)
        _save_cached("MS_Attention", split, p)
        return p.astype(np.float64)

    finally:
        POCKET_RAM = old_ram
        POCKET_FACTORY = old_factory
        collate_pocket = old_collate

# ------------------------------------------------------------
# H. External FULL metrics at all Dev-locked thresholds
# ------------------------------------------------------------
external_rows = []
final_prediction_files = {}

for split in ["TestSet300", "TestSetNew46"]:
    df = SPLIT_DF[split]
    y = np.concatenate(
        df[LABEL_COL].apply(
            lambda x: np.asarray(x, dtype=np.int8)
        ).tolist()
    )

    p_v1 = predict_v1_external(split)
    p_ms = predict_ms_external(split)
    p_final = W_V1*p_v1 + W_MS*p_ms

    pred_map = {
        "V1_Attention": p_v1,
        "MS_Attention": p_ms,
        FINAL_NAME: p_final,
    }

    for method, p in pred_map.items():
        for rule, thr in THRESHOLDS[method].items():
            gm = full_global_metrics(y, p, thr)
            pm = protein_mean_metrics_baseline_convention(df, p, thr)

            external_rows.append({
                "dataset": split,
                "method": method,
                "threshold_rule": rule,
                "threshold": float(thr),
                "threshold_selected_on": (
                    "fixed" if rule == "fixed_0.5"
                    else "DevSet1010 OOF only"
                ),
                "directly_comparable_to_baseline_fixed0.5":
                    bool(rule == "fixed_0.5"),
                **gm, **pm
            })

    # Save final ensemble per-residue predictions
    pids, ridx = [], []
    for r in df.itertuples():
        L = len(r.sequence)
        pids.extend([str(r.prot_id)] * L)
        ridx.extend(range(L))

    pred_df = pd.DataFrame({
        "prot_id": pids,
        "residue_index": ridx,
        "y_true": y,
        "prob_V1": p_v1,
        "prob_MS": p_ms,
        "prob_FINAL": p_final,
    })

    pred_fp = OUT / f"FINAL_RESIDUE_PREDICTIONS_{split}.csv.gz"
    pred_df.to_csv(pred_fp, index=False)
    final_prediction_files[split] = str(pred_fp)

external_full = pd.DataFrame(external_rows)

# ------------------------------------------------------------
# I. Add direct baselines — fixed 0.5 only
# ------------------------------------------------------------
def _first_existing(row, candidates, default=np.nan):
    for c in candidates:
        if c in row.index and pd.notna(row[c]):
            return float(row[c])
    return default

baseline_fp = (
    BASELINE_WORK
    / "results"
    / "FULL_DIRECT_BASELINE_METRICS_300_46.csv"
)

bb = pd.read_csv(baseline_fp)
bb = bb[bb.label.eq(LABEL_COL)].copy()

if "operating_point" in bb.columns:
    bb = bb[bb.operating_point.eq("fixed_0.5")].copy()

baseline_rows = []

for _, r in bb.iterrows():
    if str(r.method) not in {"bindNode24", "bindEmbed21DL"}:
        continue

    ap = _first_existing(r, ["AP_AUPRC", "AUPRC"])
    auroc = _first_existing(r, ["AUROC"])
    acc = _first_existing(r, ["accuracy", "Accuracy", "global_accuracy"])
    prec = _first_existing(
        r,
        ["global_positive_precision", "precision", "Precision"]
    )
    rec = _first_existing(
        r,
        ["global_positive_recall", "recall", "Recall"]
    )
    f1p = _first_existing(
        r,
        ["global_positive_F1", "F1", "f1"]
    )
    mcc = _first_existing(
        r,
        ["global_MCC", "MCC"]
    )
    pmf1 = _first_existing(
        r,
        ["protein_mean_F1"]
    )
    pmmcc = _first_existing(
        r,
        ["protein_mean_MCC"]
    )

    baseline_rows.append({
        "dataset": str(r.dataset),
        "method": str(r.method),
        "threshold_rule": "fixed_0.5",
        "threshold": 0.5,
        "threshold_selected_on": "reproduced baseline operating point",
        "directly_comparable_to_baseline_fixed0.5": True,

        "n_residues": np.nan,
        "n_positive": np.nan,
        "prevalence": np.nan,
        "TP": np.nan, "TN": np.nan, "FP": np.nan, "FN": np.nan,

        "AUPRC_global_residue": ap,
        "AUROC_global_residue": auroc,
        "Accuracy_global_residue": acc,
        "Balanced_accuracy_global_residue": np.nan,
        "Precision_positive_global_residue": prec,
        "Recall_positive_global_residue": rec,
        "Sensitivity_global_residue": rec,
        "Specificity_global_residue": np.nan,
        "NPV_global_residue": np.nan,
        "F1_positive_global_residue": f1p,
        "MCC_global_residue": mcc,

        # Formal micro-F1 equals accuracy for binary single-label,
        # but only fill when reproduced baseline accuracy exists.
        "F1_micro_all_classes": acc,
        "F1_macro_two_classes": np.nan,

        "protein_mean_F1_baseline_convention": pmf1,
        "protein_mean_MCC_baseline_convention": pmmcc,
        "protein_n_total": np.nan,
        "protein_n_included_baseline_convention": np.nan,
        "protein_n_excluded_TN_only": np.nan,
    })

all_metrics = pd.concat(
    [external_full, pd.DataFrame(baseline_rows)],
    ignore_index=True,
    sort=False
)

all_metrics.to_csv(
    OUT / "EXTERNAL_FULL_METRICS_LONG.csv",
    index=False
)

# ------------------------------------------------------------
# J. Paper-ready fixed-0.5 headline comparison
# ------------------------------------------------------------
headline_cols = [
    "dataset", "method", "threshold",
    "AUPRC_global_residue",
    "AUROC_global_residue",
    "Accuracy_global_residue",
    "Precision_positive_global_residue",
    "Recall_positive_global_residue",
    "F1_positive_global_residue",
    "MCC_global_residue",
    "F1_micro_all_classes",
    "F1_macro_two_classes",
    "protein_mean_F1_baseline_convention",
    "protein_mean_MCC_baseline_convention",
]

headline = all_metrics[
    all_metrics.threshold_rule.eq("fixed_0.5")
][headline_cols].copy()

headline = headline.sort_values(
    ["dataset", "AUPRC_global_residue"],
    ascending=[True, False]
)

headline.to_csv(
    OUT / "HEADLINE_FIXED05_COMPARISON.csv",
    index=False
)

# ------------------------------------------------------------
# K. Paper-ready FINAL model threshold table
# ------------------------------------------------------------
final_threshold_report = external_full[
    external_full.method.eq(FINAL_NAME)
].copy()

final_threshold_report = final_threshold_report[
    [
        "dataset", "method", "threshold_rule", "threshold",
        "AUPRC_global_residue", "AUROC_global_residue",
        "Accuracy_global_residue",
        "Balanced_accuracy_global_residue",
        "Precision_positive_global_residue",
        "Recall_positive_global_residue",
        "Specificity_global_residue",
        "F1_positive_global_residue",
        "MCC_global_residue",
        "F1_micro_all_classes",
        "F1_macro_two_classes",
        "protein_mean_F1_baseline_convention",
        "protein_mean_MCC_baseline_convention",
        "protein_n_included_baseline_convention",
    ]
]

final_threshold_report.to_csv(
    OUT / "FINAL_MODEL_ALL_THRESHOLDS.csv",
    index=False
)

# ------------------------------------------------------------
# L. Final decision JSON + display
# ------------------------------------------------------------
decision = {
    "task": TASK_NAME,
    "final_method": FINAL_NAME,
    "V1_name": V1_NAME,
    "MS_name": MS_NAME,
    "candidate_w_V1": W_V1_CAND,
    "candidate_w_MS": W_MS_CAND,
    "final_w_V1": W_V1,
    "final_w_MS": W_MS,
    "V1_Dev_mean_AUPRC": V1_DEV,
    "candidate_ensemble_Dev_mean_AUPRC": ENS_DEV_CAND,
    "final_Dev_mean_AUPRC": FINAL_DEV,
    "candidate_Dev_gain_vs_V1": ENS_DEV_CAND - V1_DEV,
    "promoted_by_0.002_rule": bool(PROMOTE),
    "crossfit_mean_AUPRC_ensemble":
        float(crossfit.held_fold_AUPRC_crossfit_ensemble.mean()),
    "crossfit_mean_AUPRC_V1":
        float(crossfit.held_fold_AUPRC_V1.mean()),
    "metric_naming": {
        "F1_positive_global_residue":
            "binary positive-class F1 after pooling all residues",
        "MCC_global_residue":
            "MCC from one pooled residue-level confusion matrix",
        "F1_micro_all_classes":
            "formal sklearn-style micro-F1 across both classes; "
            "equals accuracy in this binary single-label task",
        "protein_mean_metrics":
            "per-protein metrics averaged after excluding TN-only proteins "
            "(TP+FP+FN=0), matching reproduced baseline convention",
    },
    "threshold_rules": {
        method: {k: float(v) for k, v in rules.items()}
        for method, rules in THRESHOLDS.items()
    },
    "prediction_files": final_prediction_files,
}

(OUT / "FINAL_DECISION_AND_DEFINITIONS.json").write_text(
    json.dumps(decision, indent=2)
)

print("\n" + "="*100)
print(f"{TASK_NAME} — HEADLINE FIXED 0.5 (DIRECT BASELINE COMPARISON)")
print("="*100)
display(headline)

print("\n" + "="*100)
print(f"{TASK_NAME} — FINAL MODEL AT ALL DEV-LOCKED THRESHOLDS")
print("="*100)
display(final_threshold_report)

print("\nMetric naming:")
print("  F1_positive_global_residue = pooled binary positive-class F1")
print("  MCC_global_residue         = pooled/global residue-level MCC")
print("  F1_micro_all_classes       = formal micro-F1; equals Accuracy here")
print("  protein_mean_*             = baseline convention, TN-only excluded")

print("\nSaved:", OUT)
print("Key files:")
for fn in [
    "DEV_ENSEMBLE_WEIGHT_SEARCH.csv",
    "DEV_CROSSFIT_ENSEMBLE_AUDIT.csv",
    "DEV_THRESHOLD_LOCK.csv",
    "DEV_OOF_FULL_METRICS.csv",
    "EXTERNAL_FULL_METRICS_LONG.csv",
    "HEADLINE_FIXED05_COMPARISON.csv",
    "FINAL_MODEL_ALL_THRESHOLDS.csv",
    "FINAL_DECISION_AND_DEFINITIONS.json",
]:
    print(" -", OUT / fn)


SMALL DEV ENSEMBLE LOCK


,w_V1,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
0,0.60,0.40,0.409283,0.035424,0.395044,0.412509,0.458149,0.374628,0.375385,0.425746
1,0.55,0.45,0.409256,0.035038,0.395299,0.412701,0.457238,0.374688,0.375733,0.425922
2,0.65,0.35,0.409065,0.035797,0.394464,0.412071,0.458758,0.374274,0.374847,0.425376
3,0.50,0.50,0.409037,0.034590,0.395260,0.412780,0.456011,0.374847,0.375676,0.425870
4,0.45,0.55,0.408678,0.034145,0.394942,0.412877,0.454558,0.374607,0.375656,0.425691
5,0.70,0.30,0.408658,0.036168,0.393617,0.411329,0.459137,0.373738,0.374131,0.424957
6,0.75,0.25,0.408066,0.036586,0.392425,0.410514,0.459329,0.372861,0.373225,0.424402
7,0.40,0.60,0.408055,0.033690,0.394334,0.412767,0.452773,0.374213,0.375246,0.425273
8,0.35,0.65,0.407274,0.033119,0.393408,0.412503,0.450695,0.373863,0.374732,0.424580
9,0.80,0.20,0.407190,0.037073,0.390902,0.409508,0.459382,0.371795,0.371819,0.423447


Candidate weights: 0.6000000000000001 0.3999999999999999
Candidate Dev mean AUPRC: 0.4092833208152622
V1 Dev mean AUPRC: 0.39764655317751607
Candidate gain: 0.011636767637746126
Promote by >=0.002 rule: True
FINAL weights: 0.6000000000000001 0.3999999999999999
FINAL Dev mean AUPRC: 0.4092833208152622

Cross-fitted ensemble audit:


,held_fold,selected_w_V1_from_other_4_folds,selected_w_MS_from_other_4_folds,held_fold_AUPRC_crossfit_ensemble,held_fold_AUPRC_V1,held_fold_AUPRC_MS
0,1,0.6,0.4,0.412509,0.397298,0.402902
1,2,0.5,0.5,0.456011,0.455030,0.422164
2,3,0.6,0.4,0.374628,0.359973,0.361797
3,4,0.6,0.4,0.375385,0.358844,0.361232
4,5,0.6,0.4,0.425746,0.417088,0.415750


Crossfit mean AUPRC ensemble = 0.4088557786453836
Crossfit mean AUPRC V1       = 0.39764655317751607

DEV threshold locks:


...


SMALL — HEADLINE FIXED 0.5 (DIRECT BASELINE COMPARISON)


,dataset,method,threshold,AUPRC_global_residue,AUROC_global_residue,Accuracy_global_residue,Precision_positive_global_residue,Recall_positive_global_residue,F1_positive_global_residue,MCC_global_residue,F1_micro_all_classes,F1_macro_two_classes,protein_mean_F1_baseline_convention,protein_mean_MCC_baseline_convention
8,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,0.5,0.446015,0.868490,0.935076,0.478455,0.466206,0.472251,0.437710,0.935076,0.718831,0.353793,0.341372
4,TestSet300,MS_Attention,0.5,0.441991,0.863503,0.938873,0.511378,0.425755,0.464655,0.434571,0.938873,0.716120,0.350637,0.342336
0,TestSet300,V1_Attention,0.5,0.440255,0.867647,0.931503,0.454417,0.495136,0.473903,0.437803,0.931503,0.718635,0.347998,0.334175
26,TestSet300,bindNode24,0.5,0.431709,0.869662,0.932285,NaN,NaN,0.468245,0.432216,0.932285,NaN,0.353203,0.337026
24,TestSet300,bindEmbed21DL,0.5,0.394041,0.864302,0.925234,NaN,NaN,0.435777,0.396702,0.925234,NaN,0.331199,0.307604
20,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,0.5,0.427383,0.865695,0.927252,0.461957,0.400000,0.428752,0.391272,0.927252,0.694952,0.291153,0.261827
12,TestSetNew46,V1_Attention,0.5,0.424158,0.865581,0.925486,0.451852,0.430588,0.440964,0.401205,0.925486,0.700523,0.294249,0.262243
16,TestSetNew46,MS_Attention,0.5,0.419632,0.857929,0.926610,0.452381,0.357647,0.399474,0.363788,0.926610,0.680196,0.267372,0.237719
27,TestSetNew46,bindNode24,0.5,0.380681,0.872435,0.923077,NaN,NaN,0.407911,0.367430,0.923077,NaN,0.300463,0.266746
25,TestSetNew46,bindEmbed21DL,0.5,0.315583,0.857311,0.912157,NaN,NaN,0.374857,0.327813,0.912157,NaN,0.285816,0.253609



SMALL — FINAL MODEL AT ALL DEV-LOCKED THRESHOLDS


,dataset,method,threshold_rule,threshold,AUPRC_global_residue,AUROC_global_residue,Accuracy_global_residue,Balanced_accuracy_global_residue,Precision_positive_global_residue,Recall_positive_global_residue,Specificity_global_residue,F1_positive_global_residue,MCC_global_residue,F1_micro_all_classes,F1_macro_two_classes,protein_mean_F1_baseline_convention,protein_mean_MCC_baseline_convention,protein_n_included_baseline_convention
8,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,fixed_0.5,0.500000,0.446015,0.868490,0.935076,0.716219,0.478455,0.466206,0.966232,0.472251,0.437710,0.935076,0.718831,0.353793,0.341372,273
9,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_mcc_opt,0.515372,0.446015,0.868490,0.936129,0.712120,0.486619,0.456221,0.968018,0.470930,0.437238,0.936129,0.718472,0.354504,0.342818,270
10,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_positive_f1_opt,0.513291,0.446015,0.868490,0.936081,0.712931,0.486273,0.458013,0.967848,0.471721,0.437958,0.936081,0.718852,0.355290,0.343617,270
11,TestSet300,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_youden_j_opt,0.088084,0.446015,0.868490,0.828247,0.795249,0.231552,0.757552,0.832945,0.354690,0.354297,0.828247,0.627816,0.292920,0.276823,298
20,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,fixed_0.5,0.500000,0.427383,0.865695,0.927252,0.682937,0.461957,0.400000,0.965874,0.428752,0.391272,0.927252,0.694952,0.291153,0.261827,35
21,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_mcc_opt,0.515372,0.427383,0.865695,0.928376,0.681360,0.470588,0.395294,0.967425,0.429668,0.393460,0.928376,0.695728,0.291342,0.263575,35
22,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_positive_f1_opt,0.513291,0.427383,0.865695,0.928216,0.681273,0.469274,0.395294,0.967253,0.429119,0.392757,0.928216,0.695409,0.291055,0.263199,35
23,TestSetNew46,FINAL_DEV_LOCKED_ENSEMBLE,dev_oof_youden_j_opt,0.088084,0.427383,0.865695,0.821262,0.790694,0.241353,0.755294,0.826094,0.365812,0.357733,0.821262,0.630892,0.221548,0.197639,46



Metric naming:
  F1_positive_global_residue = pooled binary positive-class F1
  MCC_global_residue         = pooled/global residue-level MCC
  F1_micro_all_classes       = formal micro-F1; equals Accuracy here
  protein_mean_*             = baseline convention, TN-only excluded

Saved: /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_PAPER_READY
Key files:
 - /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_PAPER_READY/DEV_ENSEMBLE_WEIGHT_SEARCH.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_PAPER_READY/DEV_CROSSFIT_ENSEMBLE_AUDIT.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_PAPER_READY/DEV_THRESHOLD_LOCK.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_PAPER_READY/DEV_OOF_FULL_METRICS.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_PAPER_READY/EXTERNAL_FULL_METRICS_LONG.csv
 - /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_FINAL_PAPER_READY/HEADLINE_FIXED05